# RL fine-tuning of a generative recommender against a learned reward model

A GPT-2 decoder over item identifiers is trained to predict a user's next item and is then fine-tuned with
reinforcement learning to generate recommendation lists with a higher predicted user response. The response is
predicted by a second decoder, the reward model. The study measures what the fine-tuning changes: the predicted
response, the relevance of the lists, and how much the lists of different users differ.

**Notation.** A user's log is a sequence of events $(i_t, \rho_t, c_t, u_t)$: item $i_t$, played ratio
$\rho_t \in [0, 1]$, session continuation $c_t \in \{0, 1\}$ and return $u_t \in \{0, 1\}$. The recommender is
$\pi_\theta$, and $\pi_0$ is its state before fine-tuning. A recommendation list $a_{1:K}$ is generated item by
item from a history $h$, and $F$ is the set of the next $M$ items the user consumed after $h$.

This session tests the mechanisms behind the earlier findings with current methods. Each section states a
hypothesis and the result that would refute it.

1. **Is what RL learns one preference per item?** RL restricted to one score offset per item, or to rank-1
   updates of the transformer (LoRA), at equal KL; the main run split into its per-item and per-user parts; and the
   per-item share predicted from the reward model before any RL, through the closed form of the anchored objective.
2. **Does reaching RL's optimum need training at all?** The base recommender steered at inference by
   $\pi_0 \exp(Q/\beta)$, deciding one position at a time and looking ahead with a learned value of what follows
   (controlled decoding, Q-sharp), for the share and for adoption.
3. **Can the log alone move the frontier?** Exponential reward-weighted fine-tuning and within-user DPO on logged
   responses: no reward model, no generated lists.
4. **Must a list read its own unconfirmed items?** One head per position (multi-token prediction) against generated
   and ranked lists.
5. **Cheaper operating points.** Weight interpolation and extrapolation of the main run, and GRPO, against the KL sweep.
   Every run now records the scale of its advantages before standardisation, which the anchor weight is multiplied by.

## Package source

In [ ]:
from pathlib import Path

SOURCE_DIR = Path("/kaggle/working/src")
for package in ['gptrec', 'gptrec/baselines', 'gptrec/data', 'gptrec/evaluation', 'gptrec/models', 'gptrec/training']:
    (SOURCE_DIR / package).mkdir(parents=True, exist_ok=True)
    (SOURCE_DIR / package / "__init__.py").touch()

In [ ]:
%%writefile /kaggle/working/src/gptrec/baselines/bert4rec.py
"""BERT4Rec: a bidirectional Transformer trained to fill in hidden items (Sun et al., 2019)."""

from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import Tensor, nn

from gptrec.data.batches import EventBatch
from gptrec.data.log import PAD_ITEM
from gptrec.models.backbone import EMBEDDING_INIT_STD, TransformerConfig, new_item_embedding
from gptrec.models.sequence_recommender import MASKED_LOGIT


class BERT4Rec(nn.Module):
    """Predicts the next item by appending a mask token to the history and filling it in."""

    def __init__(self, num_items: int, config: TransformerConfig) -> None:
        super().__init__()
        self.num_items = num_items
        self.mask_token = num_items + 1
        self.max_positions = config.max_positions
        self.item_embedding = new_item_embedding(num_items + 1, config.hidden_size, PAD_ITEM)
        self.position_embedding = nn.Embedding(config.max_positions, config.hidden_size)
        nn.init.normal_(self.position_embedding.weight, std=EMBEDDING_INIT_STD)
        self.embedding_dropout = nn.Dropout(config.dropout)
        block = nn.TransformerEncoderLayer(
            config.hidden_size,
            config.num_heads,
            dim_feedforward=4 * config.hidden_size,
            dropout=config.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(block, config.num_layers, norm=nn.LayerNorm(config.hidden_size), enable_nested_tensor=False)
        self.item_bias = nn.Parameter(torch.zeros(num_items + 1))

    def hidden_states(self, tokens: Tensor) -> Tensor:
        """long [B, L] of item and mask tokens, right-padded -> [B, L, hidden]. Every position sees all real positions."""
        length = tokens.shape[1]
        if length > self.max_positions:
            raise ValueError(f"sequence of {length} tokens exceeds max_positions={self.max_positions}")
        positions = torch.arange(length, device=tokens.device)
        embedded = self.embedding_dropout(self.item_embedding(tokens) + self.position_embedding(positions))
        return self.blocks(embedded, src_key_padding_mask=tokens == PAD_ITEM)

    def item_logits(self, hidden: Tensor) -> Tensor:
        """[..., hidden] -> float32 [..., V+1] scores of the item behind a mask; PAD_ITEM can never be chosen."""
        logits = F.linear(hidden, self.item_embedding.weight[: self.num_items + 1], self.item_bias).float()
        logits[..., PAD_ITEM] = MASKED_LOGIT
        return logits

    def logits_at(self, tokens: Tensor, positions: Tensor) -> Tensor:
        """Item scores at one position per sequence. tokens: [B, L], positions: long [B] -> [B, V+1]"""
        return self.item_logits(self.hidden_states(tokens)[torch.arange(len(tokens), device=tokens.device), positions])

    def next_item_logits(self, items: Tensor, lengths: Tensor) -> Tensor:
        """Scores of the item that follows each right-padded sequence. [B, V+1]"""
        tokens = F.pad(items, (0, 1), value=PAD_ITEM)
        tokens[torch.arange(len(items), device=items.device), lengths] = self.mask_token
        return self.logits_at(tokens, lengths)


def cloze_loss(model: BERT4Rec, batch: EventBatch, mask_probability: float = 0.2) -> Tensor:
    """Cross-entropy of hidden target events given the rest of their window.

    Each target is hidden with `mask_probability`; the last event of every window is always hidden,
    because predicting it is the task the model is used for.
    """
    last_position = _last_position(batch)
    is_hidden = batch.is_target & (torch.rand(batch.items.shape, device=batch.items.device) < mask_probability)
    is_hidden[torch.arange(len(batch.items), device=batch.items.device), last_position] = True
    tokens = batch.items.masked_fill(is_hidden, model.mask_token)
    return F.cross_entropy(model.item_logits(model.hidden_states(tokens)[is_hidden]), batch.items[is_hidden])


def last_item_loss(model: BERT4Rec, batch: EventBatch) -> Tensor:
    """Cross-entropy of the last event of every window given all events before it: a deterministic validation loss."""
    last_position = _last_position(batch)
    batch_index = torch.arange(len(batch.items), device=batch.items.device)
    tokens = batch.items.clone()
    tokens[batch_index, last_position] = model.mask_token
    return F.cross_entropy(model.logits_at(tokens, last_position), batch.items[batch_index, last_position])


def one_target_per_window(batch: EventBatch) -> Tensor:
    """What `last_item_loss` averages over: one prediction per window. bool [B]"""
    return torch.ones(len(batch.items), dtype=torch.bool, device=batch.items.device)


def _last_position(batch: EventBatch) -> Tensor:
    """Index of the last real event of every right-padded window. long [B]"""
    return (batch.items != PAD_ITEM).sum(dim=1) - 1

In [ ]:
%%writefile /kaggle/working/src/gptrec/baselines/counting.py
"""Baselines that only count: item popularity, the user's own replays, and which item tends to follow which."""

from __future__ import annotations

import numpy as np
import torch
from torch import Tensor

from gptrec.data.log import PAD_ITEM, InteractionLog, ItemTable
from gptrec.models.sequence_recommender import MASKED_LOGIT


class PopularityRecommender:
    """Scores every item by its train-period popularity, the same for every user."""

    def __init__(self, items: ItemTable, device: torch.device) -> None:
        self._logits = torch.as_tensor(items.train_popularity, dtype=torch.float32, device=device).log1p()
        self._logits[PAD_ITEM] = MASKED_LOGIT

    def next_item_logits(self, items: Tensor, lengths: Tensor) -> Tensor:
        """[B, V+1], identical rows."""
        return self._logits.expand(len(items), -1)


class ReplayRecommender:
    """Recommends what the user already plays: items of the visible history, most played first, the more recent first among equals.

    No model is involved, so this is the floor for any recommender in a domain where people return to what
    they know. Items outside the history follow in order of popularity. Where history items may not be
    recommended at all, this reduces to popularity.
    """

    def __init__(self, items: ItemTable, device: torch.device) -> None:
        popularity = torch.as_tensor(items.train_popularity, dtype=torch.float32, device=device)
        self._popularity_share = popularity / popularity.sum()

    def next_item_logits(self, items: Tensor, lengths: Tensor) -> Tensor:
        """[B, V+1]: number of plays in the history, plus a recency term below 1 that orders equally played items."""
        num_users, width = items.shape
        recency = (torch.arange(1, width + 1, dtype=torch.float32, device=items.device) / (width + 1)).repeat(num_users, 1)
        plays = torch.zeros(num_users, len(self._popularity_share), device=items.device).scatter_add_(1, items, torch.ones_like(recency))
        last_played = torch.zeros_like(plays).scatter_reduce(1, items, recency, reduce="amax")
        logits = plays + last_played + self._popularity_share
        logits[:, PAD_ITEM] = MASKED_LOGIT
        return logits


class MarkovChainRecommender:
    """First-order Markov chain: scores an item by how often it directly followed the user's last item in the train period.

    Items never observed after the last item are ranked by popularity, below every observed successor.
    """

    def __init__(self, log: InteractionLog, device: torch.device, max_successors: int = 200) -> None:
        successors, transition_counts = _most_frequent_successors(log, max_successors)
        self._successors = torch.as_tensor(successors, device=device)
        self._transition_counts = torch.as_tensor(transition_counts, device=device)
        popularity = torch.as_tensor(log.items.train_popularity, dtype=torch.float32, device=device)
        self._popularity_share = popularity / popularity.sum()

    def next_item_logits(self, items: Tensor, lengths: Tensor) -> Tensor:
        """[B, V+1]"""
        last_item = items[torch.arange(len(items), device=items.device), lengths - 1]
        logits = self._popularity_share.repeat(len(items), 1)
        logits.scatter_add_(1, self._successors[last_item], self._transition_counts[last_item])
        logits[:, PAD_ITEM] = MASKED_LOGIT
        return logits


def _most_frequent_successors(log: InteractionLog, max_successors: int) -> tuple[np.ndarray, np.ndarray]:
    """For every item token, the items that most often came right after it in the train period, and how often.

    Returns (successors int64 [V+1, max_successors], counts float32 [V+1, max_successors]), most frequent first.
    """
    item = np.asarray(log.item).astype(np.int64)
    follows_same_user = np.ones(log.num_events, dtype=bool)
    follows_same_user[np.asarray(log.user_offsets[:-1])] = False
    in_train_period = np.asarray(log.timestamp) < log.split.validation_start
    is_transition = follows_same_user[1:] & in_train_period[1:]
    previous_item, next_item = item[:-1][is_transition], item[1:][is_transition]

    vocabulary_size = log.num_items + 1
    pair_code, pair_count = np.unique(previous_item * vocabulary_size + next_item, return_counts=True)
    previous_item, next_item = pair_code // vocabulary_size, pair_code % vocabulary_size
    by_item_then_frequency = np.lexsort((-pair_count, previous_item))
    previous_item, next_item, pair_count = previous_item[by_item_then_frequency], next_item[by_item_then_frequency], pair_count[by_item_then_frequency]
    rank_among_successors = np.arange(len(previous_item)) - np.searchsorted(previous_item, previous_item, side="left")
    kept = rank_among_successors < max_successors

    successors = np.full((vocabulary_size, max_successors), PAD_ITEM, dtype=np.int64)
    counts = np.zeros((vocabulary_size, max_successors), dtype=np.float32)
    successors[previous_item[kept], rank_among_successors[kept]] = next_item[kept]
    counts[previous_item[kept], rank_among_successors[kept]] = pair_count[kept]
    return successors, counts

In [ ]:
%%writefile /kaggle/working/src/gptrec/baselines/gru4rec.py
"""GRU4Rec: a recurrent next-item model (Hidasi et al., 2016), here trained with the full softmax like the GPT recommender."""

from __future__ import annotations

from torch import Tensor, nn

from gptrec.data.log import PAD_ITEM
from gptrec.models.backbone import TransformerConfig, new_item_embedding
from gptrec.models.sequence_recommender import SequenceRecommender


class GRU4Rec(SequenceRecommender):
    """Uses the width, depth and dropout of `config`; a GRU needs neither attention heads nor positions."""

    def __init__(self, num_items: int, config: TransformerConfig) -> None:
        super().__init__()
        self.item_embedding = new_item_embedding(num_items, config.hidden_size, PAD_ITEM)
        self.input_dropout = nn.Dropout(config.dropout)
        dropout_between_layers = config.dropout if config.num_layers > 1 else 0.0
        self.gru = nn.GRU(config.hidden_size, config.hidden_size, num_layers=config.num_layers, batch_first=True, dropout=dropout_between_layers)

    def hidden_states(self, items: Tensor) -> Tensor:
        states, _ = self.gru(self.input_dropout(self.item_embedding(items)))
        return states

In [ ]:
%%writefile /kaggle/working/src/gptrec/baselines/sasrec.py
"""SASRec: self-attentive sequential recommendation (Kang & McAuley, 2018).

A causal Transformer like the GPT recommender, but with the original training objective: binary
cross-entropy of the observed next item against sampled negatives, instead of the full softmax.
"""

from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import Tensor, nn

from gptrec.data.batches import EventBatch
from gptrec.data.log import PAD_ITEM
from gptrec.models.backbone import EMBEDDING_INIT_STD, TransformerConfig, new_item_embedding
from gptrec.models.sequence_recommender import SequenceRecommender
from gptrec.training.supervised import predicts_target


class SASRec(SequenceRecommender):
    def __init__(self, num_items: int, config: TransformerConfig) -> None:
        super().__init__()
        self.num_items = num_items
        self.item_embedding = new_item_embedding(num_items, config.hidden_size, PAD_ITEM)
        self.position_embedding = nn.Embedding(config.max_positions, config.hidden_size)
        nn.init.normal_(self.position_embedding.weight, std=EMBEDDING_INIT_STD)
        self.embedding_dropout = nn.Dropout(config.dropout)
        block = nn.TransformerEncoderLayer(
            config.hidden_size,
            config.num_heads,
            dim_feedforward=config.hidden_size,
            dropout=config.dropout,
            activation="relu",
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(block, config.num_layers, norm=nn.LayerNorm(config.hidden_size), enable_nested_tensor=False)

    def hidden_states(self, items: Tensor) -> Tensor:
        length = items.shape[1]
        positions = torch.arange(length, device=items.device)
        embedded = self.embedding_dropout(self.item_embedding(items) + self.position_embedding(positions))
        sees_the_future = torch.ones(length, length, dtype=torch.bool, device=items.device).triu(diagonal=1)
        return self.blocks(embedded, mask=sees_the_future)


def sampled_binary_cross_entropy(model: SASRec, batch: EventBatch, num_negatives: int = 1) -> Tensor:
    """Every target event against `num_negatives` items drawn uniformly from the catalogue.

    The paper draws negatives the user never interacted with; with a large catalogue a uniform draw
    hits one of the user's items rarely enough to skip that check.
    """
    positions = predicts_target(batch)
    states = model.hidden_states(batch.items)[:, :-1][positions]
    observed = batch.items[:, 1:][positions]
    sampled = torch.randint(1, model.num_items + 1, (len(observed), num_negatives), device=observed.device)
    item_embedding = model.item_embedding.weight
    observed_logits = (states * item_embedding[observed]).sum(dim=-1).float()
    sampled_logits = torch.einsum("nd,nkd->nk", states, item_embedding[sampled].to(states.dtype)).float()
    observed_loss = F.binary_cross_entropy_with_logits(observed_logits, torch.ones_like(observed_logits))
    sampled_loss = F.binary_cross_entropy_with_logits(sampled_logits, torch.zeros_like(sampled_logits))
    return observed_loss + sampled_loss

In [ ]:
%%writefile /kaggle/working/src/gptrec/baselines/suite.py
"""All baselines, fitted on the same train period and the same windows as the GPT recommender."""

from __future__ import annotations

from dataclasses import dataclass, field
from functools import partial
from pathlib import Path
from typing import Callable, NamedTuple

import torch
from torch import Tensor, nn

from gptrec.baselines.bert4rec import BERT4Rec, cloze_loss, last_item_loss, one_target_per_window
from gptrec.baselines.counting import MarkovChainRecommender, PopularityRecommender, ReplayRecommender
from gptrec.baselines.gru4rec import GRU4Rec
from gptrec.baselines.sasrec import SASRec, sampled_binary_cross_entropy
from gptrec.data.batches import EventBatch
from gptrec.data.log import InteractionLog
from gptrec.models.backbone import TransformerConfig
from gptrec.models.sequence_recommender import NextItemScorer
from gptrec.training.precision import seed_everything
from gptrec.training.supervised import LossFunction, Metrics, OptimizationConfig, Validation, fit_on_train_period, next_item_loss, predicts_target


@dataclass(frozen=True)
class BaselineConfig:
    optimization: OptimizationConfig = field(default_factory=OptimizationConfig)
    sasrec_negatives: int = 1
    bert4rec_mask_probability: float = 0.2
    markov_chain_successors: int = 200


class _Recipe(NamedTuple):
    """How one neural baseline is built and trained."""

    model_class: Callable[[int, TransformerConfig], nn.Module]
    training_loss: LossFunction
    validation_loss: LossFunction
    scored_positions: Callable[[EventBatch], Tensor]


def fit_baselines(
    log: InteractionLog,
    transformer: TransformerConfig,
    config: BaselineConfig,
    window_stride: int,
    device: torch.device,
    checkpoint_directory: Path,
    extra_validation: Validation | None = None,
) -> tuple[dict[str, NextItemScorer], dict[str, list[Metrics]]]:
    """Fits every baseline on the train period.

    The neural baselines share the GPT recommender's width, depth, dropout and sequence length. Their weights
    are kept in `checkpoint_directory` and loaded instead of retrained when present. `extra_validation` adds
    metrics to each training epoch's record, e.g. a ranking metric that is comparable between models whose
    losses are not. Returns the baselines by name, and the training history of those trained in this call.
    """
    sasrec_loss = partial(sampled_binary_cross_entropy, num_negatives=config.sasrec_negatives)
    recipes = {
        "GRU4Rec": _Recipe(GRU4Rec, next_item_loss, next_item_loss, predicts_target),
        "SASRec": _Recipe(SASRec, sasrec_loss, sasrec_loss, predicts_target),
        "BERT4Rec": _Recipe(BERT4Rec, partial(cloze_loss, mask_probability=config.bert4rec_mask_probability), last_item_loss, one_target_per_window),
    }
    baselines: dict[str, NextItemScorer] = {
        "most popular": PopularityRecommender(log.items, device),
        "replay history": ReplayRecommender(log.items, device),
        "Markov chain": MarkovChainRecommender(log, device, config.markov_chain_successors),
    }
    histories: dict[str, list[Metrics]] = {}
    checkpoint_directory.mkdir(parents=True, exist_ok=True)
    for name, recipe in recipes.items():
        seed_everything(config.optimization.seed)
        model = recipe.model_class(log.num_items, transformer).to(device)
        checkpoint = checkpoint_directory / f"{name}.pt"
        if checkpoint.exists():
            model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
        else:
            print(f"training {name}")
            histories[name] = fit_on_train_period(
                model,
                log,
                recipe.training_loss,
                recipe.scored_positions,
                transformer.max_positions,
                window_stride,
                config.optimization,
                device,
                validation_loss=recipe.validation_loss,
                extra_validation=extra_validation,
            )
            torch.save(model.state_dict(), checkpoint)
        baselines[name] = model.eval().requires_grad_(False)
    return baselines, histories

In [ ]:
%%writefile /kaggle/working/src/gptrec/data/batches.py
"""Turning the interaction log into tensors.

Every sequence tensor is right-padded with PAD_ITEM. The models attend causally, so a real event never
sees the padding behind it and no attention mask is needed anywhere.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Iterator

import numpy as np
import torch
from torch import Tensor

from gptrec.data.log import PAD_ITEM, InteractionLog, Period
from gptrec.tensors import TensorGroup


@dataclass
class HistoryBatch(TensorGroup):
    """What is known about each user at the moment a recommendation is requested.

    Row b holds the user's most recent events in chronological order in columns 0 .. lengths[b] - 1.
    """

    items: Tensor
    played_ratio: Tensor
    lengths: Tensor

    def repeat_interleave(self, repeats: int) -> HistoryBatch:
        """Each history `repeats` times in a row: rows b * repeats .. (b + 1) * repeats - 1 are copies of b."""
        return self.map(lambda tensor: tensor.repeat_interleave(repeats, dim=0))


@dataclass
class EvalExamples(TensorGroup):
    """Recommendation requests whose outcome is known: a user's history up to some moment and what the user did next."""

    history: HistoryBatch
    future_items: Tensor
    future_played_ratio: Tensor
    future_relative_ratio: Tensor
    future_is_organic: Tensor
    future_liked: Tensor
    future_adopts: Tensor
    user: Tensor

    def batches(self, batch_size: int) -> Iterator[EvalExamples]:
        for start in range(0, len(self), batch_size):
            yield self[start : start + batch_size]


@dataclass
class EventBatch(TensorGroup):
    """Windows of consecutive events for teacher-forced training and scoring."""

    items: Tensor
    played_ratio: Tensor
    relative_ratio: Tensor
    session_continues: Tensor
    returns: Tensor
    session_continues_known: Tensor
    returns_known: Tensor
    liked: Tensor
    disliked: Tensor
    reactions_known: Tensor
    adopts: Tensor
    adopts_known: Tensor
    is_organic: Tensor
    is_target: Tensor


@dataclass
class SwappedItems(TensorGroup):
    """Windows in which one target event each had its item replaced by an item the user did not choose then."""

    items: Tensor
    position: Tensor
    is_different: Tensor


def swap_one_target_item(batch: EventBatch) -> SwappedItems:
    """In every window, a random target event gets the item of a random target event of the batch.

    Replacements are therefore as popular as the items users really chose, and the events before the
    replaced one are untouched. Every window needs at least one target event.
    """
    position = torch.multinomial(batch.is_target.float(), num_samples=1)
    items_of_targets = batch.items[batch.is_target]
    replacement = items_of_targets[torch.randint(len(items_of_targets), position.shape, device=position.device)]
    is_different = (replacement != batch.items.gather(1, position)).squeeze(1)
    return SwappedItems(batch.items.scatter(1, position, replacement), position.squeeze(1), is_different)


def gather_right_padded(values: np.ndarray, starts: np.ndarray, ends: np.ndarray, width: int, pad_value) -> np.ndarray:
    """Stacks the slices values[starts[i] : ends[i]] into an [N, width] array, padding short slices on the right."""
    if len(starts) and (ends - starts).max() > width:
        raise ValueError(f"a slice is longer than the requested width {width}")
    positions = starts[:, None] + np.arange(width)[None, :]
    is_real = positions < ends[:, None]
    gathered = np.asarray(values)[np.minimum(positions, len(values) - 1)]
    return np.where(is_real, gathered, pad_value)


def _history_batch(log: InteractionLog, history_ends: np.ndarray, user_starts: np.ndarray, history_len: int) -> HistoryBatch:
    history_starts = np.maximum(user_starts, history_ends - history_len)
    return HistoryBatch(
        items=torch.from_numpy(gather_right_padded(log.item, history_starts, history_ends, history_len, PAD_ITEM)).long(),
        played_ratio=torch.from_numpy(gather_right_padded(log.played_ratio, history_starts, history_ends, history_len, 0.0)).float(),
        lengths=torch.from_numpy(history_ends - history_starts).long(),
    )


def _examples(log: InteractionLog, users: np.ndarray, future_starts: np.ndarray, future_limits: np.ndarray, history_len: int, max_future: int) -> EvalExamples:
    """For each user, the history before row `future_starts` and up to `max_future` events from that row on, none at or after `future_limits`."""
    user_starts = np.asarray(log.user_offsets[:-1])[users]
    future_ends = np.minimum(future_limits, future_starts + max_future)
    return EvalExamples(
        history=_history_batch(log, future_starts, user_starts, history_len),
        future_items=torch.from_numpy(gather_right_padded(log.item, future_starts, future_ends, max_future, PAD_ITEM)).long(),
        future_played_ratio=torch.from_numpy(gather_right_padded(log.played_ratio, future_starts, future_ends, max_future, 0.0)).float(),
        future_relative_ratio=torch.from_numpy(gather_right_padded(log.relative_ratio, future_starts, future_ends, max_future, 0.0)).float(),
        future_is_organic=torch.from_numpy(gather_right_padded(log.is_organic, future_starts, future_ends, max_future, True)),
        future_liked=torch.from_numpy(gather_right_padded(log.liked, future_starts, future_ends, max_future, False)),
        future_adopts=torch.from_numpy(
            gather_right_padded(log.adopts, future_starts, future_ends, max_future, False) & gather_right_padded(log.adopts_known, future_starts, future_ends, max_future, False)
        ),
        user=torch.from_numpy(users).long(),
    )


def build_eval_examples(log: InteractionLog, period: Period, history_len: int, max_future: int) -> EvalExamples:
    """A request at the start of `period` for every user who has events both before and inside it."""
    user_starts = np.asarray(log.user_offsets[:-1])
    period_starts, period_ends = log.period_rows(period)
    users = np.flatnonzero((period_starts > user_starts) & (period_ends > period_starts))
    return _examples(log, users, period_starts[users], period_ends[users], history_len, max_future)


class TrainingHistorySampler:
    """Draws random recommendation moments from the train period: a random user at a random point of their timeline.

    Each comes with what the user did next inside the train period, which may be nothing.
    """

    def __init__(self, log: InteractionLog, history_len: int, max_future: int, min_history: int = 8, seed: int = 0) -> None:
        self._log = log
        self._history_len = history_len
        self._max_future = max_future
        self._min_history = min_history
        self._random = np.random.default_rng(seed)
        self._user_starts = np.asarray(log.user_offsets[:-1])
        _, self._train_ends = log.period_rows("train")
        self._eligible_users = np.flatnonzero(self._train_ends - self._user_starts >= min_history)
        if len(self._eligible_users) == 0:
            raise ValueError(f"no user has {min_history} train events")

    def sample(self, batch_size: int) -> EvalExamples:
        users = self._random.choice(self._eligible_users, size=batch_size)
        earliest_end = self._user_starts[users] + self._min_history
        history_ends = self._random.integers(earliest_end, self._train_ends[users], endpoint=True)
        return _examples(self._log, users, history_ends, self._train_ends[users], self._history_len, self._max_future)


class EventWindows:
    """Slices of user timelines, each a run of consecutive events of one user.

    Events at or after a window's target start are targets; the events before it are context only.
    """

    def __init__(self, log: InteractionLog, window_starts: np.ndarray, window_ends: np.ndarray, target_starts: np.ndarray, seq_len: int) -> None:
        self._log = log
        self._window_starts = window_starts
        self._window_ends = window_ends
        self._target_starts = target_starts
        self.seq_len = seq_len

    @classmethod
    def for_training(cls, log: InteractionLog, seq_len: int, stride: int | None = None, users: np.ndarray | None = None) -> EventWindows:
        """Windows covering every user's train period; consecutive windows start at most `stride` events apart. All events are targets.

        The last window of a user is moved back to end at the user's last train event, so no window is a short leftover.
        """
        stride = stride or seq_len
        if not 0 < stride <= seq_len:
            raise ValueError(f"stride {stride} must be between 1 and seq_len={seq_len}, otherwise events fall between windows")
        period_starts, period_ends = (rows[_selection(users)] for rows in log.period_rows("train"))
        num_events = period_ends - period_starts
        windows_per_user = np.where(num_events >= 2, 1 + _ceil_div(np.maximum(num_events - seq_len, 0), stride), 0)
        last_window_starts = np.maximum(period_ends - seq_len, period_starts)
        window_starts = np.repeat(period_starts, windows_per_user) + stride * _index_within_group(windows_per_user)
        window_starts = np.minimum(window_starts, np.repeat(last_window_starts, windows_per_user))
        window_ends = np.minimum(window_starts + seq_len, np.repeat(period_ends, windows_per_user))
        return cls(log, window_starts, window_ends, target_starts=window_starts, seq_len=seq_len)

    @classmethod
    def for_heldout(cls, log: InteractionLog, period: Period, seq_len: int, users: np.ndarray | None = None) -> EventWindows:
        """Windows in which every event of `period` is a target exactly once, preceded by at least seq_len // 2 events of context."""
        targets_per_window = seq_len // 2
        user_starts = np.asarray(log.user_offsets[:-1])[_selection(users)]
        period_starts, period_ends = (rows[_selection(users)] for rows in log.period_rows(period))
        windows_per_user = _ceil_div(period_ends - period_starts, targets_per_window)
        target_starts = np.repeat(period_starts, windows_per_user) + targets_per_window * _index_within_group(windows_per_user)
        window_ends = np.minimum(target_starts + targets_per_window, np.repeat(period_ends, windows_per_user))
        window_starts = np.maximum(np.repeat(user_starts, windows_per_user), window_ends - seq_len)
        return cls(log, window_starts, window_ends, target_starts, seq_len)

    def __len__(self) -> int:
        return len(self._window_starts)

    def num_batches(self, batch_size: int, drop_last: bool = False) -> int:
        return len(self) // batch_size if drop_last else _ceil_div(len(self), batch_size)

    def batches(self, batch_size: int, shuffle_seed: int | None = None, drop_last: bool = False) -> Iterator[EventBatch]:
        """Iterates over all windows once; in random order when `shuffle_seed` is given."""
        order = np.arange(len(self)) if shuffle_seed is None else np.random.default_rng(shuffle_seed).permutation(len(self))
        for batch in range(self.num_batches(batch_size, drop_last)):
            yield self._batch(np.sort(order[batch * batch_size : (batch + 1) * batch_size]))

    def _batch(self, windows: np.ndarray) -> EventBatch:
        starts, ends = self._window_starts[windows], self._window_ends[windows]
        rows = starts[:, None] + np.arange(self.seq_len)[None, :]
        is_target = (rows >= self._target_starts[windows][:, None]) & (rows < ends[:, None])

        def gather(values: np.ndarray, pad_value) -> Tensor:
            return torch.from_numpy(gather_right_padded(values, starts, ends, self.seq_len, pad_value))

        return EventBatch(
            items=gather(self._log.item, PAD_ITEM).long(),
            played_ratio=gather(self._log.played_ratio, 0.0).float(),
            relative_ratio=gather(self._log.relative_ratio, 0.0).float(),
            session_continues=gather(self._log.session_continues, False).float(),
            returns=gather(self._log.returns, False).float(),
            session_continues_known=gather(self._log.session_continues_known, False),
            returns_known=gather(self._log.returns_known, False),
            liked=gather(self._log.liked, False).float(),
            disliked=gather(self._log.disliked, False).float(),
            reactions_known=gather(self._log.reactions_known, False),
            adopts=gather(self._log.adopts, False).float(),
            adopts_known=gather(self._log.adopts_known, False),
            is_organic=gather(self._log.is_organic, False),
            is_target=torch.from_numpy(is_target),
        )


def _selection(users: np.ndarray | None):
    """Index that keeps the given users, or everyone when no subset is requested."""
    return slice(None) if users is None else users


def _ceil_div(numerator, denominator):
    return -(-numerator // denominator)


def _index_within_group(group_sizes: np.ndarray) -> np.ndarray:
    """[0, 1, .., n0 - 1, 0, 1, .., n1 - 1, ...] for group sizes n0, n1, ..."""
    group_first_index = np.cumsum(group_sizes) - group_sizes
    return np.arange(group_sizes.sum()) - np.repeat(group_first_index, group_sizes)

In [ ]:
%%writefile /kaggle/working/src/gptrec/data/datasets.py
"""The datasets the pipeline can run on, by name."""

from __future__ import annotations

from pathlib import Path

import polars as pl

from gptrec.data import kion, yambda
from gptrec.paths import RAW_DATA_DIR

DATASETS = ("yambda", "kion")


def download_raw_data(dataset: str, directory: Path = RAW_DATA_DIR) -> Path:
    """Path of the dataset's interaction file, downloaded on first use."""
    if dataset == "yambda":
        return yambda.download_listens(directory=directory)
    if dataset == "kion":
        return kion.download_interactions(directory)
    raise ValueError(f"unknown dataset {dataset!r}; expected one of {DATASETS}")


def download_side_data(dataset: str, directory: Path = RAW_DATA_DIR) -> dict[str, Path]:
    """Paths of what the dataset records beyond its interactions, downloaded on first use: Yambda's likes and dislikes, KION's item table."""
    if dataset == "yambda":
        return yambda.download_reactions(directory=directory)
    if dataset == "kion":
        return {"items": kion.download_items(directory)}
    raise ValueError(f"unknown dataset {dataset!r}; expected one of {DATASETS}")


def load_raw_events(dataset: str, raw_data_path: str | Path) -> pl.DataFrame:
    """The dataset's interactions in the raw-event format of `gptrec.data.preprocess`."""
    if dataset == "yambda":
        return yambda.load_raw_events(raw_data_path)
    if dataset == "kion":
        return kion.load_raw_events(raw_data_path)
    raise ValueError(f"unknown dataset {dataset!r}; expected one of {DATASETS}")


def load_reactions(dataset: str, side_data_paths: dict[str, str | Path]) -> pl.DataFrame | None:
    """Likes and dislikes in the reaction format of `gptrec.data.preprocess`; None for a dataset without them."""
    return yambda.load_reactions(side_data_paths) if dataset == "yambda" else None


def load_item_metadata(dataset: str, side_data_paths: dict[str, str | Path]) -> pl.DataFrame | None:
    """Names and kinds of the items, keyed by the dataset's item_id; None for a dataset without them."""
    return kion.load_item_metadata(side_data_paths["items"]) if dataset == "kion" else None

In [ ]:
%%writefile /kaggle/working/src/gptrec/data/kion.py
"""Adapter for KION, the MTS online-cinema dataset (https://github.com/irsafilo/KION_DATASET).

One row per user-item pair: the date the user last watched the film or series, the seconds watched
in total, and the share of the item watched. The item table names every film and series.
"""

from __future__ import annotations

import shutil
import urllib.request
import zipfile
from pathlib import Path

import polars as pl

from gptrec.data.preprocess import SECONDS_PER_DAY
from gptrec.paths import RAW_DATA_DIR

ARCHIVE_URL = "https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_en.zip"
INTERACTIONS_FILE = "interactions.csv"
ITEMS_FILE = "items.csv"
ITEM_COLUMNS = ("content_type", "title", "release_year", "genres")


def download_interactions(directory: Path = RAW_DATA_DIR) -> Path:
    """Path of the interactions table, downloaded and unpacked on first use."""
    return _unpacked(INTERACTIONS_FILE, member_prefix="interactions", directory=directory)


def download_items(directory: Path = RAW_DATA_DIR) -> Path:
    """Path of the item table, downloaded and unpacked on first use."""
    return _unpacked(ITEMS_FILE, member_prefix="items", directory=directory)


def _unpacked(file_name: str, member_prefix: str, directory: Path) -> Path:
    """The archive member whose name starts with `member_prefix` and ends in .csv, unpacked to <directory>/kion/<file_name>."""
    path = directory / "kion" / file_name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        archive_path, _ = urllib.request.urlretrieve(ARCHIVE_URL)
        with zipfile.ZipFile(archive_path) as archive:
            member = next(
                name for name in archive.namelist()
                if not name.startswith("__MACOSX") and name.rsplit("/", 1)[-1].startswith(member_prefix) and name.endswith(".csv")
            )
            with archive.open(member) as packed, open(path, "wb") as unpacked:
                shutil.copyfileobj(packed, unpacked)
        Path(archive_path).unlink()
    return path


def load_raw_events(interactions_path: str | Path) -> pl.DataFrame:
    """The interactions table in the raw-event format of `gptrec.data.preprocess`.

    What KION does not record is filled in as follows. Timestamps are dates, so the order of a user's
    events within one day is the order of the file. The item's length is estimated from each viewing as
    seconds watched / share watched. Nothing says whether a viewing came from a recommendation, so
    every event counts as organic.
    """
    watched_share = pl.col("watched_pct") / 100.0
    return (
        pl.read_csv(interactions_path)
        .drop_nulls(["last_watch_dt", "watched_pct"])
        .select(
            user_id=pl.col("user_id").cast(pl.Int64),
            item_id=pl.col("item_id").cast(pl.Int64),
            timestamp=pl.col("last_watch_dt").str.to_date().to_physical().cast(pl.Int64) * SECONDS_PER_DAY,
            played_ratio=watched_share.clip(0.0, 1.0).cast(pl.Float32),
            duration_seconds=pl.when(pl.col("watched_pct") > 0).then(pl.col("total_dur") / watched_share).cast(pl.Float32),
            is_organic=pl.lit(True),
        )
    )


def load_item_metadata(items_path: str | Path) -> pl.DataFrame:
    """The item table: item_id and whichever of ITEM_COLUMNS it has."""
    items = pl.read_csv(items_path, infer_schema_length=10_000)
    return items.select(pl.col("item_id").cast(pl.Int64), *[column for column in ITEM_COLUMNS if column in items.columns])

In [ ]:
%%writefile /kaggle/working/src/gptrec/data/log.py
"""The preprocessed interaction log: its in-memory layout and its on-disk format."""

from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Literal

import numpy as np

PAD_ITEM = 0

Period = Literal["train", "validation", "test"]


@dataclass(frozen=True)
class TemporalSplit:
    """Global time boundaries, on the same clock as event timestamps.

    train: t < validation_start;  validation: validation_start <= t < test_start;  test: t >= test_start.
    """

    validation_start: int
    test_start: int

    def bounds(self, period: Period) -> tuple[float, float]:
        return {
            "train": (-np.inf, self.validation_start),
            "validation": (self.validation_start, self.test_start),
            "test": (self.test_start, np.inf),
        }[period]


@dataclass(frozen=True)
class ItemTable:
    """Per-item attributes indexed by item token. Row 0 belongs to PAD_ITEM and holds zeros.

    Tokens are assigned by train-period popularity: token 1 is the most popular item.
    """

    raw_id: np.ndarray
    train_popularity: np.ndarray
    duration_seconds: np.ndarray
    mean_played_ratio: np.ndarray

    @property
    def num_items(self) -> int:
        return len(self.raw_id) - 1

    @property
    def train_mean_played_ratio(self) -> float:
        """Mean played ratio over all train-period events."""
        return float(np.average(self.mean_played_ratio, weights=self.train_popularity))


@dataclass(frozen=True)
class InteractionLog:
    """Events of all users, sorted by (user, time). Event arrays are aligned: index i is one event.

    The events of user u are rows user_offsets[u] : user_offsets[u + 1].
    """

    item: np.ndarray
    played_ratio: np.ndarray
    relative_ratio: np.ndarray
    session_continues: np.ndarray
    returns: np.ndarray
    session_continues_known: np.ndarray
    returns_known: np.ndarray
    liked: np.ndarray
    disliked: np.ndarray
    reactions_known: np.ndarray
    adopts: np.ndarray
    adopts_known: np.ndarray
    is_organic: np.ndarray
    timestamp: np.ndarray
    user_offsets: np.ndarray
    items: ItemTable
    split: TemporalSplit
    directory: Path | None = None

    @property
    def num_users(self) -> int:
        return len(self.user_offsets) - 1

    @property
    def num_events(self) -> int:
        return len(self.item)

    @property
    def num_items(self) -> int:
        return self.items.num_items

    def period_rows(self, period: Period) -> tuple[np.ndarray, np.ndarray]:
        """Per-user row range [start, end) of the events that fall into `period`. Both arrays are int64 [U]."""
        period_start, period_end = self.split.bounds(period)
        return self._first_row_at_or_after(period_start), self._first_row_at_or_after(period_end)

    def _first_row_at_or_after(self, time: float) -> np.ndarray:
        user_starts, user_ends = np.asarray(self.user_offsets[:-1]), np.asarray(self.user_offsets[1:])
        earlier_events = np.concatenate([[0], np.cumsum(np.asarray(self.timestamp) < time)])
        return user_starts + earlier_events[user_ends] - earlier_events[user_starts]

    def save(self, directory: str | Path) -> InteractionLog:
        """Writes the log to `directory` and returns it re-opened from disk (memory-mapped)."""
        directory = Path(directory)
        directory.mkdir(parents=True, exist_ok=True)
        for name in _EVENT_ARRAYS:
            np.save(directory / f"{name}.npy", getattr(self, name))
        for name in _ITEM_ARRAYS:
            np.save(directory / f"item_{name}.npy", getattr(self.items, name))
        (directory / "split.json").write_text(json.dumps(asdict(self.split)))
        return InteractionLog.load(directory)

    @classmethod
    def load(cls, directory: str | Path) -> InteractionLog:
        """Opens a saved log. Event arrays are memory-mapped, so opening is instant and cheap to repeat."""
        directory = Path(directory)
        events = {name: np.load(directory / f"{name}.npy", mmap_mode="r") for name in _EVENT_ARRAYS}
        items = ItemTable(**{name: np.load(directory / f"item_{name}.npy") for name in _ITEM_ARRAYS})
        split = TemporalSplit(**json.loads((directory / "split.json").read_text()))
        return cls(**events, items=items, split=split, directory=directory)


_EVENT_ARRAYS = (
    "item",
    "played_ratio",
    "relative_ratio",
    "session_continues",
    "returns",
    "session_continues_known",
    "returns_known",
    "liked",
    "disliked",
    "reactions_known",
    "adopts",
    "adopts_known",
    "is_organic",
    "timestamp",
    "user_offsets",
)
_ITEM_ARRAYS = ("raw_id", "train_popularity", "duration_seconds", "mean_played_ratio")

In [ ]:
%%writefile /kaggle/working/src/gptrec/data/preprocess.py
"""From a raw event table to an InteractionLog: sessions, temporal split, item vocabulary, user filter.

Every dataset adapter (see `gptrec.data.yambda`) produces the same raw event table, a polars
DataFrame with one row per interaction and the columns listed in RAW_EVENT_COLUMNS. A dataset that
records likes and dislikes also produces a reaction table with the columns of REACTION_COLUMNS.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import polars as pl

from gptrec.data.log import InteractionLog, ItemTable, TemporalSplit

RAW_EVENT_COLUMNS = {
    "user_id": "any integer id",
    "item_id": "any integer id",
    "timestamp": "seconds, any origin",
    "played_ratio": "fraction of the item consumed, in [0, 1]",
    "duration_seconds": "length of the item; null where unknown",
    "is_organic": "True when the user picked the item themselves, False when it was recommended",
}
REACTION_COLUMNS = {
    "user_id": "as in the raw events",
    "item_id": "as in the raw events",
    "timestamp": "seconds, on the clock of the raw events",
    "liked": "True for a like, False for a dislike",
}

SECONDS_PER_DAY = 86_400


@dataclass(frozen=True)
class PreprocessingConfig:
    max_items: int = 30_000
    min_item_events: int = 20
    min_user_events: int = 50
    validation_days: float = 7.0
    test_days: float = 7.0
    session_gap_seconds: int = 30 * 60
    return_window_seconds: int = SECONDS_PER_DAY
    reaction_window_seconds: int = SECONDS_PER_DAY
    adoption_window_seconds: int | None = None


def build_interaction_log(raw_events: pl.DataFrame, config: PreprocessingConfig, reactions: pl.DataFrame | None = None) -> InteractionLog:
    """Builds the in-memory log. Call `.save(directory)` on the result to persist and memory-map it.

    `reactions`: likes and dislikes with the columns of REACTION_COLUMNS; without them every reaction label is unknown.
    """
    events = _in_user_time_order(raw_events.select(list(RAW_EVENT_COLUMNS)).with_row_index("source_row"))
    split = _temporal_split(events, config)
    events = events.with_columns(_retention_signals(events, split, config))
    events = events.with_columns(_reaction_labels(events, reactions, split, config))
    events = events.with_columns(_adoption_labels(events, split, config))
    in_train_period = pl.col("timestamp") < split.validation_start

    catalogue = _item_catalogue(events.filter(in_train_period), config)
    events = events.join(catalogue.select("item_id", "token"), on="item_id", how="inner")
    events = events.with_columns(_relative_ratio(events, split))
    train_events_per_user = events.filter(in_train_period).group_by("user_id").agg(pl.len().alias("num_train_events"))
    active_users = train_events_per_user.filter(pl.col("num_train_events") >= config.min_user_events)
    events = _in_user_time_order(events.join(active_users.select("user_id"), on="user_id", how="semi"))

    events_per_user = events.group_by("user_id", maintain_order=True).agg(pl.len().alias("num_events"))
    user_offsets = np.concatenate([[0], np.cumsum(events_per_user["num_events"].to_numpy())]).astype(np.int64)
    print(
        f"kept {len(events):,} of {len(raw_events):,} events ({len(events) / len(raw_events):.1%}), "
        f"{len(events_per_user):,} users, {len(catalogue):,} items"
    )
    return InteractionLog(
        item=events["token"].to_numpy().astype(np.int32),
        played_ratio=events["played_ratio"].to_numpy().astype(np.float32),
        relative_ratio=events["relative_ratio"].to_numpy().astype(np.float32),
        session_continues=events["session_continues"].to_numpy().astype(bool),
        returns=events["returns"].to_numpy().astype(bool),
        session_continues_known=events["session_continues_known"].to_numpy().astype(bool),
        returns_known=events["returns_known"].to_numpy().astype(bool),
        liked=events["liked"].to_numpy().astype(bool),
        disliked=events["disliked"].to_numpy().astype(bool),
        reactions_known=events["reactions_known"].to_numpy().astype(bool),
        adopts=events["adopts"].to_numpy().astype(bool),
        adopts_known=events["adopts_known"].to_numpy().astype(bool),
        is_organic=events["is_organic"].to_numpy().astype(bool),
        timestamp=events["timestamp"].to_numpy().astype(np.int64),
        user_offsets=user_offsets,
        items=_item_table(catalogue),
        split=split,
    )


def _in_user_time_order(events: pl.DataFrame) -> pl.DataFrame:
    """Sorted by user, then time; events with equal timestamps keep the order of the source table."""
    return events.sort(["user_id", "timestamp", "source_row"])


def _retention_signals(events: pl.DataFrame, split: TemporalSplit, config: PreprocessingConfig) -> list[pl.Series]:
    """For every event, whether the user kept going and whether they came back. Expects user-time order.

    session_continues: the user's next event follows within the session gap.
    returns: the user starts another session within the return window, i.e. has a later event that is
             more than a session gap but at most a return window away.

    That nothing followed is only a fact once the waiting time has passed inside the observed data, so each
    signal comes with a `_known` flag. Train-period events are observed until the train period ends: their
    labels never depend on held-out events. Held-out events are observed until the log ends.
    """
    timestamp, user_id = events["timestamp"].to_numpy(), events["user_id"].to_numpy()
    clock = _clock_shared_by_all_users(timestamp, user_id, longest_wait=config.session_gap_seconds + config.return_window_seconds)

    seconds_to_next_event = np.diff(clock, append=np.iinfo(np.int64).max)
    first_event_after_session = np.searchsorted(clock, clock + config.session_gap_seconds, side="right")
    has_event_after_session = first_event_after_session < len(clock)
    seconds_to_next_session = clock[np.minimum(first_event_after_session, len(clock) - 1)] - clock

    last_observed_second = _last_observed_second(timestamp, split)
    return [
        pl.Series("session_continues", seconds_to_next_event <= config.session_gap_seconds),
        pl.Series("returns", has_event_after_session & (seconds_to_next_session <= config.return_window_seconds)),
        pl.Series("session_continues_known", timestamp + config.session_gap_seconds <= last_observed_second),
        pl.Series("returns_known", timestamp + config.return_window_seconds <= last_observed_second),
    ]


def _last_observed_second(timestamp: np.ndarray, split: TemporalSplit) -> np.ndarray:
    """The last moment whose events may decide the label of each event: train-period labels never depend on held-out events. int64 [E]"""
    return np.where(timestamp < split.validation_start, split.validation_start - 1, timestamp.max())


def _reaction_labels(events: pl.DataFrame, reactions: pl.DataFrame | None, split: TemporalSplit, config: PreprocessingConfig) -> list[pl.Series]:
    """For every event, whether the user liked or disliked the item afterwards, and whether the log can tell.

    A reaction counts for the user's latest event with the same item at most `reaction_window_seconds` before
    it, so a like given during or soon after a listen labels that listen and no earlier one. A label is known
    once its window has passed inside the observed data; without a reaction table no label is known.
    Expects user-time order and the raw item ids.
    """
    timestamp = events["timestamp"].to_numpy()
    last_observed_second = _last_observed_second(timestamp, split)
    nothing = np.zeros(len(events), dtype=bool)
    if reactions is None:
        return [pl.Series("liked", nothing), pl.Series("disliked", nothing), pl.Series("reactions_known", nothing)]

    listens = events.select(
        "source_row", "user_id", "item_id", listen_timestamp=pl.col("timestamp"), last_observed_second=pl.Series(last_observed_second)
    ).sort("listen_timestamp")
    attributed = (
        reactions.select(list(REACTION_COLUMNS))
        .sort("timestamp")
        .join_asof(
            listens,
            left_on="timestamp",
            right_on="listen_timestamp",
            by=["user_id", "item_id"],
            strategy="backward",
            tolerance=config.reaction_window_seconds,
        )
        .filter(pl.col("source_row").is_not_null() & (pl.col("timestamp") <= pl.col("last_observed_second")))
        .group_by("source_row")
        .agg(liked=pl.col("liked").any(), disliked=(~pl.col("liked")).any())
    )
    source_row = events["source_row"].to_numpy().astype(np.int64)
    attributed_row = attributed["source_row"].to_numpy().astype(np.int64)
    liked, disliked = np.zeros(source_row.max() + 1, dtype=bool), np.zeros(source_row.max() + 1, dtype=bool)
    liked[attributed_row] = attributed["liked"].to_numpy().astype(bool)
    disliked[attributed_row] = attributed["disliked"].to_numpy().astype(bool)
    return [
        pl.Series("liked", liked[source_row]),
        pl.Series("disliked", disliked[source_row]),
        pl.Series("reactions_known", timestamp + config.reaction_window_seconds <= last_observed_second),
    ]


def _adoption_labels(events: pl.DataFrame, split: TemporalSplit, config: PreprocessingConfig) -> list[pl.Series]:
    """For every event, whether it is the user's first with the item and the user came back to the item later.

    The item counts as adopted when the user consumed it again more than a session gap and at most
    `adoption_window_seconds` after the first time. The label is known for first events whose window ends
    inside the observed data, and unknown for every later event and when no window is set. A replay inside
    the first session does not count: it says more about the session than about the item. Expects user-time order.
    """
    nothing = np.zeros(len(events), dtype=bool)
    if config.adoption_window_seconds is None:
        return [pl.Series("adopts", nothing), pl.Series("adopts_known", nothing)]
    by_pair = ["user_id", "item_id"]
    pairs = events.select(*by_pair, "timestamp").with_columns(
        first_timestamp=pl.col("timestamp").min().over(by_pair),
        is_first=pl.int_range(pl.len()).over(by_pair) == 0,
    )
    is_after_first_session = pl.col("timestamp") > pl.col("first_timestamp") + config.session_gap_seconds
    comeback = pairs.select(pl.when(is_after_first_session).then(pl.col("timestamp")).min().over(by_pair)).to_series()
    timestamp = events["timestamp"].to_numpy()
    is_first = pairs["is_first"].to_numpy().astype(bool)
    seconds_to_comeback = comeback.fill_null(np.iinfo(np.int64).max).to_numpy() - timestamp
    return [
        pl.Series("adopts", is_first & (seconds_to_comeback <= config.adoption_window_seconds)),
        pl.Series("adopts_known", is_first & (timestamp + config.adoption_window_seconds <= _last_observed_second(timestamp, split))),
    ]


def _relative_ratio(events: pl.DataFrame, split: TemporalSplit) -> pl.Series:
    """For every event, the share of the item's train-period events that were played less, equally played ones counting half.

    A quantile inside the item: 0.5 is what is usual for it, whatever its length or kind, and the train-period
    mean of every item is exactly 0.5. A long series that hardly anybody finishes and a film that most people
    finish are thereby put on one scale. Held-out events are ranked against the train period only.
    Expects the `token` column of catalogue items.
    """
    token = events["token"].to_numpy().astype(np.float64)
    played_ratio = events["played_ratio"].to_numpy().astype(np.float64)
    sort_key = 2.0 * token + played_ratio
    train_keys = np.sort(sort_key[events["timestamp"].to_numpy() < split.validation_start])
    first_event_of_item = np.searchsorted(train_keys, 2.0 * token, side="left")
    num_train_events = np.searchsorted(train_keys, 2.0 * token + 1.0, side="right") - first_event_of_item
    played_less = np.searchsorted(train_keys, sort_key, side="left") - first_event_of_item
    played_at_most_as_much = np.searchsorted(train_keys, sort_key, side="right") - first_event_of_item
    return pl.Series("relative_ratio", ((played_less + played_at_most_as_much) / (2.0 * num_train_events)).astype(np.float32))


def _clock_shared_by_all_users(timestamp: np.ndarray, user_id: np.ndarray, longest_wait: int) -> np.ndarray:
    """Increasing event times in which two users are never within `longest_wait` seconds of each other.

    Events are in user-time order, so one binary search over this clock finds a user's next event
    after any moment without ever returning an event of somebody else as a near one. int64 [E]
    """
    seconds_since_first_event = timestamp - timestamp.min()
    starts_new_user = np.concatenate([[False], user_id[1:] != user_id[:-1]])
    seconds_between_users = int(seconds_since_first_event.max()) + longest_wait + 1
    return np.cumsum(starts_new_user).astype(np.int64) * seconds_between_users + seconds_since_first_event


def _temporal_split(events: pl.DataFrame, config: PreprocessingConfig) -> TemporalSplit:
    last_timestamp = int(events["timestamp"].max())
    test_start = last_timestamp - round(config.test_days * SECONDS_PER_DAY)
    validation_start = test_start - round(config.validation_days * SECONDS_PER_DAY)
    return TemporalSplit(validation_start=validation_start, test_start=test_start)


def _item_catalogue(train_events: pl.DataFrame, config: PreprocessingConfig) -> pl.DataFrame:
    """One row per kept item, most popular first; `token` is its 1-based popularity rank."""
    return (
        train_events.group_by("item_id")
        .agg(
            train_popularity=pl.len(),
            duration_seconds=pl.col("duration_seconds").median(),
            mean_played_ratio=pl.col("played_ratio").mean(),
        )
        .filter(pl.col("train_popularity") >= config.min_item_events)
        .sort(["train_popularity", "item_id"], descending=[True, False])
        .head(config.max_items)
        .with_columns(pl.col("duration_seconds").fill_null(pl.col("duration_seconds").median()).fill_null(0.0))
        .with_row_index("token", offset=1)
    )


def _item_table(catalogue: pl.DataFrame) -> ItemTable:
    def with_padding_row(column: str, dtype: type) -> np.ndarray:
        values = catalogue[column].to_numpy().astype(dtype)
        return np.concatenate([np.zeros(1, dtype), values])

    return ItemTable(
        raw_id=with_padding_row("item_id", np.int64),
        train_popularity=with_padding_row("train_popularity", np.int64),
        duration_seconds=with_padding_row("duration_seconds", np.float32),
        mean_played_ratio=with_padding_row("mean_played_ratio", np.float32),
    )

In [ ]:
%%writefile /kaggle/working/src/gptrec/data/yambda.py
"""Adapter for Yambda, the Yandex Music listening log (https://huggingface.co/datasets/yandex/yambda).

The listen events are the interactions: each has the played share of the track, the track length and a flag
telling whether the user chose the track themselves or a recommender served it. Likes and dislikes are
separate events; they become labels of the listens they follow (see `gptrec.data.preprocess`).
"""

from __future__ import annotations

from pathlib import Path

import polars as pl
from huggingface_hub import hf_hub_download

from gptrec.paths import RAW_DATA_DIR

REPOSITORY = "yandex/yambda"
SIZES = ("50m", "500m", "5b")
REACTIONS = ("likes", "dislikes")


def download_listens(size: str = "50m", directory: Path = RAW_DATA_DIR) -> Path:
    """Path of the listens table of the given dataset size, downloaded from the Hugging Face Hub on first use."""
    return _downloaded("listens", size, directory)


def download_reactions(size: str = "50m", directory: Path = RAW_DATA_DIR) -> dict[str, Path]:
    """Paths of the likes and the dislikes table of the given dataset size, downloaded on first use."""
    return {reaction: _downloaded(reaction, size, directory) for reaction in REACTIONS}


def _downloaded(table: str, size: str, directory: Path) -> Path:
    if size not in SIZES:
        raise ValueError(f"unknown Yambda size {size!r}; expected one of {SIZES}")
    return Path(hf_hub_download(repo_id=REPOSITORY, repo_type="dataset", filename=f"flat/{size}/{table}.parquet", local_dir=directory / "yambda"))


def load_raw_events(listens_path: str | Path) -> pl.DataFrame:
    """The listens table in the raw-event format of `gptrec.data.preprocess`.

    Yambda timestamps are seconds since the first event of the dataset, rounded to 5 seconds.
    `played_ratio_pct` can exceed 100 when a track is replayed, so the ratio is capped at 1.
    `is_organic` is 1 for the user's own choices and 0 for the personalised feed and playlists.
    """
    return pl.read_parquet(listens_path).drop_nulls().select(
        user_id=pl.col("uid").cast(pl.Int64),
        item_id=pl.col("item_id").cast(pl.Int64),
        timestamp=pl.col("timestamp").cast(pl.Int64),
        played_ratio=(pl.col("played_ratio_pct").cast(pl.Float32) / 100.0).clip(0.0, 1.0),
        duration_seconds=pl.col("track_length_seconds").cast(pl.Float32),
        is_organic=pl.col("is_organic") == 1,
    )


def load_reactions(reaction_paths: dict[str, str | Path]) -> pl.DataFrame:
    """Likes and dislikes in the reaction format of `gptrec.data.preprocess`, on the clock of the listens."""
    return pl.concat([
        pl.read_parquet(reaction_paths[reaction]).drop_nulls(["uid", "item_id", "timestamp"]).select(
            user_id=pl.col("uid").cast(pl.Int64),
            item_id=pl.col("item_id").cast(pl.Int64),
            timestamp=pl.col("timestamp").cast(pl.Int64),
            liked=pl.lit(reaction == "likes"),
        )
        for reaction in REACTIONS
    ])

In [ ]:
%%writefile /kaggle/working/src/gptrec/decoding.py
"""Steering the base recommender at inference with the closed form of the anchored objective: no fine-tuning.

For a reward r and the anchor beta * KL(pi || pi_0), the best policy at one position of a list is

    pi*(a | s) = pi_0(a | s) * exp(Q(s, a) / beta) / Z(s),        Q(s, a) = r(s, a) + discount * U(s, a),

where U(s, a) is the discounted reward of the items that follow a. Re-ranking by r + beta * log pi_0 takes the top
of pi* with U = 0 and every candidate judged once, after the real history. `guided_lists` decides one position at
a time, after the items already chosen, and can look ahead with a U learned from the base recommender's own lists:
the reference policy's value, as in controlled decoding (Mudgal et al., 2024) and as the first step of Q-sharp
(Zhou et al., 2025), which iterates it towards the value of pi* itself. The recommender's weights never change.
"""

from __future__ import annotations

from dataclasses import dataclass, field

import torch
import torch.nn.functional as F
from torch import Tensor
from tqdm.auto import tqdm

from gptrec.data.batches import HistoryBatch, TrainingHistorySampler
from gptrec.data.log import PAD_ITEM, InteractionLog
from gptrec.models.policy import NextItemPolicy
from gptrec.models.reward_model import RewardModel
from gptrec.models.value_model import ContinuationValueModel
from gptrec.rollout import (
    GenerationConfig,
    RewardFunction,
    RewardSpec,
    SimulatedFeedback,
    append_actions,
    constrain_logits,
    generate_actions,
    simulate_feedback,
)
from gptrec.training.precision import autocast, clipped_optimizer_step, gradient_scaler, seed_everything
from gptrec.training.rl import PROGRESS_REFRESH_SECONDS, Metrics, discounted_returns


@dataclass(frozen=True)
class ContinuationTrainingConfig:
    """How U is learned: by regression on lists the base recommender samples for random moments of the train period."""

    num_steps: int = 1000
    histories_per_step: int = 512
    history_len: int = 118
    min_history: int = 8
    max_future: int = 20
    generation: GenerationConfig = field(default_factory=GenerationConfig)
    simulated_feedback: SimulatedFeedback = SimulatedFeedback.EXPECTED
    discount: float = 0.9
    learning_rate: float = 1e-4
    max_grad_norm: float = 1.0
    mixed_precision: bool = False
    seed: int = 0


@dataclass(frozen=True)
class GuidanceConfig:
    """How a list is steered at inference."""

    strength: float = 1.0
    discount: float = 0.0
    num_candidates: int = 50


def fit_continuation_values(
    reference: NextItemPolicy,
    reward_model: RewardModel,
    log: InteractionLog,
    reward: RewardSpec,
    config: ContinuationTrainingConfig,
    device: torch.device,
) -> tuple[ContinuationValueModel, list[Metrics]]:
    """Learns U for `reward`: the discounted reward of the items the base recommender samples after each item, as the reward model scores them."""
    require_guidable(reward)
    seed_everything(config.seed)
    reference = reference.to(device).eval().requires_grad_(False)
    reward_model = reward_model.to(device).eval()
    reward_function = RewardFunction(reward, log.items, device)
    value_model = ContinuationValueModel.from_policy(reference, config.generation.horizon).to(device).eval()
    optimizer = torch.optim.AdamW(value_model.parameters(), lr=config.learning_rate, weight_decay=0.0)
    scaler = gradient_scaler(device, config.mixed_precision)
    moments = TrainingHistorySampler(log, config.history_len, config.max_future, config.min_history, seed=config.seed)

    records: list[Metrics] = []
    for step in tqdm(range(1, config.num_steps + 1), desc="continuation values", mininterval=PROGRESS_REFRESH_SECONDS):
        history = moments.sample(config.histories_per_step).history.to(device)
        with torch.no_grad(), autocast(device, config.mixed_precision):
            actions = generate_actions(reference, history, config.generation).actions
            feedback = simulate_feedback(reward_model, history, actions, config.simulated_feedback)
            targets = continuation_targets(reward_function(feedback, actions), config.discount)
        with autocast(device, config.mixed_precision):
            predicted = continuation_values(value_model, history, actions)
            loss = F.mse_loss(predicted, targets)
        clipped_optimizer_step(loss, optimizer, scaler, value_model.parameters(), config.max_grad_norm)
        unexplained = (targets - predicted.detach()).var() / targets.var().clamp_min(1e-8)
        records.append({"step": step, "loss": loss.item(), "explained_variance": 1.0 - unexplained.item(), "mean_target": targets.mean().item()})
    return value_model.eval().requires_grad_(False), records


def continuation_targets(rewards: Tensor, discount: float) -> Tensor:
    """U after every item of the lists: the discounted reward of the items after it, 0 after the last. [B, K]"""
    returns = discounted_returns(rewards, discount)
    return torch.cat([returns[:, 1:], torch.zeros_like(returns[:, :1])], dim=1)


def continuation_values(value_model: ContinuationValueModel, history: HistoryBatch, actions: Tensor) -> Tensor:
    """The value model's U after every item of the lists. actions: long [B, K] -> [B, K]"""
    horizon = actions.shape[1]
    positions = history.lengths[:, None] + torch.arange(horizon, device=actions.device)[None, :]
    items = append_actions(history, actions)
    list_ranks = torch.arange(1, horizon + 1, device=actions.device).repeat(len(actions), 1)
    list_positions = torch.zeros_like(items).scatter(1, positions, list_ranks)
    return value_model(items, list_positions).gather(1, positions)


@torch.no_grad()
def guided_lists(
    reference: NextItemPolicy,
    reward_model: RewardModel,
    reward_function: RewardFunction,
    value_model: ContinuationValueModel | None,
    history: HistoryBatch,
    guidance: GuidanceConfig,
    generation: GenerationConfig,
) -> Tensor:
    """Lists built one position at a time, each taking the candidate with the highest log pi_0 + Q / strength. long [B, K]

    Q is the candidate's reward as the reward model predicts it after the history and the items already chosen,
    whose expected responses are fed back as in `simulate_feedback`, plus `guidance.discount` times U from
    `value_model`. The item constraints are those of `generation`; its sampling settings play no part.
    """
    require_guidable(reward_function.spec)
    if guidance.discount > 0 and value_model is None:
        raise ValueError("looking ahead needs a value model of what follows each item")
    horizon, num_candidates = generation.horizon, guidance.num_candidates
    num_users, width = history.items.shape
    users = torch.arange(num_users, device=history.items.device)
    trials = torch.arange(num_users * num_candidates, device=history.items.device)
    items = F.pad(history.items, (0, horizon), value=PAD_ITEM)
    played_ratio = F.pad(history.played_ratio, (0, horizon), value=0.0)
    list_positions = torch.zeros_like(items)
    chosen = history.items[:, :0]

    for step in range(horizon):
        position = history.lengths + step
        visible = width + step + 1
        state = reference.hidden_states(items[:, : width + step])[users, position - 1]
        logits = constrain_logits(reference.item_logits(state), history.items, chosen, generation)
        candidate_log_probs, candidates = F.log_softmax(logits.float(), dim=-1).topk(num_candidates, dim=-1)

        trial_position = position.repeat_interleave(num_candidates)
        trial_items = items[:, :visible].repeat_interleave(num_candidates, dim=0)
        trial_items[trials, trial_position] = candidates.reshape(-1)
        predicted = reward_model.predict(trial_items, played_ratio[:, :visible].repeat_interleave(num_candidates, dim=0))
        feedback = predicted.map(lambda values: values[trials, trial_position].reshape(num_users, num_candidates))
        q_value = reward_function.item_reward(feedback, candidates)
        if step == horizon - 1 and reward_function.spec.rewards_the_end_of_the_list:
            q_value = q_value + reward_function.spec.final_returns_weight * feedback.returns
        if guidance.discount > 0:
            trial_list_positions = list_positions[:, :visible].repeat_interleave(num_candidates, dim=0)
            trial_list_positions[trials, trial_position] = step + 1
            continuation = value_model(trial_items, trial_list_positions)[trials, trial_position].reshape(num_users, num_candidates)
            q_value = q_value + guidance.discount * continuation

        best = (candidate_log_probs + q_value.float() / guidance.strength).argmax(dim=1)
        action = candidates[users, best]
        items[users, position] = action
        list_positions[users, position] = step + 1
        played_ratio[users, position] = feedback.played_ratio[users, best].to(played_ratio.dtype)
        chosen = torch.cat([chosen, action[:, None]], dim=1)
    return chosen


def require_guidable(reward: RewardSpec) -> None:
    """Guided decoding judges each candidate for its own user, from predictions alone, one position at a time."""
    if reward.needs_other_user or reward.needs_logged_future or reward.depends_on_the_list:
        raise ValueError("a reward measured against another user, read from the logged future, or of the list as a whole cannot guide decoding")

In [ ]:
%%writefile /kaggle/working/src/gptrec/evaluation/analyses.py
"""Analyses of fine-tuned recommenders that only evaluate: along the list, across users, across items.

Each answers one question about a policy that the headline metrics leave open: what an item the user never
consumed does to the rest of a generated list; whether a simulation can trust its assumption about such items;
who loses relevance; whether the policy learned something about users or one preference per item, and whether
the reward predicted which; how far a policy moved along its own lists; and what a retention label records.
"""

from __future__ import annotations

from typing import Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import Tensor

from gptrec.data.batches import EvalExamples
from gptrec.data.log import PAD_ITEM, InteractionLog, ItemTable
from gptrec.evaluation.logged_feedback import logged_reward_per_user
from gptrec.evaluation.relevance import contains, per_user_ranking_metrics
from gptrec.evaluation.reward_model_metrics import COMPLETED_THRESHOLD, roc_auc
from gptrec.models.policy import NextItemPolicy
from gptrec.models.reward_model import RewardModel
from gptrec.rollout import GenerationConfig, SimulatedFeedback, action_log_distributions, simulate_feedback
from gptrec.training.precision import autocast


def position_hits(lists: dict[str, Tensor], future_items: Tensor) -> pd.DataFrame:
    """Per position of each list: how often the item is among the user's next items, overall, after a miss and after a hit.

    lists: name -> long [U, K]. In a generated list, a miss at position k - 1 means that the item at position k was
    chosen after an item the user never consumed. A ranked list has no such context, so the gap between the two
    after a miss is the cost of continuing from a context that did not happen. Columns: <list>/<statistic>.
    """
    columns = {}
    future_items = future_items.cpu()
    for name, items in lists.items():
        hit = contains(items.cpu(), future_items).float()
        before, now = hit[:, :-1], hit[:, 1:]
        no_previous_position = torch.tensor([float("nan")])
        columns[f"{name}/hit_rate"] = hit.mean(dim=0)
        columns[f"{name}/hit_rate_after_miss"] = torch.cat([no_previous_position, _conditional_rate(now, 1.0 - before)])
        columns[f"{name}/hit_rate_after_hit"] = torch.cat([no_previous_position, _conditional_rate(now, before)])
    position = pd.RangeIndex(1, next(iter(lists.values())).shape[1] + 1, name="position")
    return pd.DataFrame({name: values.numpy() for name, values in columns.items()}, index=position)


def _conditional_rate(event: Tensor, condition: Tensor) -> Tensor:
    """Per column, the mean of `event` over the rows where `condition` holds; NaN where it never does."""
    num_conditioned = condition.sum(dim=0)
    rate = (event * condition).sum(dim=0) / num_conditioned.clamp_min(1)
    return rate.masked_fill(num_conditioned == 0, float("nan"))


@torch.no_grad()
def feedback_assumptions_on_real_sequences(
    reward_model: RewardModel,
    examples: EvalExamples,
    modes: Sequence[SimulatedFeedback],
    steps: int,
    batch_size: int,
    device: torch.device,
    mixed_precision: bool = False,
) -> pd.DataFrame:
    """How well the reward model predicts the response to a user's real next items when the earlier responses are assumed.

    For every request with at least `steps` next events, the list is the user's real next `steps` items in their
    order. The model predicts the response to each one from the history and the items before it, fed back as
    `mode` says: as logged, or as one of the assumptions a simulation of generated items has to make. Where an
    assumption's predictions at later steps agree with what users did, a simulation can rely on it.
    Rows: step. Columns: logged/mean_played_ratio, and <mode>/<mean_predicted, bias, mae, auc>.
    """
    has_enough_future = (examples.future_items[:, steps - 1] != PAD_ITEM).nonzero().squeeze(1)
    if len(has_enough_future) == 0:
        return pd.DataFrame(index=pd.RangeIndex(1, steps + 1, name="step"))
    requests = examples[has_enough_future]
    predicted: dict[SimulatedFeedback, list[Tensor]] = {SimulatedFeedback(mode): [] for mode in modes}
    for batch in requests.batches(batch_size):
        history = batch.history.to(device)
        next_items, next_played_ratio = batch.future_items[:, :steps].to(device), batch.future_played_ratio[:, :steps].to(device)
        with autocast(device, mixed_precision):
            for mode, parts in predicted.items():
                parts.append(simulate_feedback(reward_model, history, next_items, mode, next_played_ratio).played_ratio.float().cpu())
    logged = requests.future_played_ratio[:, :steps].float()

    columns = {"logged/mean_played_ratio": logged.mean(dim=0).numpy(), "logged/num_requests": np.full(steps, float(len(requests)))}
    for mode, parts in predicted.items():
        values = torch.cat(parts)
        columns[f"{mode.value}/mean_predicted"] = values.mean(dim=0).numpy()
        columns[f"{mode.value}/bias"] = (values - logged).mean(dim=0).numpy()
        columns[f"{mode.value}/mae"] = (values - logged).abs().mean(dim=0).numpy()
        columns[f"{mode.value}/auc"] = np.array([roc_auc(logged[:, step].numpy() >= COMPLETED_THRESHOLD, values[:, step].numpy()) for step in range(steps)])
    return pd.DataFrame(columns, index=pd.RangeIndex(1, steps + 1, name="step"))


def relevance_by_history_length(
    lists: dict[str, Tensor], examples: EvalExamples, prior_events: np.ndarray, k: int, liked_threshold: float, num_groups: int = 4
) -> pd.DataFrame:
    """NDCG@k and the logged share per recommended item of each list, for users grouped by how much history they had.

    prior_events: [U], each user's number of events before the request; the groups are its quantiles.
    Rows: group. Columns: num_users and <list>/<metric>.
    """
    edges = np.unique(np.quantile(prior_events, np.linspace(0.0, 1.0, num_groups + 1)))
    group = np.clip(np.searchsorted(edges, prior_events, side="right") - 1, 0, len(edges) - 2)
    per_list = {
        name: (
            per_user_ranking_metrics(items.cpu(), examples.future_items.cpu(), [k])[f"ndcg@{k}"].numpy(),
            logged_reward_per_user(items, examples, liked_threshold)["played_ratio_per_item"].numpy(),
        )
        for name, items in lists.items()
    }
    rows = {}
    for index in range(len(edges) - 1):
        members = group == index
        if not members.any():
            continue
        row = {"num_users": float(members.sum())}
        for name, (ndcg, logged_share) in per_list.items():
            row[f"{name}/ndcg@{k}"] = float(np.nanmean(ndcg[members]))
            row[f"{name}/logged_played_ratio_per_item"] = float(logged_share[members].mean())
        rows[f"{int(edges[index])} to {int(edges[index + 1])} events"] = row
    return pd.DataFrame(rows).T


@torch.no_grad()
def logit_shift_by_item(
    policy: NextItemPolicy, reference: NextItemPolicy, examples: EvalExamples, items: ItemTable, batch_size: int, device: torch.device, max_requests: int = 2048
) -> dict[str, float]:
    """How much of what fine-tuning changed in the next-item distribution is one constant per item, the same for every user.

    For each history the shift of item j is log pi(j | h) - log pi0(j | h). If the policy learned "show more of
    item j" regardless of the user, every history shifts j alike, and the mean shift per item explains all of
    it. `r2_per_item` is that share of the variance, weighted by pi0 so that items no user would be shown do not
    count; `r2_per_item_unweighted` gives every item the same weight. The correlations relate the per-item shift
    to the item's mean played ratio, log popularity and duration, across items weighted by pi0's mass.
    """
    num_items = items.num_items
    weighted = {name: torch.zeros(num_items, dtype=torch.float64, device=device) for name in ("weight", "shift", "squared_shift")}
    unweighted = {name: torch.zeros(num_items, dtype=torch.float64, device=device) for name in ("count", "shift", "squared_shift")}
    for batch in examples[:max_requests].batches(batch_size):
        history = batch.history.to(device)
        log_probs = F.log_softmax(policy.next_item_logits(history.items, history.lengths)[:, 1:].float(), dim=-1).double()
        reference_log_probs = F.log_softmax(reference.next_item_logits(history.items, history.lengths)[:, 1:].float(), dim=-1).double()
        shift, weight = log_probs - reference_log_probs, reference_log_probs.exp()
        weighted["weight"] += weight.sum(dim=0)
        weighted["shift"] += (weight * shift).sum(dim=0)
        weighted["squared_shift"] += (weight * shift**2).sum(dim=0)
        unweighted["count"] += len(shift)
        unweighted["shift"] += shift.sum(dim=0)
        unweighted["squared_shift"] += (shift**2).sum(dim=0)

    def share_explained_by_item(weight: Tensor, shift: Tensor, squared_shift: Tensor) -> float:
        total = squared_shift.sum() - shift.sum() ** 2 / weight.sum()
        within_items = (squared_shift - shift**2 / weight.clamp_min(1e-300)).sum()
        return float(1.0 - within_items / total)

    mass = weighted["weight"].cpu().numpy()
    shift_per_item = (weighted["shift"] / weighted["weight"].clamp_min(1e-300)).cpu().numpy()
    attributes = {
        "mean_played_ratio": items.mean_played_ratio[1:].astype(np.float64),
        "log_popularity": np.log1p(items.train_popularity[1:].astype(np.float64)),
        "duration_seconds": items.duration_seconds[1:].astype(np.float64),
    }
    return {
        "r2_per_item": share_explained_by_item(weighted["weight"], weighted["shift"], weighted["squared_shift"]),
        "r2_per_item_unweighted": share_explained_by_item(unweighted["count"], unweighted["shift"], unweighted["squared_shift"]),
        "mean_shift": float((weighted["shift"].sum() / weighted["weight"].sum()).item()),
    } | {f"correlation_with_{name}": _weighted_correlation(shift_per_item, values, mass) for name, values in attributes.items()}


def _weighted_correlation(first: np.ndarray, second: np.ndarray, weight: np.ndarray) -> float:
    weight = weight / weight.sum()
    first_centred, second_centred = first - (weight * first).sum(), second - (weight * second).sum()
    covariance = (weight * first_centred * second_centred).sum()
    return float(covariance / np.sqrt((weight * first_centred**2).sum() * (weight * second_centred**2).sum()))


def exposure_lorenz(lists: dict[str, Tensor], num_items: int, num_points: int = 101) -> pd.DataFrame:
    """Lorenz curves of impressions over the catalogue: the share of all impressions that goes to the least recommended share of items.

    A diagonal means every item is shown equally often; a curve that stays at zero and jumps at the end means
    a few items take everything. Rows: share of items. Columns: lists.
    """
    share_of_items = np.linspace(0.0, 1.0, num_points)
    at_share = np.round(share_of_items * num_items).astype(int)
    curves = {}
    for name, items in lists.items():
        impressions = torch.bincount(items.flatten().cpu(), minlength=num_items + 1)[1:].double().sort().values
        cumulative = torch.cat([torch.zeros(1, dtype=torch.float64), impressions.cumsum(0)]) / impressions.sum().clamp_min(1)
        curves[name] = cumulative.numpy()[at_share]
    return pd.DataFrame(curves, index=pd.Index(share_of_items, name="share of items"))


def impressions_by_attribute(lists: dict[str, Tensor], attribute_of_item: np.ndarray) -> pd.DataFrame:
    """Share of impressions per value of an item attribute, such as film or series. attribute_of_item: [V+1]. Columns: lists."""
    shares = {}
    for name, items in lists.items():
        values = pd.Series(attribute_of_item[items.flatten().cpu().numpy()])
        shares[name] = values.value_counts(normalize=True, dropna=False)
    return pd.DataFrame(shares).fillna(0.0)


def largest_impression_changes(before: Tensor, after: Tensor, item_table: pd.DataFrame, num_items: int = 10) -> pd.DataFrame:
    """The items whose impressions grew most and shrank most from the lists `before` to the lists `after`.

    item_table: one row per item token (index) with whatever describes the items. Both lists: long [U, K].
    """
    size = len(item_table)
    impressions_before = torch.bincount(before.flatten().cpu(), minlength=size)[:size].numpy()
    impressions_after = torch.bincount(after.flatten().cpu(), minlength=size)[:size].numpy()
    table = item_table.assign(impressions_before=impressions_before, impressions_after=impressions_after)
    table = table.assign(change=table["impressions_after"] - table["impressions_before"]).drop(index=PAD_ITEM, errors="ignore")
    gained, lost = table.nlargest(num_items, "change"), table.nsmallest(num_items, "change")
    return pd.concat({"gained": gained, "lost": lost}, names=["direction", "item_token"])


@torch.no_grad()
def mean_logit_shift_per_item(
    policy: NextItemPolicy, reference: NextItemPolicy, examples: EvalExamples, batch_size: int, device: torch.device, max_requests: int = 4096
) -> Tensor:
    """Per item, the mean over histories of log pi(j | h) - log pi_0(j | h), weighted by pi_0(j | h). float [V+1], 0 for padding, on the CPU.

    Adding it to the base recommender's scores gives the part of a fine-tuning that is the same for every user;
    subtracting it from the fine-tuned recommender's scores leaves the part that depends on who receives the list.
    """
    weight_sum = shift_sum = 0.0
    for batch in examples[:max_requests].batches(batch_size):
        history = batch.history.to(device)
        log_probs = F.log_softmax(policy.next_item_logits(history.items, history.lengths).float(), dim=-1).double()
        reference_log_probs = F.log_softmax(reference.next_item_logits(history.items, history.lengths).float(), dim=-1).double()
        weight = reference_log_probs.exp()
        weight_sum = weight_sum + weight.sum(dim=0)
        shift_sum = shift_sum + (weight * (log_probs - reference_log_probs)).sum(dim=0)
    offsets = (shift_sum / weight_sum.clamp_min(1e-300)).float().cpu()
    offsets[PAD_ITEM] = 0.0
    return offsets


def item_effect_share(values: Tensor, items: Tensor, weights: Tensor) -> float:
    """Share of the within-history variation of `values` that one effect per item explains. values, items, weights: [U, N].

    values[u, n] belongs to item items[u, n] for history u, and weights[u, n] says how much the pair counts, for
    instance the base recommender's probability of the item. Values are centred within each history, so a constant
    per history, such as the normaliser of a softmax, explains nothing; the effect of an item is the weighted mean
    of its centred values. 1: every history orders the items alike; 0: an item's worth depends on the history alone.
    """
    values, weights = values.double().cpu(), weights.double().cpu()
    items = items.cpu()
    history_mean = (weights * values).sum(dim=1, keepdim=True) / weights.sum(dim=1, keepdim=True).clamp_min(1e-300)
    centred = values - history_mean
    size = int(items.max()) + 1
    weight_per_item = torch.zeros(size, dtype=torch.float64).index_add_(0, items.flatten(), weights.flatten())
    effect = torch.zeros(size, dtype=torch.float64).index_add_(0, items.flatten(), (weights * centred).flatten()) / weight_per_item.clamp_min(1e-300)
    residual = centred - effect[items]
    return float(1.0 - (weights * residual**2).sum() / (weights * centred**2).sum().clamp_min(1e-300))


@torch.no_grad()
def candidate_log_probs(policy: NextItemPolicy, examples: EvalExamples, candidates: Tensor, batch_size: int, device: torch.device) -> Tensor:
    """log pi(item | history) of each user's candidate items, before any list constraint. candidates: long [U, N] -> float [U, N], on the CPU."""
    parts = []
    for batch, batch_candidates in zip(examples.batches(batch_size), candidates.split(batch_size)):
        history = batch.history.to(device)
        log_probs = F.log_softmax(policy.next_item_logits(history.items, history.lengths).float(), dim=-1)
        parts.append(log_probs.gather(1, batch_candidates.to(device)).cpu())
    return torch.cat(parts)


@torch.no_grad()
def kl_along_lists(
    policy: NextItemPolicy,
    reference: NextItemPolicy,
    examples: EvalExamples,
    lists: Tensor,
    generation: GenerationConfig,
    batch_size: int,
    device: torch.device,
    max_requests: int = 4096,
) -> float:
    """Mean KL(pi || pi_0) per item over the states in which the given lists were built, under the list constraints. lists: long [U, K]

    One yardstick for every recommender, however it was obtained: a fine-tuning run reports its KL on the lists it
    sampled while training, a recommender made by interpolating weights has no such record.
    """
    total, count = 0.0, 0
    for batch, batch_lists in zip(examples[:max_requests].batches(batch_size), lists[:max_requests].split(batch_size)):
        history, actions = batch.history.to(device), batch_lists.to(device)
        log_distributions = action_log_distributions(policy, history, actions, generation)
        reference_log_distributions = action_log_distributions(reference, history, actions, generation)
        kl = (log_distributions.exp() * (log_distributions - reference_log_distributions)).sum(dim=-1)
        total += kl.sum().item()
        count += kl.numel()
    return total / max(1, count)


def return_rate_by_duration(log: InteractionLog, num_groups: int = 10) -> pd.DataFrame:
    """How often a user returns after an event, by the duration of the event's own item and by that of the next item the user consumed.

    Training-period events with a known return outcome and a later event of the same user. Where a log keeps one row
    per user and item, dated by its last view, a user who moves on to a long series leaves no row until the series is
    finished, and the event before it looks as if the user stayed away. If the label follows the next item's duration
    more than the event's own, a reward for returns pays for that bookkeeping. Rows: duration groups, shortest first.
    """
    user_starts, user_ends = np.asarray(log.user_offsets[:-1]), np.asarray(log.user_offsets[1:])
    train_starts, train_ends = log.period_rows("train")
    boundaries = np.zeros(log.num_events + 1, dtype=np.int64)
    np.add.at(boundaries, train_starts, 1)
    np.add.at(boundaries, train_ends, -1)
    in_train = np.cumsum(boundaries)[:-1] > 0
    has_next_event = np.ones(log.num_events, dtype=bool)
    has_next_event[user_ends[user_ends > user_starts] - 1] = False
    events = np.flatnonzero(in_train & has_next_event & np.asarray(log.returns_known))
    duration = np.asarray(log.items.duration_seconds, dtype=np.float64)
    item = np.asarray(log.item)
    returned = np.asarray(log.returns)[events].astype(np.float64)
    columns, correlation = {}, {}
    for name, item_of_event in (("own item", item[events]), ("next item", item[events + 1])):
        hours = duration[item_of_event] / 3600
        edges = np.unique(np.quantile(hours, np.linspace(0.0, 1.0, num_groups + 1)))
        group = np.clip(np.searchsorted(edges, hours, side="right") - 1, 0, len(edges) - 2)
        groups = np.arange(len(edges) - 1)
        columns[f"{name}: mean hours"] = pd.Series([hours[group == g].mean() for g in groups], index=groups + 1)
        columns[f"{name}: return rate"] = pd.Series([returned[group == g].mean() for g in groups], index=groups + 1)
        correlation[f"{name}: return rate"] = np.corrcoef(np.log1p(hours), returned)[0, 1]
    table = pd.DataFrame(columns)
    table.loc["correlation of log hours with return"] = pd.Series(correlation)
    return table.rename_axis("duration group")

In [ ]:
%%writefile /kaggle/working/src/gptrec/evaluation/behavior.py
"""What a recommender recommends, independent of whether it is relevant.

These statistics expose the usual ways of gaming a completion-rate reward: collapsing onto a few
items, drifting to popular or short items, and re-recommending what the user already consumed.
"""

from __future__ import annotations

import torch
from torch import Tensor

from gptrec.data.log import ItemTable
from gptrec.evaluation.relevance import contains, first_occurrence

HEAD_FRACTION = 0.01
LISTS_PER_PASS = 4096


def recommendation_profile(recommended: Tensor, history_items: Tensor, items: ItemTable) -> dict[str, float]:
    """Summary statistics of the lists shown to all users.

    recommended: long [U, N], history_items: long [U, H] padded with PAD_ITEM.
    """
    recommended, history_items = recommended.cpu(), history_items.cpu()
    num_users, list_length = recommended.shape
    is_first = first_occurrence(recommended)
    users_per_item = torch.bincount(recommended[is_first], minlength=items.num_items + 1)[1:].double()

    popularity = torch.as_tensor(items.train_popularity, dtype=torch.double)
    head_size = max(1, round(HEAD_FRACTION * items.num_items))
    head_threshold = popularity[1:].topk(head_size).values.min()

    most_recommended = users_per_item.topk(max(1, round(HEAD_FRACTION * items.num_items))).values
    return {
        "catalog_coverage": (users_per_item > 0).double().mean().item(),
        "item_gini": _gini(users_per_item),
        "top_exposure_share": (most_recommended.sum() / users_per_item.sum().clamp_min(1)).item(),
        "top_item_user_share": (users_per_item.max() / num_users).item(),
        "mean_user_pair_overlap": _mean_user_pair_overlap(users_per_item, num_users, list_length),
        "duplicate_rate": 1.0 - is_first.double().mean().item(),
        "history_repeat_rate": contains(recommended, history_items).double().mean().item(),
        "mean_popularity": popularity[recommended].mean().item(),
        "head_item_share": (popularity[recommended] >= head_threshold).double().mean().item(),
        "mean_duration_seconds": torch.as_tensor(items.duration_seconds)[recommended].double().mean().item(),
        "mean_item_played_ratio": torch.as_tensor(items.mean_played_ratio)[recommended].double().mean().item(),
    }


def pairwise_item_distances(lists: Tensor, unit_item_embeddings: Tensor) -> Tensor:
    """Cosine distance between every two items of each list. [B, K, K]

    lists: long [B, K]; unit_item_embeddings: [V+1, d], one row of unit length per item token.
    """
    embedded = unit_item_embeddings[lists]
    return 1.0 - (embedded @ embedded.transpose(1, 2)).float()


def intra_list_distance(lists: Tensor, unit_item_embeddings: Tensor) -> Tensor:
    """Mean cosine distance between two different items of each list: how varied the list is. float [U]"""
    list_length = lists.shape[1]
    if list_length < 2:
        return torch.zeros(len(lists))
    is_other_item = ~torch.eye(list_length, dtype=torch.bool, device=lists.device)
    num_pairs = list_length * (list_length - 1)
    distances = [(pairwise_item_distances(part, unit_item_embeddings) * is_other_item).sum(dim=(1, 2)) / num_pairs for part in lists.split(LISTS_PER_PASS)]
    return torch.cat(distances)


def list_overlap(lists: Tensor, other_lists: Tensor) -> float:
    """Average share of a user's list that also appears in the same user's other list."""
    return contains(lists.cpu(), other_lists.cpu()).double().mean().item()


def _gini(counts: Tensor) -> float:
    """Inequality of how often each item is recommended: 0 = all items equally often, 1 = a single item."""
    ascending = counts.sort().values
    num_items = len(ascending)
    rank = torch.arange(1, num_items + 1, dtype=torch.double)
    return (((2 * rank - num_items - 1) * ascending).sum() / (num_items * ascending.sum())).item()


def _mean_user_pair_overlap(users_per_item: Tensor, num_users: int, list_length: int) -> float:
    """Expected share of items two random users have in common: 0 = fully personalised, 1 = everyone gets the same list."""
    if num_users < 2:
        return float("nan")
    user_pairs_sharing_item = users_per_item * (users_per_item - 1) / 2
    num_user_pairs = num_users * (num_users - 1) / 2
    return (user_pairs_sharing_item.sum() / (num_user_pairs * list_length)).item()

In [ ]:
%%writefile /kaggle/working/src/gptrec/evaluation/logged_feedback.py
"""What users really did with the recommended items they went on to consume.

Every other reward in the study is a prediction of a reward model. A recommended item that is among the
user's next events - a hit - comes with the response the log recorded, and no model stands between that
response and the recommender. Hits are a selected sample of the recommendations, so these numbers say how
the items the user did reach were received, not how the whole list would have been.
"""

from __future__ import annotations

import torch
from torch import Tensor

from gptrec.data.batches import EvalExamples
from gptrec.data.log import PAD_ITEM
from gptrec.evaluation.relevance import first_occurrence

USUAL_RELATIVE_RATIO = 0.5


def logged_reception(lists: Tensor, examples: EvalExamples, liked_threshold: float) -> dict[str, float]:
    """The logged response to the hits of all lists together. lists: long [U, K], row u is for examples[u].

    hits_per_list: listed items that are among the user's next events.
    played_ratio_of_hits, liked_share_of_hits: how much of them the user consumed, and how often at least `liked_threshold`.
    relative_ratio_of_hits: the same response as a quantile inside the item; above 0.5 the user took to the
        recommended items more than people usually take to those items, however easy the items are to finish.
    played_ratio_per_item, relative_surplus_per_item: the same two per recommended item, an item that was not hit counting as nothing.
    liked_per_item, adopted_per_item: recommended items the user went on to like, or that were new to the user and that
        the user kept coming back to, per recommended item; zero where the log records neither.
    served_hits_per_list, played_ratio_of_served_hits: only the next events that a recommender served, where the log says which.
    """
    per_user = hits_per_user(lists, examples, liked_threshold)
    total = {name: values.sum().item() for name, values in per_user.items()}
    reward = _reward_per_item(per_user, list_length=lists.shape[1])

    def mean_over(hits: str, summed: str) -> float:
        return total[summed] / total[hits] if total[hits] > 0 else float("nan")

    return {
        "hits_per_list": total["hits"] / len(lists),
        "played_ratio_of_hits": mean_over("hits", "played_ratio"),
        "liked_share_of_hits": mean_over("hits", "liked_hits"),
        "relative_ratio_of_hits": mean_over("hits", "relative_ratio"),
        "played_ratio_per_item": reward["played_ratio_per_item"].mean().item(),
        "relative_surplus_per_item": reward["relative_surplus_per_item"].mean().item(),
        "liked_per_item": reward["liked_per_item"].mean().item(),
        "adopted_per_item": reward["adopted_per_item"].mean().item(),
        "served_hits_per_list": total["served_hits"] / len(lists),
        "played_ratio_of_served_hits": mean_over("served_hits", "served_played_ratio"),
    }


def logged_reward_per_user(lists: Tensor, examples: EvalExamples, liked_threshold: float) -> dict[str, Tensor]:
    """What each user's list earned in logged feedback per recommended item, an item that was not hit earning nothing. float [U] each.

    Defined for every user, so two recommenders can be compared user by user.
    """
    return _reward_per_item(hits_per_user(lists, examples, liked_threshold), list_length=lists.shape[1])


def _reward_per_item(per_user: dict[str, Tensor], list_length: int) -> dict[str, Tensor]:
    return {
        "played_ratio_per_item": per_user["played_ratio"] / list_length,
        "relative_surplus_per_item": (per_user["relative_ratio"] - USUAL_RELATIVE_RATIO * per_user["hits"]) / list_length,
        "liked_per_item": per_user["liked"] / list_length,
        "adopted_per_item": per_user["adopted"] / list_length,
    }


def hits_per_user(lists: Tensor, examples: EvalExamples, liked_threshold: float) -> dict[str, Tensor]:
    """Per user, the number of hits and the sums of the logged responses over them. float [U] each.

    An item the user consumed several times counts once, with the response to its first next occurrence.
    """
    lists = lists.cpu()
    future_items, played_ratio, relative_ratio = examples.future_items.cpu(), examples.future_played_ratio.cpu(), examples.future_relative_ratio.cpu()
    served_items = future_items.where(~examples.future_is_organic.cpu(), torch.full_like(future_items, PAD_ITEM))

    is_hit, position = _first_occurrence_in_pool(lists, future_items)
    is_served_hit, served_position = _first_occurrence_in_pool(lists, served_items)
    played_ratio_of_hit = played_ratio.gather(1, position)
    return {
        "hits": is_hit.sum(dim=1),
        "played_ratio": (is_hit * played_ratio_of_hit).sum(dim=1),
        "liked_hits": (is_hit * (played_ratio_of_hit >= liked_threshold)).sum(dim=1),
        "relative_ratio": (is_hit * relative_ratio.gather(1, position)).sum(dim=1),
        "liked": (is_hit * examples.future_liked.cpu().gather(1, position)).sum(dim=1),
        "adopted": (is_hit * examples.future_adopts.cpu().gather(1, position)).sum(dim=1),
        "served_hits": is_served_hit.sum(dim=1),
        "served_played_ratio": (is_served_hit * played_ratio.gather(1, served_position)).sum(dim=1),
    }


def _first_occurrence_in_pool(lists: Tensor, pools: Tensor) -> tuple[Tensor, Tensor]:
    """Where each listed item first occurs in its user's pool.

    lists: long [U, K], pools: long [U, M] padded with PAD_ITEM. Returns (float [U, K], 1 where the item occurs
    and has not appeared earlier in the list; long [U, K], the column of its first occurrence, 0 where there is none).
    """
    occurs_at = (lists[:, :, None] == pools[:, None, :]) & (lists[:, :, None] != PAD_ITEM)
    is_hit = occurs_at.any(dim=2) & first_occurrence(lists)
    return is_hit.float(), occurs_at.int().argmax(dim=2)

In [ ]:
%%writefile /kaggle/working/src/gptrec/evaluation/policy_report.py
"""Everything measured about one recommender on held-out users: relevance, predicted reward, behaviour."""

from __future__ import annotations

from dataclasses import dataclass, field
from itertools import combinations
from typing import Callable

import pandas as pd
import torch
from torch import Tensor

from gptrec.data.batches import EvalExamples, HistoryBatch
from gptrec.data.log import PAD_ITEM, ItemTable
from gptrec.evaluation.behavior import recommendation_profile
from gptrec.evaluation.logged_feedback import logged_reception, logged_reward_per_user
from gptrec.evaluation.relevance import contains, per_user_ranking_metrics, ranking_metrics
from gptrec.evaluation.significance import paired_bootstrap
from gptrec.models.policy import NextItemPolicy
from gptrec.models.reward_model import Feedback, RewardModel
from gptrec.rollout import (
    GeneratedActions,
    GenerationConfig,
    SimulatedFeedback,
    generate_actions,
    histories_of_other_users,
    recommend_top_k,
    simulate_feedback,
)
from gptrec.training.precision import autocast


@dataclass(frozen=True)
class EvaluationConfig:
    ks: tuple[int, ...] = (1, 5, 10)
    generation: GenerationConfig = field(default_factory=lambda: GenerationConfig(greedy=True))
    simulated_feedback: SimulatedFeedback = SimulatedFeedback.EXPECTED
    liked_threshold: float = 0.5
    batch_size: int = 256
    mixed_precision: bool = False


@dataclass
class PolicyEvaluation:
    """Scalar metrics plus the lists they were computed from, for further analysis. Tensors are on the CPU."""

    metrics: dict[str, float]
    top_k: Tensor
    generated: GeneratedActions
    feedback: dict[str, Feedback]
    feedback_for_other_user: dict[str, Feedback]

    def generated_lists(self) -> ListEvaluation:
        """The item-by-item generated lists in the form shared with lists from any other source."""
        metrics = {name: value for name, value in list_metrics(self.metrics, "next_k").items() if "/" in name}
        metrics |= {name: value for name, value in self.metrics.items() if name.startswith(("predicted/", "training/"))}
        return ListEvaluation(metrics, self.generated.actions, self.feedback, self.feedback_for_other_user)


@dataclass
class ListEvaluation:
    """What is measured about one list per user, however the lists were produced. Tensors are on the CPU.

    Metric names: <target set>/<metric>, profile/<statistic>, logged/<statistic>, predicted/<reward model>/<signal>.
    """

    metrics: dict[str, float]
    lists: Tensor
    feedback: dict[str, Feedback]
    feedback_for_other_user: dict[str, Feedback]


@torch.no_grad()
def evaluate_policy(
    policy: NextItemPolicy,
    examples: EvalExamples,
    items: ItemTable,
    reward_models: dict[str, RewardModel],
    config: EvaluationConfig,
    device: torch.device,
) -> PolicyEvaluation:
    """Recommends to every user in `examples` in two ways and scores both.

    "top_k" ranks all items by next-item probability. "next_k" generates the list item by item, feeding
    each choice back - the mode RL fine-tuning optimises. Metric names read <list>/<target set or "profile">/<metric>.
    """
    policy.eval()
    top_k_parts, generated_parts = [], []
    reception = _ReceptionOfBatches(reward_models, config)
    for batch in examples.batches(config.batch_size):
        history = batch.history.to(device)
        with autocast(device, config.mixed_precision):
            top_k_parts.append(recommend_top_k(policy, history, max(config.ks), config.generation).cpu())
            generated = generate_actions(policy, history, config.generation)
            generated_parts.append(generated.cpu())
            reception.add(history, generated.actions)

    top_k = torch.cat(top_k_parts)
    generated = GeneratedActions.concatenate(generated_parts)
    feedback, feedback_for_other_user = reception.of_all_batches()

    metrics = _prefixed("top_k", list_quality(top_k, examples, items, config))
    metrics |= _prefixed("next_k", list_quality(generated.actions, examples, items, config))
    metrics["next_k/policy_entropy"] = generated.entropy.mean().item()
    metrics |= predicted_reception(feedback, feedback_for_other_user, generated.actions, items)
    return PolicyEvaluation(metrics, top_k, generated, feedback, feedback_for_other_user)


@torch.no_grad()
def collect_lists(make_lists: Callable[[HistoryBatch], Tensor], examples: EvalExamples, batch_size: int, device: torch.device) -> Tensor:
    """One list per user of `examples` from any function of a batch of histories (given on `device`). long [U, K], on the CPU."""
    return torch.cat([make_lists(batch.history.to(device)).cpu() for batch in examples.batches(batch_size)])


@torch.no_grad()
def evaluate_lists(
    lists: Tensor,
    examples: EvalExamples,
    items: ItemTable,
    reward_models: dict[str, RewardModel],
    config: EvaluationConfig,
    device: torch.device,
) -> ListEvaluation:
    """Relevance, behaviour profile and predicted reception of given lists. lists: long [U, K], row u is for examples[u]."""
    lists = lists.cpu()
    reception = _ReceptionOfBatches(reward_models, config)
    for batch_lists, batch in zip(lists.split(config.batch_size), examples.batches(config.batch_size)):
        with autocast(device, config.mixed_precision):
            reception.add(batch.history.to(device), batch_lists.to(device))
    feedback, feedback_for_other_user = reception.of_all_batches()
    metrics = list_quality(lists, examples, items, config) | predicted_reception(feedback, feedback_for_other_user, lists, items)
    return ListEvaluation(metrics, lists, feedback, feedback_for_other_user)


class _ReceptionOfBatches:
    """Collects, batch by batch, what every reward model predicts for lists: from their own users and from other users."""

    def __init__(self, reward_models: dict[str, RewardModel], config: EvaluationConfig) -> None:
        self._reward_models = reward_models
        self._simulated_feedback = config.simulated_feedback
        self._own: dict[str, list[Feedback]] = {name: [] for name in reward_models}
        self._other: dict[str, list[Feedback]] = {name: [] for name in reward_models}

    def add(self, history: HistoryBatch, lists: Tensor) -> None:
        """history and lists of one batch, on the device of the reward models."""
        other_users = histories_of_other_users(history)
        for name, reward_model in self._reward_models.items():
            self._own[name].append(simulate_feedback(reward_model, history, lists, self._simulated_feedback).cpu())
            self._other[name].append(simulate_feedback(reward_model, other_users, lists, self._simulated_feedback).cpu())

    def of_all_batches(self) -> tuple[dict[str, Feedback], dict[str, Feedback]]:
        """(response of the users the lists were made for, response of other users), each: reward model name -> [U, K]."""
        own = {name: Feedback.concatenate(parts) for name, parts in self._own.items()}
        other = {name: Feedback.concatenate(parts) for name, parts in self._other.items()}
        return own, other


def list_quality(lists: Tensor, examples: EvalExamples, items: ItemTable, config: EvaluationConfig) -> dict[str, float]:
    """Everything about the lists that needs no reward model.

    Relevance against each target set, the behaviour profile, and the logged response to the listed items the
    users went on to consume. Names: <target set>/<metric>, profile/<statistic>, logged/<statistic>.
    """
    reachable_ks = [k for k in config.ks if k <= lists.shape[1]]
    quality: dict[str, float] = {}
    for target_name, relevant in relevance_targets(examples, items, config.liked_threshold).items():
        quality |= _prefixed(target_name, ranking_metrics(lists, relevant, reachable_ks))
    quality |= _prefixed("profile", recommendation_profile(lists, examples.history.items, items))
    return quality | _prefixed("logged", logged_reception(lists, examples, config.liked_threshold))


def predicted_reception(feedback: dict[str, Feedback], feedback_for_other_user: dict[str, Feedback], lists: Tensor, items: ItemTable) -> dict[str, float]:
    """The mean response every reward model predicts to the lists. Names: predicted/<reward model>/<signal>."""
    duration = torch.as_tensor(items.duration_seconds)
    reception = {}
    for name, predicted in feedback.items():
        signals = per_list_reception(predicted, feedback_for_other_user[name]) | {"played_seconds": (predicted.played_ratio * duration[lists]).mean(dim=1)}
        reception |= {f"predicted/{name}/{signal}": values.mean().item() for signal, values in signals.items()}
    return reception


def per_list_reception(predicted: Feedback, predicted_for_other_user: Feedback) -> dict[str, Tensor]:
    """The predicted response to each list, averaged over its items: signal name -> float [U].

    `personal_played_ratio` is what the list earns with its own user beyond what it would earn with another
    one: zero on average for lists that do not depend on the user. `chosen_played_ratio` counts an item the
    user is predicted not to take up as zero. `relative_ratio` is the played ratio as a quantile among
    everyone who consumed the item, which no item earns by being easy to finish. `returns_after_list` is the
    probability of a return after the last item, the end of the session the list would make.
    """
    played_ratio, played_ratio_for_other_user = predicted.played_ratio.mean(dim=1), predicted_for_other_user.played_ratio.mean(dim=1)
    relative_ratio, relative_ratio_for_other_user = predicted.relative_ratio.mean(dim=1), predicted_for_other_user.relative_ratio.mean(dim=1)
    return {
        "played_ratio": played_ratio,
        "played_ratio_for_other_user": played_ratio_for_other_user,
        "personal_played_ratio": played_ratio - played_ratio_for_other_user,
        "relative_ratio": relative_ratio,
        "personal_relative_ratio": relative_ratio - relative_ratio_for_other_user,
        "session_continues": predicted.session_continues.mean(dim=1),
        "returns": predicted.returns.mean(dim=1),
        "is_chosen": predicted.is_chosen.mean(dim=1),
        "chosen_played_ratio": (predicted.is_chosen * predicted.played_ratio).mean(dim=1),
        "liked": predicted.liked.mean(dim=1),
        "disliked": predicted.disliked.mean(dim=1),
        "adopts": predicted.adopts.mean(dim=1),
        "returns_after_list": predicted.returns[:, -1],
    }


def paired_comparison(before: ListEvaluation, after: ListEvaluation, examples: EvalExamples, items: ItemTable, config: EvaluationConfig) -> pd.DataFrame:
    """Per-user change from `before` to `after`, with bootstrap intervals: relevance, logged and predicted reception. One row per quantity."""
    k = max(k for k in config.ks if k <= before.lists.shape[1])
    targets = relevance_targets(examples, items, config.liked_threshold)
    compared = (
        ("all", f"ndcg@{k}"),
        ("liked", f"ndcg@{k}"),
        ("skipped", f"recall@{k}"),
        ("above_usual", f"ndcg@{k}"),
        ("below_usual", f"recall@{k}"),
        ("new", f"ndcg@{k}"),
        ("repeat", f"ndcg@{k}"),
    )
    rows = {}
    for target_name, metric in compared:
        per_user_before = per_user_ranking_metrics(before.lists, targets[target_name], [k])[metric]
        per_user_after = per_user_ranking_metrics(after.lists, targets[target_name], [k])[metric]
        rows[f"{target_name}/{metric}"] = paired_bootstrap(per_user_before, per_user_after)
    logged_before = logged_reward_per_user(before.lists, examples, config.liked_threshold)
    logged_after = logged_reward_per_user(after.lists, examples, config.liked_threshold)
    for quantity in logged_before:
        rows[f"logged/{quantity}"] = paired_bootstrap(logged_before[quantity], logged_after[quantity])
    for name in before.feedback:
        reception_before = per_list_reception(before.feedback[name], before.feedback_for_other_user[name])
        reception_after = per_list_reception(after.feedback[name], after.feedback_for_other_user[name])
        for signal in reception_before:
            rows[f"predicted/{name}/{signal}"] = paired_bootstrap(reception_before[signal], reception_after[signal])
    return pd.DataFrame(rows).T


def relevance_targets(examples: EvalExamples, items: ItemTable, liked_threshold: float) -> dict[str, Tensor]:
    """The items each user consumed next, split by how they responded. Each value is long [U, M], PAD_ITEM-padded.

    "liked" and "skipped" split at a fixed played ratio: a recommender that learned to avoid skips should keep
    the first and lose the second. That split favours items that are easy to finish. "above_usual" and
    "below_usual" split at the item's own mean played ratio in the train period, so every item can fall on
    either side: they tell whether the user took to the item more than people usually do. "repeat" and "new"
    split by whether the item is in the visible history: where people return to what they know, replaying
    the history already finds the first, and only the second asks for a recommender.
    """
    future_items = examples.future_items.cpu()
    played_ratio = examples.future_played_ratio.cpu()
    usual_played_ratio = torch.as_tensor(items.mean_played_ratio)[future_items]
    is_in_history = contains(future_items, examples.history.items.cpu())
    no_item = torch.full_like(future_items, PAD_ITEM)
    return {
        "all": future_items,
        "liked": future_items.where(played_ratio >= liked_threshold, no_item),
        "skipped": future_items.where(played_ratio < liked_threshold, no_item),
        "above_usual": future_items.where(played_ratio >= usual_played_ratio, no_item),
        "below_usual": future_items.where(played_ratio < usual_played_ratio, no_item),
        "repeat": future_items.where(is_in_history, no_item),
        "new": future_items.where(~is_in_history, no_item),
    }


def per_step_table(evaluation: PolicyEvaluation, examples: EvalExamples, items: ItemTable) -> pd.DataFrame:
    """One row per generation step: how the generated item and its predicted reception change along the list.

    Later steps condition on more generated - not observed - items, so drift shows up as a trend over rows.
    """
    actions = evaluation.generated.actions
    columns = {
        "policy_entropy": evaluation.generated.entropy.mean(0),
        "in_true_continuation": contains(actions, examples.future_items.cpu()).float().mean(0),
        "in_history": contains(actions, examples.history.items.cpu()).float().mean(0),
        "popularity": torch.as_tensor(items.train_popularity, dtype=torch.float32)[actions].mean(0),
        "duration_seconds": torch.as_tensor(items.duration_seconds)[actions].mean(0),
    }
    for name, predicted in evaluation.feedback.items():
        for signal, values in vars(predicted).items():
            columns[f"predicted_{signal}/{name}"] = values.mean(0)
    for first, second in combinations(evaluation.feedback, 2):
        disagreement = (evaluation.feedback[first].played_ratio - evaluation.feedback[second].played_ratio).abs()
        columns[f"played_ratio_disagreement/{first}_vs_{second}"] = disagreement.mean(0)
    step = pd.RangeIndex(1, actions.shape[1] + 1, name="step")
    return pd.DataFrame({name: values.numpy() for name, values in columns.items()}, index=step)


def popularity_baseline_metrics(examples: EvalExamples, items: ItemTable, config: EvaluationConfig) -> dict[str, float]:
    """Relevance and profile of showing every user the most popular items: the floor a recommender must beat."""
    most_popular = torch.as_tensor(items.train_popularity).topk(max(config.ks)).indices
    lists = most_popular.repeat(len(examples), 1)
    metrics: dict[str, float] = {}
    for target_name, relevant in relevance_targets(examples, items, config.liked_threshold).items():
        metrics |= _prefixed(f"top_k/{target_name}", ranking_metrics(lists, relevant, config.ks))
    return metrics | _prefixed("top_k/profile", recommendation_profile(lists, examples.history.items, items))


def list_metrics(metrics: dict[str, float], list_name: str) -> dict[str, float]:
    """The metrics of one kind of list ("top_k" or "next_k") with that prefix removed, so both kinds can share rows."""
    prefix = f"{list_name}/"
    return {name.removeprefix(prefix): value for name, value in metrics.items() if name.startswith(prefix)}


def metrics_table(evaluations: dict[str, PolicyEvaluation]) -> pd.DataFrame:
    """Metrics of several recommenders side by side: one column per recommender."""
    return pd.DataFrame({name: evaluation.metrics for name, evaluation in evaluations.items()})


def _prefixed(prefix: str, metrics: dict[str, float]) -> dict[str, float]:
    return {f"{prefix}/{name}": value for name, value in metrics.items()}

In [ ]:
%%writefile /kaggle/working/src/gptrec/evaluation/relevance.py
"""Ranking quality of recommendation lists against the items users actually consumed next."""

from __future__ import annotations

from typing import Sequence

import torch
from torch import Tensor

from gptrec.data.log import PAD_ITEM


def ranking_metrics(recommended: Tensor, relevant: Tensor, ks: Sequence[int]) -> dict[str, float]:
    """Recall@K, NDCG@K, MAP@K and HitRate@K, averaged over the users who have at least one relevant item.

    recommended: long [U, N], best first. A repeated item scores only at its first position.
    relevant: long [U, M], in any order, padded with PAD_ITEM. Repeats count as one relevant item.
    """
    per_user = per_user_ranking_metrics(recommended, relevant, ks)
    has_relevant = ~next(iter(per_user.values())).isnan()
    metrics = {"num_users": float(has_relevant.sum())}
    return metrics | {name: values[has_relevant].mean().item() if has_relevant.any() else float("nan") for name, values in per_user.items()}


def per_user_ranking_metrics(recommended: Tensor, relevant: Tensor, ks: Sequence[int]) -> dict[str, Tensor]:
    """The same metrics for every user separately: metric name -> float [U], NaN for users without relevant items."""
    if max(ks) > recommended.shape[1]:
        raise ValueError(f"cannot compute metrics @{max(ks)} from lists of {recommended.shape[1]} items")
    hits = (contains(recommended, relevant) & first_occurrence(recommended)).float()
    num_relevant = _count_distinct_items(relevant)
    undefined = num_relevant == 0
    num_relevant = num_relevant.clamp(min=1)

    per_user = {}
    for k in ks:
        hits_at_k = hits[:, :k]
        rank = torch.arange(1, k + 1, device=hits.device)
        discount = 1.0 / torch.log2(rank + 1.0)
        best_possible_hits = num_relevant.clamp(max=k)
        ideal_dcg = discount.cumsum(0)[best_possible_hits - 1]
        precision_at_rank = hits_at_k.cumsum(1) / rank

        per_user[f"recall@{k}"] = hits_at_k.sum(1) / num_relevant
        per_user[f"ndcg@{k}"] = (hits_at_k * discount).sum(1) / ideal_dcg
        per_user[f"map@{k}"] = (precision_at_rank * hits_at_k).sum(1) / best_possible_hits
        per_user[f"hit_rate@{k}"] = hits_at_k.amax(1)
    return {name: values.masked_fill(undefined, float("nan")) for name, values in per_user.items()}


def first_occurrence(lists: Tensor) -> Tensor:
    """bool [U, N]: False where the item already appeared earlier in the same list."""
    length = lists.shape[1]
    same_item = lists[:, :, None] == lists[:, None, :]
    earlier_position = torch.ones(length, length, dtype=torch.bool, device=lists.device).tril(diagonal=-1)
    return ~(same_item & earlier_position).any(dim=2)


def contains(lists: Tensor, pools: Tensor) -> Tensor:
    """bool [U, N]: whether lists[u, i] occurs anywhere in pools[u]. PAD_ITEM never counts as contained."""
    return ((lists[:, :, None] == pools[:, None, :]).any(dim=2)) & (lists != PAD_ITEM)


def _count_distinct_items(items: Tensor) -> Tensor:
    """Number of distinct non-padding items in each row. long [U]"""
    ordered = items.sort(dim=1).values
    starts_new_run = torch.ones_like(ordered, dtype=torch.bool)
    starts_new_run[:, 1:] = ordered[:, 1:] != ordered[:, :-1]
    return (starts_new_run & (ordered != PAD_ITEM)).sum(dim=1)

In [ ]:
%%writefile /kaggle/working/src/gptrec/evaluation/reward_model_metrics.py
"""How well a reward model predicts held-out feedback, and whether it beats history-free baselines."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch

from gptrec.data.batches import EventBatch, EventWindows, swap_one_target_item
from gptrec.data.log import ItemTable
from gptrec.evaluation.relevance import contains
from gptrec.models.reward_model import RewardModel
from gptrec.rollout import ScoredCandidates
from gptrec.training.precision import autocast

COMPLETED_THRESHOLD = 0.5


@dataclass
class HeldoutFeedback:
    """Observed and predicted feedback for held-out events. All arrays are aligned, one entry per event."""

    item: np.ndarray
    played_ratio: np.ndarray
    relative_ratio: np.ndarray
    session_continues: np.ndarray
    returns: np.ndarray
    session_continues_known: np.ndarray
    returns_known: np.ndarray
    is_organic: np.ndarray
    liked: np.ndarray
    disliked: np.ndarray
    reactions_known: np.ndarray
    adopts: np.ndarray
    adopts_known: np.ndarray
    predicted_played_ratio: np.ndarray
    predicted_relative_ratio: np.ndarray
    predicted_session_continues: np.ndarray
    predicted_returns: np.ndarray
    predicted_is_chosen: np.ndarray
    predicted_is_chosen_if_swapped: np.ndarray
    predicted_liked: np.ndarray
    predicted_disliked: np.ndarray
    predicted_adopts: np.ndarray
    recent_played_ratio: np.ndarray
    recent_relative_ratio: np.ndarray


@torch.no_grad()
def predict_heldout_feedback(reward_model: RewardModel, windows: EventWindows, batch_size: int, device: torch.device, mixed_precision: bool = True) -> HeldoutFeedback:
    """Teacher-forced predictions for every target event of `windows`: the model sees the true feedback on all earlier events.

    One target event per window is also scored with another item in its place, to see whether the model can tell.
    """
    reward_model.eval()
    collected: dict[str, list[np.ndarray]] = {name: [] for name in HeldoutFeedback.__dataclass_fields__}
    for batch in windows.batches(batch_size):
        batch = batch.to(device)
        with autocast(device, mixed_precision):
            predicted = reward_model.predict(batch.items, batch.played_ratio)
            is_chosen_if_swapped = _is_chosen_if_swapped(reward_model, batch)
        per_event = {
            "item": batch.items,
            "played_ratio": batch.played_ratio,
            "relative_ratio": batch.relative_ratio,
            "session_continues": batch.session_continues,
            "returns": batch.returns,
            "session_continues_known": batch.session_continues_known,
            "returns_known": batch.returns_known,
            "is_organic": batch.is_organic,
            "liked": batch.liked,
            "disliked": batch.disliked,
            "reactions_known": batch.reactions_known,
            "adopts": batch.adopts,
            "adopts_known": batch.adopts_known,
            "predicted_played_ratio": predicted.played_ratio,
            "predicted_relative_ratio": predicted.relative_ratio,
            "predicted_session_continues": predicted.session_continues,
            "predicted_returns": predicted.returns,
            "predicted_is_chosen": predicted.is_chosen,
            "predicted_is_chosen_if_swapped": is_chosen_if_swapped,
            "predicted_liked": predicted.liked,
            "predicted_disliked": predicted.disliked,
            "predicted_adopts": predicted.adopts,
            "recent_played_ratio": _mean_of_preceding(batch.played_ratio),
            "recent_relative_ratio": _mean_of_preceding(batch.relative_ratio),
        }
        for name, values in per_event.items():
            collected[name].append(values[batch.is_target].cpu().numpy())
    return HeldoutFeedback(**{name: np.concatenate(chunks) for name, chunks in collected.items()})


def _is_chosen_if_swapped(reward_model: RewardModel, batch: EventBatch) -> torch.Tensor:
    """Predicted probability of being chosen for one swapped-in item per window, NaN at every other position. [B, L]"""
    swapped = swap_one_target_item(batch)
    predicted = reward_model.predict(swapped.items, batch.played_ratio).is_chosen.gather(1, swapped.position[:, None])
    predicted = predicted.masked_fill(~swapped.is_different[:, None], float("nan"))
    return torch.full_like(batch.played_ratio, float("nan")).scatter(1, swapped.position[:, None], predicted.to(batch.played_ratio.dtype))


def _mean_of_preceding(values: torch.Tensor) -> torch.Tensor:
    """Mean of the values before each position of its window. NaN at the first position. [B, L] -> [B, L]"""
    num_preceding = torch.arange(values.shape[1], device=values.device)
    sum_of_preceding = values.cumsum(dim=1) - values
    return sum_of_preceding / num_preceding


def reward_model_report(heldout: HeldoutFeedback, items: ItemTable) -> pd.DataFrame:
    """Played-ratio prediction quality of the model next to baselines that ignore the user or the item."""
    train_mean = items.train_mean_played_ratio
    predictors = {
        "reward model": heldout.predicted_played_ratio,
        "train mean": np.full_like(heldout.played_ratio, train_mean),
        "item mean": items.mean_played_ratio[heldout.item],
        "user recent mean": np.nan_to_num(heldout.recent_played_ratio, nan=train_mean),
    }
    return pd.DataFrame({name: prediction_quality(heldout.played_ratio, predicted) for name, predicted in predictors.items()}).T


def relative_ratio_report(heldout: HeldoutFeedback) -> pd.DataFrame:
    """Prediction quality of the relative played ratio next to predictors that know nothing about the item.

    `auc` is for the event "the user played more of the item than people usually do". No item statistic can
    predict it, because every item's mean relative ratio is 0.5 by construction.
    """
    usual_for_the_item = 0.5
    predictors = {
        "reward model": heldout.predicted_relative_ratio,
        "usual for the item": np.full_like(heldout.relative_ratio, usual_for_the_item),
        "user recent mean": np.nan_to_num(heldout.recent_relative_ratio, nan=usual_for_the_item),
    }
    return pd.DataFrame({name: prediction_quality(heldout.relative_ratio, predicted) for name, predicted in predictors.items()}).T


def quality_by_origin(heldout: HeldoutFeedback) -> pd.DataFrame:
    """Played-ratio prediction quality separately for events the user chose and events a recommender served.

    RL asks the reward model about recommendations, so its quality on recommended events is the one that matters.
    """
    origins = {"chosen by the user": heldout.is_organic, "recommended": ~heldout.is_organic}
    rows = {}
    for origin, has_origin in origins.items():
        if has_origin.any():
            quality = prediction_quality(heldout.played_ratio[has_origin], heldout.predicted_played_ratio[has_origin])
            rows[origin] = quality | {"mean_played_ratio": float(heldout.played_ratio[has_origin].mean()), "num_events": float(has_origin.sum())}
    return pd.DataFrame(rows).T


RETENTION_SIGNALS = ("session_continues", "returns")


def retention_quality(heldout: HeldoutFeedback) -> pd.DataFrame:
    """Prediction quality of each retention signal on the events whose outcome the log knows, with how often it is true."""
    rows = {}
    for signal in RETENTION_SIGNALS:
        is_known = getattr(heldout, f"{signal}_known")
        observed, predicted = getattr(heldout, signal)[is_known], getattr(heldout, f"predicted_{signal}")[is_known]
        quality = prediction_quality(observed, predicted) if is_known.any() else dict.fromkeys(("mae", "rmse", "cross_entropy", "auc"), float("nan"))
        rows[signal] = quality | {"base_rate": float(observed.mean()) if is_known.any() else float("nan"), "num_events": float(is_known.sum())}
    return pd.DataFrame(rows).T


REACTION_SIGNALS = {"liked": "reactions_known", "disliked": "reactions_known", "adopts": "adopts_known"}


def reaction_quality(heldout: HeldoutFeedback) -> pd.DataFrame:
    """Prediction quality of likes, dislikes and adoption on the events whose outcome the log knows, with how often each happens."""
    rows = {}
    for signal, known in REACTION_SIGNALS.items():
        is_known = getattr(heldout, known)
        observed, predicted = getattr(heldout, signal)[is_known], getattr(heldout, f"predicted_{signal}")[is_known]
        quality = prediction_quality(observed, predicted) if is_known.any() else dict.fromkeys(("mae", "rmse", "cross_entropy", "auc"), float("nan"))
        rows[signal] = quality | {"base_rate": float(observed.mean()) if is_known.any() else float("nan"), "num_events": float(is_known.sum())}
    return pd.DataFrame(rows).T


def choice_quality(heldout: HeldoutFeedback) -> dict[str, float]:
    """Whether the model tells the item a user chose from another item put in its place.

    `auc` is the probability that a chosen item is scored above a swapped-in one; 0.5 means the model
    cannot tell and its reward says nothing about relevance.
    """
    was_probed = ~np.isnan(heldout.predicted_is_chosen_if_swapped)
    of_chosen_items, of_swapped_items = heldout.predicted_is_chosen, heldout.predicted_is_chosen_if_swapped[was_probed]
    is_chosen_item = np.concatenate([np.ones(len(of_chosen_items), dtype=bool), np.zeros(len(of_swapped_items), dtype=bool)])
    return {
        "auc": roc_auc(is_chosen_item, np.concatenate([of_chosen_items, of_swapped_items])),
        "mean_probability_of_chosen_items": float(of_chosen_items.mean()),
        "mean_probability_of_swapped_items": float(of_swapped_items.mean()) if was_probed.any() else float("nan"),
        "num_swapped_items": float(was_probed.sum()),
    }


def prediction_quality(observed: np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    """Error of predictions in [0, 1] against observations in [0, 1]."""
    observed, predicted = observed.astype(np.float64), predicted.astype(np.float64)
    clipped = np.clip(predicted, 1e-6, 1 - 1e-6)
    return {
        "mae": float(np.abs(observed - predicted).mean()),
        "rmse": float(np.sqrt(((observed - predicted) ** 2).mean())),
        "cross_entropy": float(-(observed * np.log(clipped) + (1 - observed) * np.log(1 - clipped)).mean()),
        "auc": roc_auc(observed >= COMPLETED_THRESHOLD, predicted),
    }


def roc_auc(is_positive: np.ndarray, score: np.ndarray) -> float:
    """Probability that a random positive is scored above a random negative; ties count as half."""
    num_positive, num_negative = int(is_positive.sum()), int((~is_positive).sum())
    if num_positive == 0 or num_negative == 0:
        return float("nan")
    _, tie_group, group_size = np.unique(score, return_inverse=True, return_counts=True)
    last_rank_in_group = np.cumsum(group_size)
    mean_rank = (last_rank_in_group - (group_size - 1) / 2)[tie_group]
    rank_sum_of_positives = mean_rank[is_positive].sum()
    return float((rank_sum_of_positives - num_positive * (num_positive + 1) / 2) / (num_positive * num_negative))


def calibration_table(observed: np.ndarray, predicted: np.ndarray, num_bins: int = 10) -> pd.DataFrame:
    """Mean prediction against mean observation inside equal-width prediction bins; a calibrated model has them equal."""
    bin_index = np.clip((predicted * num_bins).astype(int), 0, num_bins - 1)
    table = pd.DataFrame({"bin": bin_index, "predicted": predicted, "observed": observed})
    return table.groupby("bin").agg(mean_predicted=("predicted", "mean"), mean_observed=("observed", "mean"), num_events=("observed", "size"))


def error_by_group(heldout: HeldoutFeedback, group_of_item: np.ndarray) -> pd.DataFrame:
    """Played-ratio bias and absolute error per item group, e.g. popularity or duration buckets.

    group_of_item: [V+1], the group label of every item token.
    """
    error = heldout.predicted_played_ratio.astype(np.float64) - heldout.played_ratio
    table = pd.DataFrame({"group": group_of_item[heldout.item], "error": error, "absolute_error": np.abs(error)})
    return table.groupby("group").agg(bias=("error", "mean"), mae=("absolute_error", "mean"), num_events=("error", "size"))


def signal_spread_over_candidates(candidates: ScoredCandidates, future_items: torch.Tensor) -> pd.DataFrame:
    """What each predicted signal offers a recommender that chooses among its own candidates. One row per signal.

    spread_between_candidates: standard deviation over one user's candidates, averaged over users. A signal
        that hardly differs between the items a user could be shown gives RL and re-ranking nothing to choose by.
    spread_between_users: standard deviation over users of the mean over their candidates.
    relevance_auc: probability that a candidate the user really consumed next has a higher value than one the
        user did not; at 0.5 the signal knows nothing about relevance. The recommender's own log-probability
        is the yardstick.

    candidates: on the CPU; future_items: long [U, M], what each user really consumed next.
    """
    is_relevant = contains(candidates.items, future_items).flatten().numpy()
    predicted = candidates.feedback
    signals = vars(predicted) | {"chosen_played_ratio": predicted.is_chosen * predicted.played_ratio, "recommender_log_prob": candidates.log_probs}
    rows = {}
    for name, values in signals.items():
        values = values.float()
        rows[name] = {
            "spread_between_candidates": values.std(dim=1).mean().item(),
            "spread_between_users": values.mean(dim=1).std().item(),
            "relevance_auc": roc_auc(is_relevant, values.flatten().numpy()),
        }
    return pd.DataFrame(rows).T

In [ ]:
%%writefile /kaggle/working/src/gptrec/evaluation/significance.py
"""Is a difference between two recommenders more than noise between users?"""

from __future__ import annotations

import torch
from torch import Tensor

RESAMPLES_PER_PASS = 100


def paired_bootstrap(before: Tensor, after: Tensor, num_resamples: int = 2000, seed: int = 0) -> dict[str, float]:
    """Mean per-user change (after - before) with a 95% bootstrap interval over users.

    before, after: float [U], the same quantity for the same users under two recommenders. Users for whom
    either value is NaN are left out. The change is significant at the 5% level when the interval excludes 0.
    """
    defined = ~(before.isnan() | after.isnan())
    before, after = before[defined].double(), after[defined].double()
    change = after - before
    if len(change) == 0:
        return {name: float("nan") for name in ("before", "after", "change", "change_ci_low", "change_ci_high")} | {"num_users": 0.0}
    generator = torch.Generator().manual_seed(seed)
    resampled_means = []
    for _ in range(0, num_resamples, RESAMPLES_PER_PASS):
        users = torch.randint(len(change), (RESAMPLES_PER_PASS, len(change)), generator=generator)
        resampled_means.append(change[users].mean(dim=1))
    interval = torch.quantile(torch.cat(resampled_means), torch.tensor([0.025, 0.975], dtype=torch.double))
    return {
        "before": before.mean().item(),
        "after": after.mean().item(),
        "change": change.mean().item(),
        "change_ci_low": interval[0].item(),
        "change_ci_high": interval[1].item(),
        "num_users": float(len(change)),
    }

In [ ]:
%%writefile /kaggle/working/src/gptrec/experiment.py
"""The whole study as a sequence of resumable stages.

    1. prepare_data                 raw events -> interaction log
    2. pretrain_policy              the base recommender
    3. train_reward_models          "train" (drives RL) and "heldout" (only judges), on disjoint users
    4. compare_baselines            popularity, replaying history, Markov chain, GRU4Rec, SASRec, BERT4Rec against the base recommender
    5. finetune                     RL fine-tuning, compared with the base recommender
    6. sweep_kl_weight              the trade-off between relevance and reward as the anchor weakens
       sweep_kl_budget              the same trade-off with the distance from the base recommender fixed instead of the anchor weight
       compare_advantage_estimators the leave-one-out baseline against the value model, at equal weights and at equal distances
       compare_simulated_feedback   fine-tuning with a sampled response to the generated items instead of the expected one
       explain_reward_gain          how much of the gain depends on who receives the list, and how far item averages alone go
       compare_alternatives_to_rl   liked-only fine-tuning, re-ranking, off-policy REINFORCE, best of several sampled lists
       sweep_rewards                rewards for continuing the session and for returning, each at several anchor weights
       sweep_relevance_in_reward    relevance asked for by the reward itself, alone and together with the anchor
       sweep_personal_rewards       rewards that pay only for what depends on the user
       sweep_relative_reward        a reward for the played ratio relative to what is usual for the item
       compare_list_reward          a reward that depends on the list as a whole, against greedy ways of maximising it
       ablate_sequence_credit       fine-tuning that credits each item with its own reward only, or sees one item per episode
       analyse_multistep
    7. diagnose_return_label        whether a return label follows the item after the event rather than the event
       restrict_finetuning_capacity RL that may change only one score offset per item, or only low-rank updates of the transformer
       compare_group_relative_advantages  GRPO's group-normalised advantages, at equal KL budgets
       decompose_policy_change      the main run split into one offset per item and the rest; the reward's own split, before any RL
       align_at_inference           the base recommender steered by the closed form of the anchored objective, with look-ahead
       learn_from_logged_preferences  reward-weighted fine-tuning and within-user DPO on logged responses: no reward model
       predict_lists_without_feedback lists from one head per position, none of them reading an unconfirmed item
       interpolate_weights          operating points between the base and the fine-tuned weights, and beyond
       control_for_continued_training learning from the log against the same epoch of training without its weights

A stage whose grid of settings is empty in the configuration does nothing.

Every stage reads what it needs from the experiment's home directory and writes its checkpoints and
reports back there. A stage whose checkpoint already exists does not train again, so an interrupted
run continues where it stopped. A home directory belongs to one configuration; `reuse_artifacts` starts
one from the models and the finished runs of an earlier session that differed only in the grids of its sweeps.
"""

from __future__ import annotations

import functools
import json
import re
import shutil
import stat
import time
from dataclasses import asdict, dataclass, field, replace
from functools import cached_property
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn.functional as F

from gptrec import plots
from gptrec.baselines.suite import BaselineConfig, fit_baselines
from gptrec.data.batches import EvalExamples, EventWindows, build_eval_examples
from gptrec.data.log import InteractionLog, Period
from gptrec.data.preprocess import PreprocessingConfig, build_interaction_log
from gptrec.decoding import ContinuationTrainingConfig, GuidanceConfig, fit_continuation_values, guided_lists
from gptrec.evaluation.analyses import (
    candidate_log_probs,
    exposure_lorenz,
    feedback_assumptions_on_real_sequences,
    impressions_by_attribute,
    item_effect_share,
    kl_along_lists,
    largest_impression_changes,
    logit_shift_by_item,
    mean_logit_shift_per_item,
    position_hits,
    relevance_by_history_length,
    return_rate_by_duration,
)
from gptrec.evaluation.behavior import intra_list_distance, list_overlap, recommendation_profile
from gptrec.evaluation.policy_report import (
    EvaluationConfig,
    ListEvaluation,
    PolicyEvaluation,
    collect_lists,
    evaluate_lists,
    evaluate_policy,
    list_metrics,
    metrics_table,
    paired_comparison,
    per_step_table,
    popularity_baseline_metrics,
)
from gptrec.evaluation.relevance import ranking_metrics
from gptrec.evaluation.reward_model_metrics import (
    HeldoutFeedback,
    calibration_table,
    choice_quality,
    error_by_group,
    predict_heldout_feedback,
    quality_by_origin,
    reaction_quality,
    relative_ratio_report,
    retention_quality,
    reward_model_report,
    signal_spread_over_candidates,
)
from gptrec.models.adapters import Adaptation, interpolated, with_item_offsets
from gptrec.models.backbone import TransformerConfig
from gptrec.models.multi_token import MultiTokenHeads, load_multi_token_heads, multi_token_lists, save_multi_token_heads, train_multi_token_heads
from gptrec.models.policy import NextItemPolicy
from gptrec.models.reward_model import Feedback, RewardModel
from gptrec.models.sequence_recommender import NextItemScorer
from gptrec.models.value_model import ContinuationValueModel
from gptrec.reports import ReportWriter
from gptrec.rollout import (
    SIMULATION_ASSUMPTIONS,
    RatioBaseline,
    RewardFunction,
    RewardSpec,
    ScoredCandidates,
    SimulatedFeedback,
    best_of_sampled_lists,
    recommend_top_k,
    rerank,
    score_candidates,
)
from gptrec.training.checkpoint import load_continuation_value_model, load_policy, load_reward_model, save_continuation_value_model, save_model
from gptrec.training.off_policy import LoggedReturnConfig, finetune_on_logged_returns
from gptrec.training.precision import autocast
from gptrec.training.preference import LoggedPreferenceConfig, RewardWeightedConfig, finetune_on_logged_preferences, finetune_with_reward_weights
from gptrec.training.rl import AdvantageEstimator, RLConfig, finetune_with_rl
from gptrec.training.supervised import (
    FeedbackLossWeights,
    OptimizationConfig,
    PolicyTrainingConfig,
    RewardModelTrainingConfig,
    finetune_on_liked_items,
    train_policy,
    train_reward_model,
)

DEFAULT_REWARDS = (
    ("played ratio", RewardSpec()),
    ("played time", RewardSpec(weigh_ratio_by_duration=True)),
    ("session", RewardSpec(played_ratio_weight=0.0, session_continues_weight=1.0)),
    ("return", RewardSpec(played_ratio_weight=0.0, returns_weight=1.0)),
    ("ratio + return", RewardSpec(returns_weight=1.0)),
    ("ratio + session + return", RewardSpec(session_continues_weight=1.0, returns_weight=1.0)),
)
CHOSEN_RATIO = RewardSpec(weigh_by_choice=True)
LOGGED_RELEVANCE = RewardSpec(played_ratio_weight=0.0, logged_relevance_weight=1.0)
PERSONAL_RATIO = RewardSpec(ratio_baseline=RatioBaseline.OTHER_USER)
RATIO_ABOVE_ITEM_MEAN = RewardSpec(ratio_baseline=RatioBaseline.ITEM_MEAN)
RELATIVE_RATIO = RewardSpec(played_ratio_weight=0.0, relative_ratio_weight=1.0)
LIKES = RewardSpec(played_ratio_weight=0.0, liked_weight=1.0, disliked_weight=1.0)
ADOPTION = RewardSpec(played_ratio_weight=0.0, adoption_weight=1.0)
RETURN_AFTER_THE_LIST = RewardSpec(played_ratio_weight=0.0, final_returns_weight=1.0)
JUDGED_SIGNALS = ("played_ratio", "personal_played_ratio", "session_continues", "returns", "relative_ratio")
REALISED_KL = "training/kl_to_reference"
TRAINING_STATISTICS = {
    REALISED_KL: "kl_to_reference",
    "training/anchor_weight": "reference_kl_weight",
    "training/clip_fraction": "clip_fraction",
    "training/value_explained_variance": "value_explained_variance",
    "training/advantage_scale": "advantage_scale",
}
END_OF_TRAINING_STEPS = 50
INTRA_LIST_DISTANCE = "profile/intra_list_distance"
FEEDBACK_ASSUMPTION_STEPS = 5
TRAINING_CURVES = [
    "reward",
    "kl_to_reference",
    "reference_kl_weight",
    "entropy",
    "distinct_action_share",
    "logged_hit_rate",
    "history_repeat_rate",
    "mean_action_popularity",
    "mean_action_duration_seconds",
    "gradient_norm",
    "clip_fraction",
    "value_explained_variance",
]
TRAINING_CURVE_SMOOTHING_STEPS = 20
CONFIG_RECORD = "experiment_config.json"
RUN_TIME_SAFETY_FACTOR = 1.25
RESERVE_FOR_LAST_STAGES_SECONDS = 30 * 60
RUN_GRIDS = (
    "kl_weights",
    "rewards",
    "reward_kl_weights",
    "choice_reward_kl_weights",
    "logged_relevance_weights",
    "logged_relevance_kl_weights",
    "personal_reward_kl_weights",
    "credit_ablation_kl_weights",
    "rerank_relevance_weights",
    "importance_weight_caps",
    "sampled_list_counts",
    "evaluate_sweeps_on_validation",
    "relevance_tolerance",
    "kl_budgets",
    "estimator_kl_weights",
    "high_learning_rate_factor",
    "sampled_feedback_kl_weights",
    "relative_reward_kl_weights",
    "diversity_weights",
    "diversity_kl_weights",
    "masked_feedback_kl_weights",
    "return_label_check",
    "capacity_kl_budgets",
    "low_ranks",
    "group_relative_kl_budgets",
    "analyse_policy_change",
    "guidance_strengths",
    "guidance_discounts",
    "guided_rewards",
    "reward_weighted_temperatures",
    "reward_weighted_responses",
    "preference_betas",
    "multi_token_lists",
    "interpolation_factors",
    "continued_training_control",
)
KL_BATCH_SIZE = 256
CONTINUED_TRAINING = "continued training"
LIST_KINDS = ("ranked", "generated")


@dataclass(frozen=True)
class ExperimentConfig:
    preprocessing: PreprocessingConfig = field(default_factory=PreprocessingConfig)
    max_users: int | None = None
    transformer: TransformerConfig = field(default_factory=TransformerConfig)
    window_stride: int = 64
    policy_optimization: OptimizationConfig = field(default_factory=OptimizationConfig)
    reward_model_optimization: OptimizationConfig = field(default_factory=lambda: OptimizationConfig(epochs=5))
    reward_model_loss: FeedbackLossWeights = field(default_factory=FeedbackLossWeights)
    baselines: BaselineConfig = field(default_factory=BaselineConfig)
    horizon: int = 10
    max_future: int = 20
    ks: tuple[int, ...] = (1, 5, 10)
    evaluation_batch_size: int = 256
    rl: RLConfig = field(default_factory=RLConfig)
    sweep_steps: int = 1000
    kl_weights: tuple[float, ...] = (0.0, 0.03, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0, 3.0)
    rewards: tuple[tuple[str, RewardSpec], ...] = DEFAULT_REWARDS[1:]
    reward_kl_weights: tuple[float, ...] = (0.1, 0.3, 1.0)
    choice_reward_kl_weights: tuple[float, ...] = (0.0, 0.1, 0.3)
    logged_relevance_weights: tuple[float, ...] = (10.0, 3.0)
    logged_relevance_kl_weights: tuple[float, ...] = (0.0, 0.03, 0.1, 0.3)
    personal_reward_kl_weights: tuple[float, ...] = (0.0, 0.1, 0.3)
    credit_ablation_kl_weights: tuple[float, ...] = (0.1, 0.3)
    liked_finetuning: OptimizationConfig = field(default_factory=lambda: OptimizationConfig(epochs=1, learning_rate=1e-4))
    rerank_candidates: int = 100
    rerank_relevance_weights: tuple[float, ...] = (0.0, 0.01, 0.03, 0.1, 0.3, 1.0)
    logged_returns: LoggedReturnConfig = field(default_factory=LoggedReturnConfig)
    importance_weight_caps: tuple[float, ...] = (0.5, 2.0, 10.0)
    sampled_list_counts: tuple[int, ...] = (2, 8, 32)
    sample_from_top: int = 4
    evaluate_sweeps_on_validation: bool = True
    relevance_tolerance: float = 0.05
    kl_budgets: tuple[float, ...] = ()
    estimator_kl_weights: tuple[float, ...] = ()
    leave_one_out_rollouts: int = 4
    high_learning_rate_factor: float | None = None
    sampled_feedback_kl_weights: tuple[float, ...] = ()
    relative_reward_kl_weights: tuple[float, ...] = ()
    diversity_weights: tuple[float, ...] = ()
    diversity_kl_weights: tuple[float, ...] = (0.3,)
    masked_feedback_kl_weights: tuple[float, ...] = ()
    hidden_response_share: float = 0.5
    return_label_check: bool = False
    capacity_kl_budgets: tuple[float, ...] = ()
    low_ranks: tuple[int, ...] = (1, 8)
    item_offset_learning_rate: float = 3e-3
    low_rank_learning_rate: float = 1e-4
    group_relative_kl_budgets: tuple[float, ...] = ()
    analyse_policy_change: bool = False
    guidance_strengths: tuple[float, ...] = ()
    guidance_discounts: tuple[float, ...] = (0.0, 0.9)
    guidance_candidates: int = 50
    guidance_batch_size: int = 256
    guided_rewards: tuple[tuple[str, RewardSpec], ...] = (("played ratio", RewardSpec()),)
    continuation_training: ContinuationTrainingConfig = field(default_factory=ContinuationTrainingConfig)
    reward_weighted_temperatures: tuple[float, ...] = ()
    reward_weighted_responses: tuple[str, ...] = ("relative_ratio",)
    preference_betas: tuple[float, ...] = ()
    preference_margin: float = 0.1
    preference_lookahead: int = 20
    preference_finetuning: OptimizationConfig = field(default_factory=lambda: OptimizationConfig(epochs=1, learning_rate=1e-4))
    multi_token_lists: bool = False
    multi_token_optimization: OptimizationConfig = field(default_factory=lambda: OptimizationConfig(epochs=3, learning_rate=1e-3))
    multi_token_targets_per_head: int | None = None
    interpolation_factors: tuple[float, ...] = ()
    continued_training_control: bool = False

    def __post_init__(self) -> None:
        if self.horizon >= self.transformer.max_positions:
            raise ValueError("the horizon must leave room for a history inside max_positions")
        if max(self.ks) > self.horizon:
            raise ValueError(f"metrics @{max(self.ks)} need generated lists of at least that length, but horizon={self.horizon}")
        if self.rerank_candidates < self.horizon:
            raise ValueError("re-ranking needs at least as many candidates as the list is long")
        if self.rl.histories_per_step % self.leave_one_out_rollouts != 0:
            raise ValueError("leave_one_out_rollouts must divide histories_per_step, so that both estimators see as many lists per iteration")

    @property
    def history_len(self) -> int:
        """Events of history shown to the models: what is left of their position budget after the generated items."""
        return self.transformer.max_positions - self.horizon

    def rl_config(self, **overrides) -> RLConfig:
        generation = replace(self.rl.generation, horizon=self.horizon)
        return replace(self.rl, **({"history_len": self.history_len, "max_future": self.max_future, "generation": generation} | overrides))

    def sweep_run(self, reward: RewardSpec, kl_weight: float, **overrides) -> RLConfig:
        """A fine-tuning run of sweep length that differs from the main run in its reward, its anchor weight and `overrides`."""
        return self.rl_config(num_steps=self.sweep_steps, reward=reward, reference_kl_weight=kl_weight, **overrides)

    def evaluation_config(self, **overrides) -> EvaluationConfig:
        """Greedy lists under the same item constraints and simulated feedback the policy is fine-tuned with."""
        generation = replace(self.rl.generation, horizon=self.horizon, greedy=True, temperature=1.0)
        config = EvaluationConfig(self.ks, generation, self.rl.simulated_feedback, batch_size=self.evaluation_batch_size)
        return replace(config, **overrides)


def smoke_test_config(config: ExperimentConfig, max_users: int) -> ExperimentConfig:
    """`config` shrunk until every stage runs in minutes, even on a CPU: few users, tiny models, a handful of steps.

    What is specific to the dataset - split lengths, session gap, generation constraints - is kept, and so is
    every grid that `config` leaves empty, so the same code paths run and a stage the configuration switches off
    stays off. The numbers a smoke test produces mean nothing.
    """
    optimization = OptimizationConfig(epochs=1, batch_size=16, warmup_steps=5)
    horizon = 4
    preprocessing = replace(config.preprocessing, max_items=1_000, min_item_events=5, min_user_events=min(config.preprocessing.min_user_events, 20))
    rl = replace(config.rl, num_steps=6, histories_per_step=16, ppo_epochs=2, value_warmup_steps=2, evaluate_every=3, mixed_precision=True)
    switched_off = {name: getattr(config, name) for name in RUN_GRIDS if not getattr(config, name)}
    shrunk = replace(
        config,
        preprocessing=preprocessing,
        max_users=max_users,
        transformer=TransformerConfig(hidden_size=32, num_layers=1, num_heads=2, max_positions=24),
        window_stride=24,
        policy_optimization=optimization,
        reward_model_optimization=optimization,
        baselines=BaselineConfig(optimization=optimization, markov_chain_successors=20),
        horizon=horizon,
        max_future=10,
        ks=(1, horizon),
        evaluation_batch_size=64,
        rl=rl,
        sweep_steps=3,
        kl_weights=(0.0, 1.0),
        reward_kl_weights=(0.3,),
        choice_reward_kl_weights=(0.0,),
        logged_relevance_weights=(1.0,),
        logged_relevance_kl_weights=(0.0, 1.0),
        personal_reward_kl_weights=(1.0,),
        credit_ablation_kl_weights=(1.0,),
        liked_finetuning=optimization,
        rerank_candidates=20,
        rerank_relevance_weights=(0.0, 1.0),
        logged_returns=replace(config.logged_returns, optimization=optimization),
        importance_weight_caps=(0.5,),
        sampled_list_counts=(2,),
        kl_budgets=(0.5,),
        estimator_kl_weights=(1.0,),
        leave_one_out_rollouts=2,
        high_learning_rate_factor=10.0,
        sampled_feedback_kl_weights=(1.0,),
        relative_reward_kl_weights=(1.0,),
        diversity_weights=(1.0,),
        diversity_kl_weights=(1.0,),
        masked_feedback_kl_weights=(1.0,),
        return_label_check=True,
        capacity_kl_budgets=(0.5,),
        low_ranks=(1,),
        group_relative_kl_budgets=(0.5,),
        analyse_policy_change=True,
        guidance_strengths=(1.0,),
        guidance_discounts=(0.0, 0.9),
        guidance_candidates=8,
        guidance_batch_size=32,
        continuation_training=replace(config.continuation_training, num_steps=3, histories_per_step=16),
        reward_weighted_temperatures=(1.0,),
        preference_betas=(0.1,),
        preference_finetuning=optimization,
        multi_token_lists=True,
        multi_token_optimization=optimization,
        multi_token_targets_per_head=None if config.multi_token_targets_per_head is None else 16,
        interpolation_factors=(0.5,),
        continued_training_control=True,
    )
    return replace(shrunk, **switched_off)


def run_smoke_test(
    raw_events: pl.DataFrame,
    config: ExperimentConfig,
    home: str | Path,
    device: torch.device,
    max_users: int = 200,
    reactions: pl.DataFrame | None = None,
    item_metadata: pl.DataFrame | None = None,
) -> None:
    """Runs all stages of `config`, shrunk, on the first `max_users` users: finds out in minutes whether anything is broken.

    A second session then starts from the artifacts of the first with one more anchor weight in its sweep, the
    way a session tops up an earlier one. `max_users` must leave enough data after filtering; datasets with
    short histories need more users.
    """
    home = Path(home)
    first_session = smoke_test_config(config, max_users)
    Experiment(first_session, home / "first_session", device, show_figures=False).run(raw_events, reactions, item_metadata)

    second_session = replace(first_session, kl_weights=(*first_session.kl_weights, 0.5))
    reuse_artifacts(home / "first_session", home / "second_session", second_session)
    continued = Experiment(second_session, home / "second_session", device, show_figures=False)
    continued.prepare_data(raw_events, reactions)
    continued.pretrain_policy()
    continued.train_reward_models()
    continued.finetune()
    continued.sweep_kl_weight()


def reuse_artifacts(previous_home: str | Path, home: str | Path, config: ExperimentConfig) -> None:
    """Starts `home` from the interaction log, the models and the finished runs of an earlier session, so that only what is missing is trained.

    A checkpoint is found by the name of its run, and the name does not say which base recommender, reward
    models or optimiser settings produced it. The earlier session must therefore have had the configuration
    `config` in everything but the grids of its sweeps (`RUN_GRIDS`).
    """
    previous_home, home = Path(previous_home), Path(home)
    absent = [part for part in (CONFIG_RECORD, "data", "artifacts") if not (previous_home / part).exists()]
    if absent:
        raise FileNotFoundError(f"{previous_home} is not the home directory of a session: it has no {', '.join(absent)}")
    previous = json.loads((previous_home / CONFIG_RECORD).read_text())
    current = json.loads(json.dumps(asdict(config)))
    differing = sorted(name for name in current if name not in RUN_GRIDS and previous.get(name) != current[name])
    if differing:
        raise ValueError(f"{previous_home} was produced with a different {', '.join(differing)}: its checkpoints do not fit this configuration")
    for part in ("data", "artifacts"):
        copied = home / part
        shutil.copytree(previous_home / part, copied, dirs_exist_ok=True, copy_function=shutil.copyfile)
        for directory in (copied, *(path for path in copied.rglob("*") if path.is_dir())):
            directory.chmod(directory.stat().st_mode | stat.S_IWUSR)


def _timed(stage):
    """Makes a stage of `Experiment` record how long it took, in the "stage_minutes" report."""

    @functools.wraps(stage)
    def timed_stage(experiment: Experiment, *args, **kwargs):
        started = time.monotonic()
        try:
            return stage(experiment, *args, **kwargs)
        finally:
            experiment._record_stage_time(stage.__name__, time.monotonic() - started)

    return timed_stage


class Experiment:
    def __init__(
        self,
        config: ExperimentConfig,
        home: str | Path,
        device: torch.device,
        show_figures: bool = True,
        time_budget_hours: float | None = None,
    ) -> None:
        """`time_budget_hours` counts from now. Sweep runs that would not finish inside it are skipped and
        reported, so a session with a hard time limit ends with whatever was completed instead of with nothing."""
        self.config = config
        self.device = device
        self._deadline = None if time_budget_hours is None else time.monotonic() + time_budget_hours * 3600
        self._seconds_per_rl_step: float | None = None
        self.home = Path(home)
        self.log_directory = self.home / "data" / "processed"
        self.checkpoints = self.home / "artifacts"
        _bind_directory_to_config(self.home, config)
        self.reports = ReportWriter(self.checkpoints / "reports", show_figures)
        (self.reports.directory / CONFIG_RECORD).write_text((self.home / CONFIG_RECORD).read_text())
        self._examples: dict[str, EvalExamples] = {}
        self._test_evaluations: dict[str, PolicyEvaluation] = {}
        self._validation_metrics: dict[str, dict[str, float]] = {}
        self._reranked_evaluations: dict[str, ListEvaluation] = {}
        self._stage_seconds: dict[str, float] = {}

    def run(self, raw_events: pl.DataFrame, reactions: pl.DataFrame | None = None, item_metadata: pl.DataFrame | None = None) -> None:
        """All stages in order. `reactions` and `item_metadata` are what a dataset records beyond its interactions, if anything."""
        self.prepare_data(raw_events, reactions)
        self.pretrain_policy()
        self.train_reward_models()
        self.compare_baselines()
        self.finetune()
        self.sweep_kl_weight()
        self.sweep_kl_budget()
        self.compare_advantage_estimators()
        self.compare_simulated_feedback()
        self.explain_reward_gain()
        self.compare_alternatives_to_rl()
        self.sweep_rewards()
        self.sweep_relevance_in_reward()
        self.sweep_personal_rewards()
        self.sweep_relative_reward()
        self.compare_list_reward()
        self.ablate_sequence_credit()
        self.compare_feedback_masking()
        self.analyse_multistep()
        self.analyse_users_and_items(item_metadata)
        self.diagnose_return_label()
        self.restrict_finetuning_capacity()
        self.compare_group_relative_advantages()
        self.decompose_policy_change()
        self.align_at_inference()
        self.learn_from_logged_preferences()
        self.predict_lists_without_feedback()
        self.interpolate_weights()
        self.control_for_continued_training()


    @_timed
    def prepare_data(self, raw_events: pl.DataFrame, reactions: pl.DataFrame | None = None) -> pd.Series:
        """Builds the interaction log from raw events and, where the dataset has them, likes and dislikes (see `gptrec.data.preprocess`), and summarises it."""
        log_is_complete = (self.log_directory / "split.json").exists()
        if not log_is_complete:
            if self.config.max_users is not None:
                raw_events = _events_of_first_users(raw_events, self.config.max_users)
            build_interaction_log(raw_events, self.config.preprocessing, reactions).save(self.log_directory)
        self.reports.figure("feedback_overview", plots.feedback_overview(self.log))
        return self.reports.table("data_summary", _log_summary(self.log))


    @_timed
    def pretrain_policy(self) -> pd.DataFrame:
        """Trains the base recommender. Returns its test relevance, ranked and generated, next to the popularity floor."""
        if not self._pretrained_policy_path.exists():
            training = PolicyTrainingConfig(self.config.transformer, self.config.policy_optimization, self.config.window_stride)
            policy, history = train_policy(self.log, training, self.device, extra_validation=self._validation_relevance)
            save_model(policy, self._pretrained_policy_path)
            history = self.reports.table("policy_training", pd.DataFrame(history).set_index("epoch"))
            self.reports.figure("policy_training", plots.small_multiples(history, x_label="epoch"))

        evaluation = self._evaluate(self.pretrained_policy, "test", reward_models={})
        popularity_floor = popularity_baseline_metrics(self.examples("test"), self.log.items, self.config.evaluation_config())
        relevance = pd.DataFrame({
            "most popular": list_metrics(popularity_floor, "top_k"),
            "GPT top-k": list_metrics(evaluation.metrics, "top_k"),
            "GPT next-k": list_metrics(evaluation.metrics, "next_k"),
        })
        return self.reports.table("policy_test_relevance", relevance)


    @_timed
    def train_reward_models(self) -> pd.DataFrame:
        """Trains two reward models on disjoint halves of the users. Returns their played-ratio quality on the test period.

        "train" will produce the rewards during RL. "heldout" never influences the policy and only judges it:
        a reward gain that "heldout" does not confirm is exploitation of the errors of "train".
        """
        self._train_reward_model_pair("", self._reward_model_training())

        test_windows = EventWindows.for_heldout(self.log, "test", self.config.transformer.max_positions)
        predictions = {
            name: predict_heldout_feedback(reward_model, test_windows, self.config.evaluation_batch_size, self.device)
            for name, reward_model in self.reward_models.items()
        }
        items = self.log.items
        self.reports.table("reward_model_retention", self._retention_quality(test_predictions=predictions))
        self.reports.table("reward_model_reactions", pd.concat({name: reaction_quality(predicted) for name, predicted in predictions.items()}, names=["reward model", "signal"]))
        self.reports.table("reward_model_choice", pd.DataFrame({name: choice_quality(predicted) for name, predicted in predictions.items()}).T)
        self.reports.table("reward_signal_spread", signal_spread_over_candidates(self._scored_test_candidates, self.examples("test").future_items))
        calibration = {name: calibration_table(predicted.played_ratio, predicted.predicted_played_ratio) for name, predicted in predictions.items()}
        self.reports.figure("reward_model_calibration", plots.calibration_curves(calibration))
        self.reports.table("reward_model_error_by_popularity_decile", error_by_group(predictions["train"], _decile(items.train_popularity)))
        self.reports.table("reward_model_error_by_duration_decile", error_by_group(predictions["train"], _decile(items.duration_seconds)))
        self.reports.table("reward_model_quality_by_origin", quality_by_origin(predictions["train"]))
        agreement = np.corrcoef(predictions["train"].predicted_played_ratio, predictions["heldout"].predicted_played_ratio)[0, 1]
        self.reports.table("reward_model_agreement", pd.Series({"correlation of predicted played ratio": agreement}))

        relative_ratio_quality = pd.concat({name: relative_ratio_report(predicted) for name, predicted in predictions.items()}, names=["reward model", "predictor"])
        self.reports.table("reward_model_relative_ratio", relative_ratio_quality)
        played_ratio_quality = pd.concat(
            {name: reward_model_report(predicted, items) for name, predicted in predictions.items()},
            names=["reward model", "predictor"],
        )
        return self.reports.table("reward_model_played_ratio", played_ratio_quality)


    @_timed
    def compare_baselines(self) -> pd.DataFrame:
        """Fits the baselines and returns their test results next to the GPT recommender's, one column per model.

        Every model ranks the whole catalogue from the same histories under the same item constraints. The last
        column is the GPT recommender generating its list item by item instead of ranking.
        """
        baselines, histories = fit_baselines(
            self.log,
            self.config.transformer,
            self.config.baselines,
            self.config.window_stride,
            self.device,
            self.checkpoints / "baselines",
            extra_validation=self._validation_ranking_quality,
        )
        for name, history in histories.items():
            self.reports.table(f"baseline_training_{_file_name(name)}", pd.DataFrame(history).set_index("epoch"))
        rankers: dict[str, NextItemScorer] = baselines | {"GPT top-k": self.pretrained_policy}
        results = {name: self._evaluate_ranking(ranker).metrics for name, ranker in rankers.items()}
        results["GPT next-k"] = self._test_evaluation("pretrained", self.pretrained_policy).generated_lists().metrics
        return self.reports.table("baselines", pd.DataFrame(results))


    @_timed
    def finetune(self) -> pd.DataFrame:
        """Fine-tunes the recommender with RL. Returns every test metric of the base and the fine-tuned recommender."""
        if not self._rl_policy_path.exists():
            policy, records = self._run_rl(self.config.rl_config(), evaluate=self._monitor)
            save_model(policy, self._rl_policy_path)
            records = self.reports.table("rl_training", pd.DataFrame(records).set_index("step"))
            smoothed = records.reindex(columns=TRAINING_CURVES).rolling(TRAINING_CURVE_SMOOTHING_STEPS, min_periods=1).mean()
            self.reports.figure("rl_training_sampling", plots.small_multiples(smoothed, x_label="RL iteration"))
            self.reports.figure("rl_training_validation", plots.small_multiples(records.filter(like="validation/").dropna(), x_label="RL iteration"))

        pretrained, rl = self._test_evaluation("pretrained", self.pretrained_policy), self._test_evaluation("rl", self.rl_policy)
        pretrained.metrics.setdefault(REALISED_KL, 0.0)
        if REALISED_KL not in rl.metrics:
            rl.metrics |= self._end_of_training("rl_training")
        exposure = _exposure_change(self.log, before=pretrained, after=rl)
        self.reports.table("rl_exposure_change", exposure)
        attributes = ["popularity", "duration_seconds", "mean_played_ratio"]
        rank_correlation = exposure[attributes].rank().corrwith(exposure.exposure_change.rank())
        self.reports.table("rl_exposure_change_drivers", rank_correlation)
        self.reports.table("rl_list_overlap_with_pretrained", pd.Series({
            "next_k": list_overlap(rl.generated.actions, pretrained.generated.actions),
            "top_k": list_overlap(rl.top_k, pretrained.top_k),
        }))

        significance = paired_comparison(pretrained.generated_lists(), rl.generated_lists(), self.examples("test"), self.log.items, self.config.evaluation_config())
        self.reports.table("rl_vs_pretrained_significance", significance)

        comparison = metrics_table({"pretrained": pretrained, "rl": rl})
        comparison["relative_change"] = comparison["rl"] / comparison["pretrained"] - 1
        return self.reports.table("rl_vs_pretrained", comparison)


    @_timed
    def sweep_kl_weight(self) -> pd.DataFrame:
        """One fine-tuning run per KL weight: the frontier between relevance and predicted reward."""
        metrics = self._sweep("kl_sweep", self._kl_sweep_runs)
        k = max(self.config.ks)
        reward_rows = {judge: f"predicted/{judge}/played_ratio" for judge in self.reward_models}
        frontier = plots.reward_relevance_frontier(metrics, [f"top_k/all/ndcg@{k}", f"next_k/liked/ndcg@{k}"], reward_rows)
        self.reports.figure("kl_sweep_frontier", frontier)
        runs = {"RL, KL sweep": _runs_table({"pretrained": self._pretrained_lists} | self._generated_lists_of(self._kl_sweep_runs)).sort_values(REALISED_KL)}
        measures = [f"all/ndcg@{k}", "predicted/heldout/played_ratio", "profile/catalog_coverage"]
        self.reports.figure("kl_sweep_by_realised_kl", plots.trade_off(runs, reward_column=REALISED_KL, measure_columns=measures))
        if self.config.evaluate_sweeps_on_validation:
            self.reports.table("kl_sweep_selection", self._selection_on_validation(["pretrained", *self._kl_sweep_runs]))
        return metrics.reindex(self._summary_rows)

    @_timed
    def sweep_kl_budget(self) -> pd.DataFrame:
        """One fine-tuning run per KL budget: the anchor weight adapts until the policy sits that far from the base recommender.

        An anchor weight is measured against the standardised advantages, so the same weight is a different
        constraint under another estimator, reward or dataset. A budget is the constraint itself. If the runs
        land on the frontier of the weight sweep, the budget is a way of walking that frontier in comparable steps.
        """
        if not self.config.kl_budgets:
            return pd.DataFrame()
        metrics = self._sweep("kl_budget_sweep", self._kl_budget_runs)
        curves = {
            "fixed anchor weight": self._generated_lists_of(self._kl_sweep_runs),
            "KL budget": self._generated_lists_of(self._kl_budget_runs),
            "pretrained": {"pretrained": self._pretrained_lists},
        }
        self._trade_off_figures("kl_budget_sweep", curves, signals=JUDGED_SIGNALS[:1])
        return metrics.reindex(self._summary_rows)

    @_timed
    def compare_advantage_estimators(self) -> pd.DataFrame:
        """Does the anchor weight mean the same under another advantage estimator, and do the estimators differ at equal distance?

        "leave-one-out" replaces the value model by the mean return of the other lists sampled for the same
        history. The number of lists per iteration, the learning rate and the passes over each batch are those
        of the KL sweep, so only the baseline differs from its run at the same weight. Under a KL budget both
        estimators end equally far from the base recommender and are compared on what they found there. One
        more run multiplies the policy's learning rate, to see whether PPO's clipping ever takes part.
        """
        rollouts = self.config.leave_one_out_rollouts
        leave_one_out = {
            "advantage_estimator": AdvantageEstimator.LEAVE_ONE_OUT,
            "rollouts_per_history": rollouts,
            "histories_per_step": self.config.rl.histories_per_step // rollouts,
        }
        reward, main_weight = self.config.rl.reward, self.config.rl.reference_kl_weight
        at_equal_weight = {
            f"leave-one-out, kl={weight}": self.config.sweep_run(reward, weight, **leave_one_out)
            for weight in sorted(self.config.estimator_kl_weights, reverse=True)
        }
        at_equal_distance = {
            f"leave-one-out, kl_budget={budget}": self.config.sweep_run(reward, main_weight, target_kl=budget, **leave_one_out)
            for budget in (sorted(self.config.kl_budgets) if at_equal_weight else [])
        }
        faster = {}
        if self.config.high_learning_rate_factor is not None:
            factor = self.config.high_learning_rate_factor
            faster[f"learning rate x{factor}, kl={main_weight}"] = self.config.sweep_run(reward, main_weight, learning_rate=factor * self.config.rl.learning_rate)
        if not (at_equal_weight or faster):
            return pd.DataFrame()

        metrics = self._sweep("advantage_estimators", at_equal_weight | at_equal_distance | faster)
        curves = {
            "value model, fixed anchor weight": self._generated_lists_of(self._kl_sweep_runs),
            "value model, KL budget": self._generated_lists_of(self._kl_budget_runs),
            "leave-one-out, fixed anchor weight": self._generated_lists_of(at_equal_weight),
            "leave-one-out, KL budget": self._generated_lists_of(at_equal_distance),
            "higher learning rate": self._generated_lists_of(faster),
            "pretrained": {"pretrained": self._pretrained_lists},
        }
        k = max(self.config.ks)
        by_distance = {method: _runs_table(runs).sort_values(REALISED_KL) for method, runs in curves.items() if runs}
        measures = [f"all/ndcg@{k}", "predicted/heldout/played_ratio", "profile/catalog_coverage"]
        self.reports.figure("advantage_estimators_by_realised_kl", plots.trade_off(by_distance, reward_column=REALISED_KL, measure_columns=measures))
        return metrics.reindex(self._summary_rows)

    @_timed
    def compare_simulated_feedback(self) -> pd.DataFrame:
        """Fine-tuning with a sampled response to the generated items instead of the expected one.

        The expected played ratio of an item is typically near one half, a response the reward model has hardly
        seen: real ones are close to 0 or to 1. Sampling "finished" or "skipped" with the predicted probability
        keeps the simulated history inside what the model was trained on. Every policy is then judged under each
        assumption, which separates what a policy learned from how its lists are scored.
        """
        anchor_weights = sorted(self.config.sampled_feedback_kl_weights, reverse=True)
        if not anchor_weights:
            return pd.DataFrame()
        reward = self.config.rl.reward
        with_expected = {f"kl={weight}": self.config.sweep_run(reward, weight) for weight in anchor_weights}
        with_sampled = {
            f"sampled feedback, kl={weight}": self.config.sweep_run(reward, weight, simulated_feedback=SimulatedFeedback.SAMPLED) for weight in anchor_weights
        }
        metrics = self._sweep("simulated_feedback_sweep", with_expected | with_sampled)

        predicted_reward = {}
        for run_name in [*with_expected, *with_sampled]:
            if self._sweep_checkpoint(run_name).exists():
                policy = load_policy(self._sweep_checkpoint(run_name), self.device)
                predicted_reward[run_name] = {
                    mode.value: self._evaluate(policy, "test", simulated_feedback=mode).metrics["predicted/heldout/played_ratio"] for mode in SIMULATION_ASSUMPTIONS
                }
        by_assumption = pd.DataFrame(predicted_reward).T.rename_axis("fine-tuning run").rename_axis("feedback assumed when judging", axis=1)
        self.reports.table("simulated_feedback_cross_judged", by_assumption)
        return metrics.reindex(self._summary_rows)

    @_timed
    def explain_reward_gain(self) -> pd.DataFrame:
        """Whose reward is it? Returns, per method, how much of its predicted gain other users would give the same lists.

        A list earns `played_ratio` with the user it was made for and `played_ratio_for_other_user` with somebody
        else. If a method's gain over the base recommender shows up in full for other users, the method learned
        which items everybody finishes, not what this user will finish. The last rows re-rank the base
        recommender's candidates by each item's mean played ratio in the train period - no reward model and no
        RL - to see how far that knowledge alone goes.
        """
        k = max(self.config.ks)
        by_reward_model = self._reranked("re-ranking", self._scored_test_candidates, self.config.rl.reward)
        by_item_mean = self._reranked("re-ranking by item mean", self._candidates_scored_by_item_mean, RewardSpec())
        kl_sweep = self._generated_lists_of(self._kl_sweep_runs)
        methods = {"pretrained": self._pretrained_lists, "RL": self._rl_lists} | kl_sweep | by_reward_model | by_item_mean
        metrics = _runs_table(methods)
        with_own_user, with_other_user = metrics["predicted/heldout/played_ratio"], metrics["predicted/heldout/played_ratio_for_other_user"]
        gain = with_own_user - with_own_user["pretrained"]
        gain_with_other_user = with_other_user - with_other_user["pretrained"]
        decomposition = pd.DataFrame({
            f"all/ndcg@{k}": metrics[f"all/ndcg@{k}"],
            f"liked/ndcg@{k}": metrics[f"liked/ndcg@{k}"],
            "train-period mean played ratio of the listed items": metrics["profile/mean_item_played_ratio"],
            "predicted played ratio": with_own_user,
            "predicted played ratio for other users": with_other_user,
            "gain": gain,
            "gain for other users": gain_with_other_user,
            "personal share of the gain": (1 - gain_with_other_user / gain).where(gain.abs() > 1e-9),
        })
        curves = {
            "RL, KL sweep": kl_sweep,
            "re-ranking by the reward model": by_reward_model,
            "re-ranking by item mean": by_item_mean,
            "pretrained": {"pretrained": self._pretrained_lists},
            "RL, main run": {"RL": self._rl_lists},
        }
        self._trade_off_figures("reward_gain", curves, signals=JUDGED_SIGNALS[:2])
        return self.reports.table("reward_gain_decomposition", decomposition)

    @_timed
    def compare_alternatives_to_rl(self) -> pd.DataFrame:
        """Ways of raising the reward without on-policy RL, next to RL. Returns one column per method and setting.

        "liked-only fine-tuning" continues next-item training on liked items only: logged feedback, no reward model.
        "off-policy REINFORCE" (Matveev et al.) weighs every logged item by the logged return that followed it.
        "re-ranking" reorders the base recommender's candidates by predicted reward.
        "best of N sampled lists" (the test-time scaling of Matveev et al.) samples several lists and serves
        the one with the highest predicted return.
        The figures draw relevance against what the reward model that no method used predicts: RL earns its
        complexity only where its curve lies above the others.
        """
        pretrained, rl = self._pretrained_lists, self._test_evaluation("rl", self.rl_policy).generated_lists()
        liked_only = self._test_evaluation("liked-only", self._liked_only_policy()).generated_lists()
        off_policy = {
            f"off-policy REINFORCE, cap={cap}": self._test_evaluation(f"off-policy, cap={cap}", self._logged_return_policy(cap)).generated_lists()
            for cap in self.config.importance_weight_caps
        }
        reranking = self._reranked("re-ranking", self._scored_test_candidates, self.config.rl.reward)
        best_of_sampled = {f"best of {count} sampled lists": self._evaluate_lists(self._best_of_sampled_lists(count)) for count in self.config.sampled_list_counts}

        significance = paired_comparison(pretrained, liked_only, self.examples("test"), self.log.items, self.config.evaluation_config())
        self.reports.table("liked_only_vs_pretrained_significance", significance)
        curves = {
            "RL, KL sweep": self._generated_lists_of(self._kl_sweep_runs),
            "re-ranking, weight sweep": reranking,
            "best of N sampled lists": best_of_sampled,
            "off-policy REINFORCE, cap sweep": off_policy,
            "pretrained": {"pretrained": pretrained},
            "RL, main run": {"RL": rl},
            "liked-only fine-tuning": {"liked-only fine-tuning": liked_only},
        }
        self._trade_off_figures("alternatives_to_rl", curves, signals=JUDGED_SIGNALS[:2])

        methods = {"pretrained": pretrained, "RL": rl, "liked-only fine-tuning": liked_only} | off_policy | reranking | best_of_sampled
        return self.reports.table("alternatives_to_rl", _runs_table(methods).T)

    @_timed
    def sweep_rewards(self) -> pd.DataFrame:
        """One fine-tuning run per alternative reward and anchor weight.

        The retention rewards pay for what the user is predicted to do after the item: continue the session, or
        come back for another one. Each figure puts relevance against one signal as the held-out reward model
        predicts it, one line per reward running from the strongest anchor to the weakest.
        """
        anchor_weights = sorted(self.config.reward_kl_weights, reverse=True)
        runs_by_reward = {
            reward_name: {f"{reward_name}, kl={weight}": self.config.sweep_run(reward, weight) for weight in anchor_weights}
            for reward_name, reward in self.config.rewards
        }
        runs_by_reward |= {
            f"{reward_name}, discount=0": {f"{reward_name}, discount=0, kl={weight}": self.config.sweep_run(reward, weight, discount=0.0) for weight in anchor_weights}
            for reward_name, reward in self.config.rewards
            if reward.rewards_the_end_of_the_list
        }
        metrics = self._sweep("reward_sweep", {name: run for runs in runs_by_reward.values() for name, run in runs.items()})
        curves = {"played ratio": self._generated_lists_of(self._kl_sweep_runs)}
        curves |= {reward_name: self._generated_lists_of(runs) for reward_name, runs in runs_by_reward.items()}
        curves["pretrained"] = {"pretrained": self._pretrained_lists}
        self._trade_off_figures("reward_sweep", curves, signals=(*JUDGED_SIGNALS, *self._signals_rewarded_by(reward for _, reward in self.config.rewards)))
        return metrics.reindex(self._summary_rows)

    @staticmethod
    def _signals_rewarded_by(rewards) -> tuple[str, ...]:
        """The predicted signals, beyond JUDGED_SIGNALS, that some of `rewards` pay for."""
        signals = []
        for reward in rewards:
            signals += ["liked"] if reward.liked_weight or reward.disliked_weight else []
            signals += ["adopts"] if reward.adoption_weight else []
            signals += ["returns_after_list"] if reward.rewards_the_end_of_the_list else []
        return tuple(dict.fromkeys(signals))

    @_timed
    def sweep_relevance_in_reward(self) -> pd.DataFrame:
        """Can the reward itself ask for relevance, and what does the anchor add to it?

        "chosen ratio" weighs the predicted played ratio by the reward model's own estimate that the user
        would take the item up at all. "logged relevance" is the reward of Petrov & Macdonald: the item is among
        the user's real next items. It runs alone with no anchor (their NDCG tuning), and added to the played
        ratio at several weights and several anchor weights: the reward term protects hits on the user's real
        continuation, the anchor protects everything else the base recommender does, such as its variety.
        """
        chosen_ratio = {f"chosen ratio, kl={weight}": self.config.sweep_run(CHOSEN_RATIO, weight) for weight in sorted(self.config.choice_reward_kl_weights, reverse=True)}
        logged_relevance_only = {"logged relevance, kl=0.0": self.config.sweep_run(LOGGED_RELEVANCE, kl_weight=0.0)}
        anchor_weights = sorted(self.config.logged_relevance_kl_weights, reverse=True)
        ratio_and_logged_relevance = {
            f"ratio + {weight} x logged relevance": {
                f"ratio + {weight} x logged relevance, kl={anchor}": self.config.sweep_run(RewardSpec(logged_relevance_weight=weight), anchor)
                for anchor in anchor_weights
            }
            for weight in sorted(self.config.logged_relevance_weights, reverse=True)
        }
        anchored_runs = {name: run for anchor_sweep in ratio_and_logged_relevance.values() for name, run in anchor_sweep.items()}
        metrics = self._sweep("relevance_reward_sweep", chosen_ratio | logged_relevance_only | anchored_runs)
        curves = {"played ratio, anchor sweep": self._generated_lists_of(self._kl_sweep_runs), "chosen ratio, anchor sweep": self._generated_lists_of(chosen_ratio)}
        curves |= {f"{reward_name}, anchor sweep": self._generated_lists_of(anchor_sweep) for reward_name, anchor_sweep in ratio_and_logged_relevance.items()}
        curves |= {"logged relevance only, no anchor": self._generated_lists_of(logged_relevance_only), "pretrained": {"pretrained": self._pretrained_lists}}
        self._trade_off_figures("relevance_reward_sweep", curves, signals=JUDGED_SIGNALS[:2])
        return metrics.reindex(self._summary_rows)

    @_timed
    def sweep_personal_rewards(self) -> pd.DataFrame:
        """Fine-tuning for the part of the played ratio that depends on who receives the list.

        "personal ratio" pays the predicted played ratio minus what the same reward model predicts for the same
        list shown to another user; a policy that ignores its user earns zero, so collapsing onto the items
        everybody finishes no longer pays. "ratio above item mean" subtracts the item's mean played ratio in the
        train period instead, which needs no second prediction but leaves the reward model's errors about
        items in the reward. Re-ranking by the personal ratio shows what the same reward gives without RL.
        """
        anchor_weights = sorted(self.config.personal_reward_kl_weights, reverse=True)
        against_other_user = {f"personal ratio, kl={weight}": self.config.sweep_run(PERSONAL_RATIO, weight) for weight in anchor_weights}
        against_item_mean = {f"ratio above item mean, kl={weight}": self.config.sweep_run(RATIO_ABOVE_ITEM_MEAN, weight) for weight in anchor_weights}
        metrics = self._sweep("personal_reward_sweep", against_other_user | against_item_mean)
        curves = {
            "played ratio, anchor sweep": self._generated_lists_of(self._kl_sweep_runs),
            "personal ratio, anchor sweep": self._generated_lists_of(against_other_user),
            "ratio above item mean, anchor sweep": self._generated_lists_of(against_item_mean),
            "re-ranking by personal ratio": self._reranked("re-ranking by personal ratio", self._scored_test_candidates, PERSONAL_RATIO),
            "pretrained": {"pretrained": self._pretrained_lists},
        }
        self._trade_off_figures("personal_reward_sweep", curves, signals=("personal_played_ratio", "played_ratio"))
        return metrics.reindex(self._summary_rows)

    @_timed
    def sweep_relative_reward(self) -> pd.DataFrame:
        """Fine-tuning for the played ratio relative to what is usual for the item.

        The plain played ratio pays for items that are easy to finish, whoever receives them: a film over a
        series, a short track over a long one. The relative ratio is the user's played ratio as a quantile among
        everyone who consumed the same item. It is 0.5 for every item on average, so a policy earns it only by
        matching items to the users who take to them more than others do. Re-ranking the base recommender's
        candidates by the same signal is the reference without RL.
        """
        anchor_weights = sorted(self.config.relative_reward_kl_weights, reverse=True)
        if not anchor_weights:
            return pd.DataFrame()
        runs = {f"relative ratio, kl={weight}": self.config.sweep_run(RELATIVE_RATIO, weight) for weight in anchor_weights}
        metrics = self._sweep("relative_reward_sweep", runs)
        reranked = self._reranked("re-ranking by relative ratio", self._scored_test_candidates, RELATIVE_RATIO)
        self.reports.table("relative_reward_reranking", _runs_table(reranked).T)
        curves = {
            "played ratio, anchor sweep": self._generated_lists_of(self._kl_sweep_runs),
            "relative ratio, anchor sweep": self._generated_lists_of(runs),
            "re-ranking by relative ratio": reranked,
            "pretrained": {"pretrained": self._pretrained_lists},
        }
        self._trade_off_figures("relative_reward_sweep", curves, signals=("relative_ratio", "played_ratio"))
        return metrics.reindex(self._summary_rows)

    @_timed
    def compare_list_reward(self) -> pd.DataFrame:
        """A reward that depends on the list as a whole: the main run's reward plus each item's distance to the items before it.

        Summed over a list, the second term is the list's mean pairwise distance in the base recommender's
        embedding space. An item's worth then depends on its neighbours, and scoring items one at a time no
        longer finds the best list. RL is compared with the greedy ways of maximising the same reward: maximal
        marginal relevance over the base recommender's candidates, which fills the list one position at a time,
        and the best of several sampled lists. The last column of the saved table is the reward itself as the
        held-out reward model sees it, per item of the list.
        """
        if not self.config.diversity_weights:
            return pd.DataFrame()
        k = max(self.config.ks)
        anchor_weights = sorted(self.config.diversity_kl_weights, reverse=True)
        rewards = {f"ratio + {weight} x distance": replace(self.config.rl.reward, diversity_weight=weight) for weight in sorted(self.config.diversity_weights)}
        rl_runs = {label: {f"{label}, kl={anchor}": self.config.sweep_run(reward, anchor) for anchor in anchor_weights} for label, reward in rewards.items()}
        metrics = self._sweep("list_reward_sweep", {name: run for runs in rl_runs.values() for name, run in runs.items()})

        methods = {}
        for label, reward in rewards.items():
            curves = {
                "RL, anchor sweep": self._generated_lists_of(rl_runs[label]),
                "greedy re-ranking, weight sweep": self._reranked(f"greedy re-ranking, {label}", self._scored_test_candidates, reward),
                "best of N sampled lists": {
                    f"best of {count} sampled lists, {label}": self._evaluate_lists(self._best_of_sampled_lists(count, reward)) for count in self.config.sampled_list_counts
                },
                "played ratio only, anchor sweep": self._generated_lists_of(self._kl_sweep_runs),
                "pretrained": {"pretrained": self._pretrained_lists},
            }
            measures = [f"all/ndcg@{k}", INTRA_LIST_DISTANCE, "profile/catalog_coverage"]
            self._trade_off_figures(f"list_reward_{_file_name(label)}", curves, signals=JUDGED_SIGNALS[:1], measures=measures)
            table = _runs_table({name: evaluation for runs in curves.values() for name, evaluation in runs.items()})
            table["list reward (held-out)"] = table["predicted/heldout/played_ratio"] + reward.diversity_weight * table[INTRA_LIST_DISTANCE] / 2
            methods[label] = table
        self.reports.table("list_reward_methods", pd.concat(methods, names=["reward", "method"]))
        return metrics.reindex(self._summary_rows)

    @_timed
    def ablate_sequence_credit(self) -> pd.DataFrame:
        """Does fine-tuning use the fact that a list is a sequence?

        "discount=0" credits every item with its own reward only, never with the rewards of the items after it.
        "horizon=1" samples one item per episode while fine-tuning, so nothing is learned from a generated
        context; its lists are still generated item by item when evaluated. Both are compared with runs that
        credit every item with the whole rest of the list: at the same anchor weights, and under the same KL
        budgets. Changing the credit changes what a weight means, so only the budget runs compare policies
        that moved equally far; for one-item episodes the distance is that of the first position only.
        """
        anchor_weights = sorted(self.config.credit_ablation_kl_weights, reverse=True)
        budgets = sorted(self.config.kl_budgets)
        reward, one_item, main_weight = self.config.rl.reward, replace(self.config.rl.generation, horizon=1), self.config.rl.reference_kl_weight
        own_reward_only = {f"discount=0, kl={weight}": self.config.sweep_run(reward, weight, discount=0.0) for weight in anchor_weights}
        own_reward_only |= {f"discount=0, kl_budget={budget}": self.config.sweep_run(reward, main_weight, discount=0.0, target_kl=budget) for budget in budgets}
        one_item_episodes = {f"horizon=1, kl={weight}": self.config.sweep_run(reward, weight, generation=one_item) for weight in anchor_weights}
        one_item_episodes |= {f"horizon=1, kl_budget={budget}": self.config.sweep_run(reward, main_weight, generation=one_item, target_kl=budget) for budget in budgets}
        metrics = self._sweep("sequence_credit_ablation", own_reward_only | one_item_episodes)
        whole_rest_of_the_list = [f"kl={weight}" for weight in anchor_weights] + [f"kl_budget={budget}" for budget in budgets]
        curves = {
            "whole rest of the list": self._generated_lists_of(whole_rest_of_the_list),
            "own reward only (discount 0)": self._generated_lists_of(own_reward_only),
            "one item per episode (horizon 1)": self._generated_lists_of(one_item_episodes),
            "pretrained": {"pretrained": self._pretrained_lists},
        }
        self._trade_off_figures("sequence_credit_ablation", curves, signals=JUDGED_SIGNALS[:1])
        return metrics.reindex(self._summary_rows)

    @_timed
    def analyse_multistep(self) -> pd.DataFrame:
        """How the generated list changes along its positions. Returns NDCG of ranked against generated lists.

        From the second position on, the recommender conditions on items the user never consumed and the
        reward model on feedback that was never observed; drift shows as a trend over positions.
        """
        examples, items = self.examples("test"), self.log.items
        evaluations = {"pretrained": self._test_evaluation("pretrained", self.pretrained_policy), "rl": self._test_evaluation("rl", self.rl_policy)}
        step_tables = {name: self.reports.table(f"per_step_{name}", per_step_table(evaluation, examples, items)) for name, evaluation in evaluations.items()}
        measures = [
            "in_true_continuation",
            "policy_entropy",
            "predicted_played_ratio/train",
            "predicted_is_chosen/train",
            "predicted_returns/train",
            "played_ratio_disagreement/train_vs_heldout",
        ]
        by_measure = {measure: pd.DataFrame({name: table[measure] for name, table in step_tables.items()}) for measure in measures}
        self.reports.figure("per_step", plots.series_per_panel(by_measure, x_label="position in the generated list"))

        predicted_by_mode = {}
        for mode in SIMULATION_ASSUMPTIONS:
            evaluation = self._evaluate(self.rl_policy, "test", simulated_feedback=mode)
            predicted_by_mode[mode.value] = per_step_table(evaluation, examples, items)["predicted_played_ratio/train"]
        predicted_by_mode = self.reports.table("per_step_by_simulated_feedback", pd.DataFrame(predicted_by_mode))
        by_mode_figure = plots.series_per_panel({"predicted played ratio (RL policy)": predicted_by_mode}, x_label="position in the generated list")
        self.reports.figure("per_step_by_simulated_feedback", by_mode_figure)

        lists = {f"{name}, {kind}": lists for name, evaluation in evaluations.items() for kind, lists in (("ranked", evaluation.top_k), ("generated", evaluation.generated.actions))}
        self.reports.table("position_hits", position_hits(lists, examples.future_items))

        reward_models = {"train": self.reward_models["train"]}
        if self._reward_model_path("masked_train").exists():
            reward_models["masked train"] = self.masked_reward_models["masked_train"]
        steps = min(FEEDBACK_ASSUMPTION_STEPS, self.config.horizon)
        assumptions = {}
        for name, reward_model in reward_models.items():
            modes = [SimulatedFeedback.LOGGED, *SIMULATION_ASSUMPTIONS] + ([SimulatedFeedback.UNKNOWN] if reward_model.config.unknown_feedback else [])
            assumptions[name] = feedback_assumptions_on_real_sequences(
                reward_model, examples, modes, steps, self.config.evaluation_batch_size, self.device, mixed_precision=True
            )
        self.reports.table("feedback_assumptions_on_real_sequences", pd.concat(assumptions, axis=1, names=["reward model", "quantity"]))

        generation_gap = pd.DataFrame({
            (name, list_name): {k: evaluation.metrics[f"{list_name}/all/ndcg@{k}"] for k in self.config.ks}
            for name, evaluation in evaluations.items()
            for list_name in ("top_k", "next_k")
        }).rename_axis("k")
        return self.reports.table("ndcg_top_k_vs_next_k", generation_gap)

    @_timed
    def compare_feedback_masking(self) -> pd.DataFrame:
        """A reward model that can be told "the response to this item is unknown", against models that must be told an assumed one.

        The masked reward models are trained like the others, on the same halves of the users, except that in a
        share of the training windows the responses to the last events are hidden (see `hide_last_responses`).
        A generated item can then be appended as what it is, shown with an unknown response, and the simulation
        needs no assumption. The policy is fine-tuned against the masked training model with unknown feedback and
        judged like every other run; the runs of the anchor sweep are also judged by the masked held-out model.
        """
        anchor_weights = sorted(self.config.masked_feedback_kl_weights, reverse=True)
        if not anchor_weights:
            return pd.DataFrame()
        masked = self.masked_reward_models
        test_windows = EventWindows.for_heldout(self.log, "test", self.config.transformer.max_positions)
        quality = {
            name: reward_model_report(predict_heldout_feedback(reward_model, test_windows, self.config.evaluation_batch_size, self.device), self.log.items).loc["reward model"]
            for name, reward_model in (self.reward_models | masked).items()
        }
        self.reports.table("reward_model_masked_quality", pd.DataFrame(quality).T)

        runs = {
            f"unknown feedback, kl={weight}": self.config.sweep_run(self.config.rl.reward, weight, simulated_feedback=SimulatedFeedback.UNKNOWN)
            for weight in anchor_weights
        }
        metrics = self._sweep("masked_feedback_sweep", runs, reward_model=masked["masked_train"])

        judged_by_masked_model = {}
        for run_name in ["pretrained", *self._kl_sweep_runs, *runs]:
            if run_name == "pretrained" or self._sweep_checkpoint(run_name).exists():
                policy = self.pretrained_policy if run_name == "pretrained" else load_policy(self._sweep_checkpoint(run_name), self.device)
                evaluation = self._evaluate(policy, "test", reward_models={"masked_heldout": masked["masked_heldout"]}, simulated_feedback=SimulatedFeedback.UNKNOWN)
                judged_by_masked_model[run_name] = {
                    "predicted played ratio, unknown feedback": evaluation.metrics["predicted/masked_heldout/played_ratio"],
                    "predicted played ratio, expected feedback": self._sweep_or_pretrained_metric(run_name, "predicted/heldout/played_ratio"),
                }
        self.reports.table("masked_feedback_judged", pd.DataFrame(judged_by_masked_model).T)
        return metrics.reindex(self._summary_rows)

    def _sweep_or_pretrained_metric(self, run_name: str, metric: str) -> float:
        evaluation = self._test_evaluation("pretrained", self.pretrained_policy) if run_name == "pretrained" else self._sweep_evaluation(run_name)
        return evaluation.metrics[metric]

    @_timed
    def analyse_users_and_items(self, item_metadata: pl.DataFrame | None = None) -> pd.DataFrame:
        """Who loses relevance, what the policy learned about items, and how impressions spread over the catalogue.

        Users are grouped by how many events they had before the request. The logit shift asks whether the
        fine-tuned policy changed its preferences per user or one item at a time. With `item_metadata` (item_id
        plus attributes such as the title and whether it is a film or a series), the items that gained and lost
        the most impressions are named. Returns the logit-shift table: one row per fine-tuned policy.
        """
        examples, items, k = self.examples("test"), self.log.items, max(self.config.ks)
        pretrained, rl = self._test_evaluation("pretrained", self.pretrained_policy), self._test_evaluation("rl", self.rl_policy)
        user_starts = np.asarray(self.log.user_offsets[:-1])
        test_starts, _ = self.log.period_rows("test")
        users = examples.user.numpy()
        prior_events = test_starts[users] - user_starts[users]
        main_lists = {"pretrained, ranked": pretrained.top_k, "pretrained, generated": pretrained.generated.actions, "RL, generated": rl.generated.actions}
        by_length = relevance_by_history_length(main_lists, examples, prior_events, k, self.config.evaluation_config().liked_threshold)
        self.reports.table("relevance_by_history_length", by_length)

        policies = {"RL": self.rl_policy} | {name: load_policy(self._sweep_checkpoint(name), self.device) for name in self._kl_sweep_runs if self._sweep_checkpoint(name).exists()}
        shift = pd.DataFrame({
            name: logit_shift_by_item(policy, self.pretrained_policy, examples, items, self.config.evaluation_batch_size, self.device) for name, policy in policies.items()
        }).T
        self.reports.table("logit_shift_by_item", shift)

        swept_lists = {name: self._sweep_evaluation(name).generated.actions for name in self._kl_sweep_runs if self._sweep_checkpoint(name).exists()}
        all_lists = main_lists | swept_lists
        self.reports.table("exposure_lorenz", exposure_lorenz(all_lists, items.num_items))
        concentration = {
            name: {"catalog_coverage": profile["catalog_coverage"], "item_gini": profile["item_gini"], "top_exposure_share": profile["top_exposure_share"]}
            for name, lists in all_lists.items()
            for profile in [recommendation_profile(lists, examples.history.items, items)]
        }
        self.reports.table("exposure_concentration", pd.DataFrame(concentration).T)

        if item_metadata is not None:
            item_table = self._item_table(item_metadata)
            if "content_type" in item_table:
                self.reports.table("impressions_by_content_type", impressions_by_attribute(all_lists, item_table["content_type"].to_numpy()))
            self.reports.table("largest_impression_changes", largest_impression_changes(pretrained.generated.actions, rl.generated.actions, item_table))
        return shift

    def _item_table(self, item_metadata: pl.DataFrame) -> pd.DataFrame:
        """One row per item token (row 0 is padding): the attributes of `item_metadata` next to the log's statistics of the item."""
        items = self.log.items
        tokens = pd.DataFrame({
            "item_id": items.raw_id,
            "train_popularity": items.train_popularity,
            "mean_played_ratio": items.mean_played_ratio,
            "duration_seconds": items.duration_seconds,
        })
        metadata = pd.DataFrame({column: item_metadata[column].to_numpy() for column in item_metadata.columns}).drop_duplicates("item_id")
        return tokens.merge(metadata, on="item_id", how="left").set_axis(pd.RangeIndex(len(tokens), name="item_token"))



    @_timed
    def diagnose_return_label(self) -> pd.DataFrame:
        """Whether a user's return after an event is recorded by the event or by the item that came after it.

        Data only. On a log with one row per user and item, dated by the last view, the item after an event decides
        when the next row appears; a reward for returns then pays for the bookkeeping, see `return_rate_by_duration`.
        """
        if not self.config.return_label_check:
            return pd.DataFrame()
        return self.reports.table("return_rate_by_duration", return_rate_by_duration(self.log))

    @_timed
    def restrict_finetuning_capacity(self) -> pd.DataFrame:
        """How much of what RL reaches needs more than one offset per item: fine-tuning held to item offsets or to low-rank updates.

        Every run holds the same KL budgets, so they are compared at an equal distance from the base recommender.
        Item offsets can only re-weight the catalogue, for everybody alike. Low-rank updates of the transformer change
        how a history is read, but no item's embedding. If the offsets reach the full run's predicted reward at its
        relevance, what RL learned is a re-weighting of the catalogue. If rank 1 matches full fine-tuning, a policy
        gradient carried as little information per episode as is reported for language models.
        """
        if not self.config.capacity_kl_budgets:
            return pd.DataFrame()
        reward, weight = self.config.rl.reward, self.config.rl.reference_kl_weight
        full = {f"kl_budget={budget}": self.config.sweep_run(reward, weight, target_kl=budget) for budget in sorted(self.config.capacity_kl_budgets)}
        capacity_runs = self._capacity_runs
        metrics = self._sweep("capacity_sweep", full | capacity_runs)
        curves = {"every weight": self._generated_lists_of(full)}
        for label in dict.fromkeys(name.split(", kl_budget=")[0] for name in capacity_runs):
            curves[f"{label} only"] = self._generated_lists_of({name: run for name, run in capacity_runs.items() if name.startswith(f"{label},")})
        curves["pretrained"] = {"pretrained": self._pretrained_lists}
        self._trade_off_figures("capacity_sweep", curves, signals=JUDGED_SIGNALS[:1])
        return metrics.reindex(self._summary_rows)

    @_timed
    def compare_group_relative_advantages(self) -> pd.DataFrame:
        """GRPO (Shao et al., 2024), the estimator of current generative-recommender RL, at the KL budgets of the other estimators.

        Group-relative advantages compare the lists sampled for one history with each other and divide by their
        spread, so every history weighs the same; nothing else differs from the leave-one-out runs.
        """
        runs = self._group_relative_runs
        if not runs:
            return pd.DataFrame()
        metrics = self._sweep("group_relative_advantages", runs)
        leave_one_out = {f"leave-one-out, kl_budget={budget}": None for budget in sorted(self.config.group_relative_kl_budgets)}
        curves = {
            "value model": self._generated_lists_of(self._kl_budget_runs),
            "leave-one-out": self._generated_lists_of(leave_one_out),
            "group-relative": self._generated_lists_of(runs),
            "pretrained": {"pretrained": self._pretrained_lists},
        }
        self._trade_off_figures("group_relative_advantages", curves, signals=JUDGED_SIGNALS[:1])
        return metrics.reindex(self._summary_rows)

    @_timed
    def decompose_policy_change(self) -> pd.DataFrame:
        """What the main run changed, split into one offset per item and the rest, and whether the reward predicted the split.

        The run's change of log-probabilities is averaged per item over validation histories. "base + RL's item
        offsets" adds those averages to the base recommender; "RL - its item offsets" removes them from the
        fine-tuned one. If the first reproduces RL and the second does little, RL learned a re-weighting of the
        catalogue. The returned table then compares, on the base recommender's candidates for every test user, the
        share of within-history variation that one effect per item explains: in the reward model's predictions,
        which the closed form pi* = pi_0 exp(r / beta) / Z turns into the optimal change of log-probabilities at any
        beta, and in what each fine-tuned recommender changed. RL above the reward's share amplified the item part.
        """
        if not self.config.analyse_policy_change:
            return pd.DataFrame()
        offsets = mean_logit_shift_per_item(self.rl_policy, self.pretrained_policy, self.examples("validation"), self.config.evaluation_batch_size, self.device)
        parts = {
            "pretrained": self._pretrained_lists,
            "RL": self._rl_lists,
            "base + RL's item offsets": self._test_evaluation("base + RL's item offsets", with_item_offsets(self.pretrained_policy, offsets)).generated_lists(),
            "RL - its item offsets": self._test_evaluation("RL - its item offsets", with_item_offsets(self.rl_policy, -offsets)).generated_lists(),
        }
        self.reports.table("policy_change_parts", _runs_table(parts).T)

        candidates, examples, batch_size = self._scored_test_candidates, self.examples("test"), self.config.evaluation_batch_size
        weights = candidates.log_probs.exp()
        reward = RewardFunction(self.config.rl.reward, self.log.items, "cpu").item_reward(
            candidates.feedback, candidates.items, feedback_for_other_user=candidates.feedback_for_other_user
        )
        shares = {"reward model's predictions (the optimum's change)": item_effect_share(reward, candidates.items, weights)}
        base_log_probs = candidate_log_probs(self.pretrained_policy, examples, candidates.items, batch_size, self.device)
        for name, policy in self._finetuned_policies().items():
            shift = candidate_log_probs(policy, examples, candidates.items, batch_size, self.device) - base_log_probs
            shares[name] = item_effect_share(shift, candidates.items, weights)
        return self.reports.table("item_effect_share", pd.Series(shares, name="within-history variation explained by one effect per item"))

    @_timed
    def align_at_inference(self) -> pd.DataFrame:
        """The base recommender steered at inference by the closed form of the anchored objective: no fine-tuning.

        Each position takes the candidate with the highest log pi_0 + Q / beta among the base recommender's most likely
        next items, after the items already chosen (see `guided_lists`). With discount 0, Q is the item's own reward;
        above 0 it adds the learned worth of what follows. Re-ranking once after the real history is the same closed
        form without either. For a reward paid only after the last item, nothing but look-ahead can see it coming.
        Returns one column per method: RL, re-ranking, and each reward, discount and strength of guidance.
        """
        strengths = sorted(self.config.guidance_strengths, reverse=True)
        if not strengths:
            return pd.DataFrame()
        guided_by_reward = {}
        for reward_name, reward in self.config.guided_rewards:
            reward_function = RewardFunction(reward, self.log.items, self.device)
            guided = {}
            for discount in self.config.guidance_discounts:
                value_model = self._continuation_value_model(reward_name, reward, discount) if discount > 0 else None
                for strength in strengths:
                    name = f"{reward_name}, guided, discount={discount}, strength={strength}"
                    guidance = GuidanceConfig(strength=strength, discount=discount, num_candidates=self.config.guidance_candidates)
                    guided[name] = self._evaluate_lists(self._guided_lists(reward_function, value_model, guidance))
            guided_by_reward[reward_name] = guided

        reranked = self._reranked("re-ranking", self._scored_test_candidates, self.config.rl.reward)
        rl_on_rewards = self._generated_lists_of({f"{name}, kl={weight}": None for name, _ in self.config.rewards for weight in self.config.reward_kl_weights})
        for (reward_name, reward), guided in zip(self.config.guided_rewards, guided_by_reward.values()):
            curves = {"RL, anchor sweep": self._generated_lists_of(self._kl_sweep_runs), "re-ranking once, real history": reranked}
            for discount in self.config.guidance_discounts:
                curves[f"guided, discount={discount}"] = {name: lists for name, lists in guided.items() if f", discount={discount}," in name}
            curves["RL on this reward"] = {name: lists for name, lists in rl_on_rewards.items() if name.startswith(f"{reward_name}, kl=")}
            curves["pretrained"] = {"pretrained": self._pretrained_lists}
            self._trade_off_figures(f"inference_time_alignment_{_file_name(reward_name)}", curves, signals=(*JUDGED_SIGNALS[:1], *self._signals_rewarded_by([reward])))

        methods = {"pretrained": self._pretrained_lists, "RL": self._rl_lists} | rl_on_rewards | reranked
        for guided in guided_by_reward.values():
            methods |= guided
        return self.reports.table("inference_time_alignment", _runs_table(methods).T)

    @_timed
    def learn_from_logged_preferences(self) -> pd.DataFrame:
        """Fine-tuning on logged responses alone, with current preference-learning methods: no reward model, no generated lists.

        Reward-weighted fine-tuning weighs each logged item by exp(response / temperature); within-user DPO learns
        from pairs of items one user consumed after the same history, one above and one below the usual response for
        it. Each method is compared with the base recommender user by user, and with RL and liked-only fine-tuning.
        Returns one column per method.
        """
        temperatures, betas = self.config.reward_weighted_temperatures, self.config.preference_betas
        if not (temperatures or betas):
            return pd.DataFrame()
        methods = {}
        for response in self.config.reward_weighted_responses:
            for temperature in temperatures:
                name = f"reward-weighted, {response}, temperature={temperature}"
                methods[name] = self._test_evaluation(name, self._reward_weighted_policy(response, temperature)).generated_lists()
        for beta in betas:
            name = f"within-user DPO, beta={beta}"
            methods[name] = self._test_evaluation(name, self._logged_preference_policy(beta)).generated_lists()

        pretrained, evaluation_config = self._pretrained_lists, self.config.evaluation_config()
        for name, lists in methods.items():
            significance = paired_comparison(pretrained, lists, self.examples("test"), self.log.items, evaluation_config)
            self.reports.table(f"logged_preferences_significance_{_file_name(name)}", significance)
        liked_only = self._test_evaluation("liked-only", self._liked_only_policy()).generated_lists()
        k = max(self.config.ks)
        curves = {
            "RL, anchor sweep": self._generated_lists_of(self._kl_sweep_runs),
            **{name: {name: lists} for name, lists in methods.items()},
            "liked-only fine-tuning": {"liked-only fine-tuning": liked_only},
            "pretrained": {"pretrained": pretrained},
        }
        measures = [f"all/ndcg@{k}", "logged/relative_surplus_per_item", "profile/catalog_coverage"]
        self._trade_off_figures("logged_preferences", curves, signals=JUDGED_SIGNALS[:1], measures=measures)
        everything = {"pretrained": pretrained, "RL": self._rl_lists, "liked-only fine-tuning": liked_only} | methods
        return self.reports.table("logged_preference_methods", _runs_table(everything).T)

    @_timed
    def predict_lists_without_feedback(self) -> pd.DataFrame:
        """Lists from one head per position, read off the real history, against ranked and generated lists of the base recommender.

        Generation feeds each chosen item back as if it had been consumed; ranking scores every position from the real
        history but with one distribution. One head per position does neither. The position hits show what each costs
        after a miss and gains after a hit. Returns one column per kind of list.
        """
        if not self.config.multi_token_lists:
            return pd.DataFrame()
        heads, generation = self._multi_token_heads(), self.config.evaluation_config().generation

        def one_head_per_position(history):
            return multi_token_lists(self.pretrained_policy, heads, history, generation)

        examples = self.examples("test")
        predicted = collect_lists(one_head_per_position, examples, self.config.evaluation_batch_size, self.device)
        pretrained = self._test_evaluation("pretrained", self.pretrained_policy)
        lists = {"ranked": pretrained.top_k, "generated": pretrained.generated.actions, "one head per position": predicted}
        self.reports.table("multi_token_position_hits", position_hits(lists, examples.future_items))
        evaluations = {"ranked": self._evaluate_lists(pretrained.top_k), "generated": pretrained.generated_lists(), "one head per position": self._evaluate_lists(predicted)}
        return self.reports.table("multi_token_lists", _runs_table(evaluations).T)

    @_timed
    def interpolate_weights(self) -> pd.DataFrame:
        """Operating points from weight space: theta_0 + a (theta_RL - theta_0) for each factor a, against the anchor sweep.

        No point needs training. Below 1 the factor interpolates towards the base recommender (WiSE-FT, WARP);
        above 1 it extrapolates past the main run (ExPO). Every recommender's distance from the base one is measured
        the same way, as the KL along its own lists, since an interpolated one has no training record.
        Returns one column per recommender.
        """
        factors = sorted(self.config.interpolation_factors)
        if not factors:
            return pd.DataFrame()
        generation, examples = self.config.evaluation_config().generation, self.examples("test")

        def with_distance(name: str, policy: NextItemPolicy) -> ListEvaluation:
            evaluation = self._test_evaluation(name, policy)
            lists = evaluation.generated_lists()
            lists.metrics["kl_along_own_lists"] = kl_along_lists(policy, self.pretrained_policy, examples, evaluation.generated.actions, generation, KL_BATCH_SIZE, self.device)
            return lists

        interpolations = {f"weights x{factor}": with_distance(f"weights x{factor}", interpolated(self.pretrained_policy, self.rl_policy, factor)) for factor in factors}
        anchor_sweep = {name: with_distance(name, load_policy(self._sweep_checkpoint(name), self.device)) for name in self._kl_sweep_runs if self._sweep_checkpoint(name).exists()}
        pretrained = with_distance("pretrained", self.pretrained_policy)
        main_run = with_distance("rl", self.rl_policy)
        curves = {
            "RL, anchor sweep": anchor_sweep,
            "weights between and beyond": interpolations | {"weights x1.0": main_run},
            "pretrained": {"pretrained": pretrained},
        }
        self._trade_off_figures("weight_interpolation", curves, signals=JUDGED_SIGNALS[:1])
        k = max(self.config.ks)
        by_distance = {method: _runs_table(runs).sort_values("kl_along_own_lists") for method, runs in curves.items() if runs}
        measures = [f"all/ndcg@{k}", "predicted/heldout/played_ratio", "profile/catalog_coverage"]
        self.reports.figure("weight_interpolation_by_kl", plots.trade_off(by_distance, reward_column="kl_along_own_lists", measure_columns=measures))
        everything = {"pretrained": pretrained, "RL": main_run} | anchor_sweep | interpolations
        return self.reports.table("weight_interpolation", _runs_table(everything).T)

    @_timed
    def control_for_continued_training(self) -> pd.DataFrame:
        """Learning from the log against the same epoch of training without its weights, on ranked as well as generated lists.

        Reward-weighted fine-tuning and within-user DPO train the base recommender for one more epoch at a lower
        learning rate. The base recommender stopped when its validation loss stopped falling, before its learning rate
        had decayed, so that epoch may raise relevance by itself. Continued training, in which every logged item weighs
        one, is the epoch alone: what a method gains over it is what its weights or its preferences add. Every
        recommender is judged on its ranked lists too, since the base recommender's ranked lists are its stronger ones
        on some datasets. Returns one column per recommender and kind of list.
        """
        if not self.config.continued_training_control:
            return pd.DataFrame()
        recommenders = {"pretrained": self.pretrained_policy, CONTINUED_TRAINING: self._continued_training_policy()}
        for response in self.config.reward_weighted_responses:
            for temperature in self.config.reward_weighted_temperatures:
                recommenders[f"reward-weighted, {response}, temperature={temperature}"] = self._reward_weighted_policy(response, temperature)
        for beta in self.config.preference_betas:
            recommenders[f"within-user DPO, beta={beta}"] = self._logged_preference_policy(beta)

        lists: dict[tuple[str, str], ListEvaluation] = {}
        for name, policy in recommenders.items():
            evaluation = self._test_evaluation(name, policy)
            lists[(name, "ranked")] = self._evaluate_lists(evaluation.top_k)
            lists[(name, "generated")] = evaluation.generated_lists()

        learned_from_the_log = [name for name in recommenders if name not in ("pretrained", CONTINUED_TRAINING)]
        compared = [(CONTINUED_TRAINING, "pretrained")] + [(name, reference) for reference in ("pretrained", CONTINUED_TRAINING) for name in learned_from_the_log]
        examples, evaluation_config = self.examples("test"), self.config.evaluation_config()
        significance = {
            (f"{name} vs {reference}", kind): paired_comparison(lists[(reference, kind)], lists[(name, kind)], examples, self.log.items, evaluation_config)
            for name, reference in compared
            for kind in LIST_KINDS
        }
        self.reports.table("continued_training_significance", pd.concat(significance, names=["comparison", "lists", "quantity"]))
        table = _runs_table({f"{name}, {kind}": evaluation for (name, kind), evaluation in lists.items()}).T
        return self.reports.table("continued_training_control", table)

    @property
    def _capacity_runs(self) -> dict[str, RLConfig]:
        """Runs of the main reward held at each capacity KL budget, with fine-tuning restricted to item offsets or to low-rank updates."""
        reward, weight = self.config.rl.reward, self.config.rl.reference_kl_weight
        budgets = sorted(self.config.capacity_kl_budgets)
        runs = {
            f"item offsets, kl_budget={budget}": self.config.sweep_run(
                reward, weight, target_kl=budget, adaptation=Adaptation.ITEM_OFFSETS, learning_rate=self.config.item_offset_learning_rate
            )
            for budget in budgets
        }
        for rank in self.config.low_ranks:
            runs |= {
                f"rank-{rank} updates, kl_budget={budget}": self.config.sweep_run(
                    reward, weight, target_kl=budget, adaptation=Adaptation.LOW_RANK, low_rank=rank, learning_rate=self.config.low_rank_learning_rate
                )
                for budget in budgets
            }
        return runs

    @property
    def _group_relative_runs(self) -> dict[str, RLConfig]:
        """Runs of the main reward with GRPO's advantages: as many lists per iteration as the other runs, in groups per history."""
        rollouts = self.config.leave_one_out_rollouts
        group_relative = {
            "advantage_estimator": AdvantageEstimator.GROUP_RELATIVE,
            "rollouts_per_history": rollouts,
            "histories_per_step": self.config.rl.histories_per_step // rollouts,
            "normalize_advantages": False,
        }
        reward, weight = self.config.rl.reward, self.config.rl.reference_kl_weight
        return {f"group-relative, kl_budget={budget}": self.config.sweep_run(reward, weight, target_kl=budget, **group_relative) for budget in sorted(self.config.group_relative_kl_budgets)}

    def _finetuned_policies(self) -> dict[str, NextItemPolicy]:
        """The main run and every trained run of the main reward's sweeps: anchor weights, KL budgets, restricted capacity, GRPO."""
        full_capacity = [f"kl_budget={budget}" for budget in sorted(self.config.capacity_kl_budgets)]
        names = dict.fromkeys([*self._kl_sweep_runs, *self._kl_budget_runs, *full_capacity, *self._capacity_runs, *self._group_relative_runs])
        return {"RL": self.rl_policy} | {name: load_policy(self._sweep_checkpoint(name), self.device) for name in names if self._sweep_checkpoint(name).exists()}

    def _continuation_value_model(self, reward_name: str, reward: RewardSpec, discount: float) -> ContinuationValueModel:
        """U for `reward` at `discount`, learned from the base recommender's own lists on first use."""
        path = self.checkpoints / "continuation_values" / f"{_file_name(reward_name)}_discount_{discount}.pt"
        if not path.exists():
            training = replace(
                self.config.continuation_training,
                history_len=self.config.history_len,
                max_future=self.config.max_future,
                min_history=self.config.rl.min_history,
                generation=replace(self.config.rl.generation, horizon=self.config.horizon),
                discount=discount,
                mixed_precision=self.config.rl.mixed_precision,
            )
            value_model, records = fit_continuation_values(self.pretrained_policy, self.reward_models["train"], self.log, reward, training, self.device)
            save_continuation_value_model(value_model, path)
            self.reports.table(f"continuation_training_{_file_name(reward_name)}_discount_{discount}", pd.DataFrame(records).set_index("step"))
        return load_continuation_value_model(path, self.device)

    def _guided_lists(self, reward_function: RewardFunction, value_model: ContinuationValueModel | None, guidance: GuidanceConfig) -> torch.Tensor:
        """One guided list per test user, steered by the "train" reward model."""
        generation = self.config.evaluation_config().generation

        def guided(history):
            with autocast(self.device, self.config.rl.mixed_precision):
                return guided_lists(self.pretrained_policy, self.reward_models["train"], reward_function, value_model, history, guidance, generation)

        return collect_lists(guided, self.examples("test"), self.config.guidance_batch_size, self.device)

    def _reward_weighted_policy(self, response: str, temperature: float) -> NextItemPolicy:
        path = self.checkpoints / f"policy_reward_weighted_{response}_{temperature}.pt"
        if not path.exists():
            config = RewardWeightedConfig(self.config.preference_finetuning, response, temperature)
            policy, history = finetune_with_reward_weights(self.pretrained_policy, self.log, config, self.config.window_stride, self.device)
            save_model(policy, path)
            self.reports.table(f"reward_weighted_training_{response}_{temperature}", pd.DataFrame(history).set_index("epoch"))
        return load_policy(path, self.device)

    def _continued_training_policy(self) -> NextItemPolicy:
        """The base recommender trained for one more epoch exactly as reward-weighted fine-tuning trains it, with every logged item weighing one."""
        path = self.checkpoints / "policy_continued_training.pt"
        if not path.exists():
            config = RewardWeightedConfig(self.config.preference_finetuning, response=None)
            policy, history = finetune_with_reward_weights(self.pretrained_policy, self.log, config, self.config.window_stride, self.device)
            save_model(policy, path)
            self.reports.table("continued_next_item_training", pd.DataFrame(history).set_index("epoch"))
        return load_policy(path, self.device)

    def _logged_preference_policy(self, beta: float) -> NextItemPolicy:
        path = self.checkpoints / f"policy_within_user_dpo_{beta}.pt"
        if not path.exists():
            config = LoggedPreferenceConfig(self.config.preference_finetuning, beta, self.config.preference_margin, self.config.preference_lookahead)
            policy, history = finetune_on_logged_preferences(self.pretrained_policy, self.log, config, self.config.window_stride, self.device)
            save_model(policy, path)
            self.reports.table(f"within_user_dpo_training_{beta}", pd.DataFrame(history).set_index("epoch"))
        return load_policy(path, self.device)

    def _multi_token_heads(self) -> MultiTokenHeads:
        path = self.checkpoints / "multi_token_heads.pt"
        if not path.exists():
            heads, history = train_multi_token_heads(
                self.pretrained_policy,
                self.log,
                self.config.horizon,
                self.config.multi_token_optimization,
                self.config.window_stride,
                self.device,
                max_targets_per_head=self.config.multi_token_targets_per_head,
            )
            save_multi_token_heads(heads, path)
            self.reports.table("multi_token_training", pd.DataFrame(history).set_index("epoch"))
        return load_multi_token_heads(path, self.device)

    @cached_property
    def log(self) -> InteractionLog:
        return InteractionLog.load(self.log_directory)

    @cached_property
    def pretrained_policy(self) -> NextItemPolicy:
        return load_policy(self._pretrained_policy_path, self.device)

    @cached_property
    def rl_policy(self) -> NextItemPolicy:
        return load_policy(self._rl_policy_path, self.device)

    @cached_property
    def reward_models(self) -> dict[str, RewardModel]:
        return {name: load_reward_model(self._reward_model_path(name), self.device) for name in ("train", "heldout")}

    @cached_property
    def masked_reward_models(self) -> dict[str, RewardModel]:
        """The two reward models that can read "unknown response" (see `compare_feedback_masking`), trained on first use."""
        training = replace(self._reward_model_training(), hidden_response_share=self.config.hidden_response_share, max_hidden_responses=self.config.horizon)
        self._train_reward_model_pair("masked_", training)
        return {name: load_reward_model(self._reward_model_path(name), self.device) for name in ("masked_train", "masked_heldout")}

    def _reward_model_training(self) -> RewardModelTrainingConfig:
        return RewardModelTrainingConfig(self.config.transformer, self.config.reward_model_optimization, self.config.window_stride, loss_weights=self.config.reward_model_loss)

    def _train_reward_model_pair(self, prefix: str, training: RewardModelTrainingConfig) -> None:
        """Trains the reward models named <prefix>train and <prefix>heldout on disjoint halves of the users, unless they exist."""
        all_users = np.arange(self.log.num_users)
        for name, users in {f"{prefix}train": all_users[all_users % 2 == 0], f"{prefix}heldout": all_users[all_users % 2 == 1]}.items():
            if not self._reward_model_path(name).exists():
                reward_model, history = train_reward_model(self.log, training, self.device, users=users)
                save_model(reward_model, self._reward_model_path(name))
                self.reports.table(f"reward_model_{name}_training", pd.DataFrame(history).set_index("epoch"))

    def examples(self, period: Period) -> EvalExamples:
        """One recommendation request per user at the start of `period`."""
        if period not in self._examples:
            self._examples[period] = build_eval_examples(self.log, period, self.config.history_len, self.config.max_future)
        return self._examples[period]

    @property
    def _pretrained_policy_path(self) -> Path:
        return self.checkpoints / "policy_pretrained.pt"

    @property
    def _rl_policy_path(self) -> Path:
        return self.checkpoints / "policy_rl.pt"

    def _reward_model_path(self, name: str) -> Path:
        return self.checkpoints / f"reward_model_{name}.pt"

    def _evaluate(self, policy: NextItemPolicy, period: Period, reward_models: dict[str, RewardModel] | None = None, **evaluation_overrides) -> PolicyEvaluation:
        reward_models = self.reward_models if reward_models is None else reward_models
        config = self.config.evaluation_config(**evaluation_overrides)
        evaluation = evaluate_policy(policy, self.examples(period), self.log.items, reward_models, config, self.device)
        if self._pretrained_policy_path.exists():
            evaluation.metrics[f"top_k/{INTRA_LIST_DISTANCE}"] = self._mean_intra_list_distance(evaluation.top_k)
            evaluation.metrics[f"next_k/{INTRA_LIST_DISTANCE}"] = self._mean_intra_list_distance(evaluation.generated.actions)
        return evaluation

    def _evaluate_lists(self, lists: torch.Tensor) -> ListEvaluation:
        """Test-period evaluation of one given list per test user."""
        evaluation = evaluate_lists(lists, self.examples("test"), self.log.items, self.reward_models, self.config.evaluation_config(), self.device)
        evaluation.metrics[INTRA_LIST_DISTANCE] = self._mean_intra_list_distance(evaluation.lists)
        return evaluation

    @cached_property
    def _unit_item_embeddings(self) -> torch.Tensor:
        """The base recommender's item embeddings scaled to unit length, on the CPU: the space in which lists are called varied. [V+1, d]"""
        return F.normalize(self.pretrained_policy.item_embedding.weight.detach().float().cpu(), dim=-1)

    def _mean_intra_list_distance(self, lists: torch.Tensor) -> float:
        return intra_list_distance(lists.cpu(), self._unit_item_embeddings).mean().item()

    def _evaluate_ranking(self, ranker: NextItemScorer) -> ListEvaluation:
        """Test-period evaluation of the lists a model produces by ranking the catalogue once, as long as the generated lists."""
        config = self.config.evaluation_config()

        def top_of_ranking(history):
            return recommend_top_k(ranker, history, self.config.horizon, config.generation)

        return self._evaluate_lists(collect_lists(top_of_ranking, self.examples("test"), config.batch_size, self.device))

    def _liked_only_policy(self) -> NextItemPolicy:
        path = self.checkpoints / "policy_liked_only.pt"
        if not path.exists():
            liked_threshold = self.config.evaluation_config().liked_threshold
            policy, history = finetune_on_liked_items(
                self.pretrained_policy, self.log, self.config.liked_finetuning, self.config.window_stride, liked_threshold, self.device
            )
            save_model(policy, path)
            self.reports.table("liked_only_training", pd.DataFrame(history).set_index("epoch"))
        return load_policy(path, self.device)

    def _logged_return_policy(self, importance_weight_cap: float) -> NextItemPolicy:
        """The base recommender fine-tuned with off-policy REINFORCE on logged sessions, for the main run's reward."""
        path = self.checkpoints / f"policy_off_policy_cap_{importance_weight_cap}.pt"
        if not path.exists():
            config = replace(self.config.logged_returns, reward=self.config.rl.reward, importance_weight_cap=importance_weight_cap)
            policy, history = finetune_on_logged_returns(self.pretrained_policy, self.log, config, self.config.window_stride, self.device)
            save_model(policy, path)
            self.reports.table(f"off_policy_training_cap_{importance_weight_cap}", pd.DataFrame(history).set_index("epoch"))
        return load_policy(path, self.device)

    def _best_of_sampled_lists(self, num_lists: int, reward: RewardSpec | None = None) -> torch.Tensor:
        """For every test user, the best of `num_lists` lists sampled from the base recommender, as the "train" reward model sees it.

        Lists are compared by `reward`, the main run's unless another is given.
        """
        config = self.config.evaluation_config()
        sampling = replace(config.generation, sample_from_top=self.config.sample_from_top)
        reward_function = RewardFunction(reward or self.config.rl.reward, self.log.items, self.device, item_embeddings=self.pretrained_policy.item_embedding.weight)

        def best_list(history):
            return best_of_sampled_lists(
                self.pretrained_policy, self.reward_models["train"], reward_function, history, num_lists, self.config.rl.discount, sampling, config.simulated_feedback
            )

        return collect_lists(best_list, self.examples("test"), config.batch_size, self.device)

    @cached_property
    def _scored_test_candidates(self) -> ScoredCandidates:
        """The base recommender's most likely next items for every test user, each judged by the "train" reward model. On the CPU."""
        config = self.config.evaluation_config()
        scored = [
            score_candidates(self.pretrained_policy, self.reward_models["train"], batch.history.to(self.device), self.config.rerank_candidates, config.generation).cpu()
            for batch in self.examples("test").batches(config.batch_size)
        ]
        return ScoredCandidates.concatenate(scored)

    def _retention_quality(self, test_predictions: dict[str, HeldoutFeedback]) -> pd.DataFrame:
        """How well both reward models predict the retention signals, on held-out events whose outcome the log knows.

        A return window as long as the test period leaves almost no test event with a known outcome. The
        validation period then carries the evidence, although it also picked the best training epoch.
        """
        validation_windows = EventWindows.for_heldout(self.log, "validation", self.config.transformer.max_positions)
        quality = {}
        for name, reward_model in self.reward_models.items():
            validation_predictions = predict_heldout_feedback(reward_model, validation_windows, self.config.evaluation_batch_size, self.device)
            quality[(name, "validation")] = retention_quality(validation_predictions)
            quality[(name, "test")] = retention_quality(test_predictions[name])
        return pd.concat(quality, names=["reward model", "period", "signal"])

    @property
    def _pretrained_lists(self) -> ListEvaluation:
        return self._test_evaluation("pretrained", self.pretrained_policy).generated_lists()

    @property
    def _rl_lists(self) -> ListEvaluation:
        return self._test_evaluation("rl", self.rl_policy).generated_lists()

    def _sweep_evaluation(self, run_name: str) -> PolicyEvaluation:
        """Test evaluation of a trained sweep run, together with how far the run ended from the base recommender."""
        if run_name not in self._test_evaluations:
            self._test_evaluations[run_name] = self._evaluate(load_policy(self._sweep_checkpoint(run_name), self.device), "test")
        evaluation = self._test_evaluations[run_name]
        if REALISED_KL not in evaluation.metrics:
            evaluation.metrics |= self._end_of_training(f"sweep_training_{_file_name(run_name)}")
        return evaluation

    def _end_of_training(self, training_report: str) -> dict[str, float]:
        """How a recorded fine-tuning run ended: each of `TRAINING_STATISTICS` averaged over its last iterations.

        The first is the KL from the fine-tuned policy to the base recommender, in nats per generated item.
        """
        records = pd.read_csv(self.reports.directory / f"{training_report}.csv")
        return {
            statistic: float(records[column].dropna().tail(END_OF_TRAINING_STEPS).mean()) if column in records else float("nan")
            for statistic, column in TRAINING_STATISTICS.items()
        }

    def _metrics_on_validation(self, run_name: str) -> dict[str, float]:
        """All metrics of the base recommender ("pretrained") or of a trained sweep run on validation requests, computed once."""
        if run_name not in self._validation_metrics:
            policy = self.pretrained_policy if run_name == "pretrained" else load_policy(self._sweep_checkpoint(run_name), self.device)
            self._validation_metrics[run_name] = self._evaluate(policy, "validation").metrics
        return self._validation_metrics[run_name]

    def _selection_on_validation(self, run_names: list[str]) -> pd.DataFrame:
        """Which of the runs would be chosen without looking at the test period. One row per run evaluated on validation requests.

        The chosen run has the highest reward on validation requests, as the held-out reward model predicts it,
        among the runs that keep the validation NDCG within `relevance_tolerance` of the base recommender's.
        """
        k = max(self.config.ks)
        relevance, reward = f"next_k/all/ndcg@{k}", "predicted/heldout/played_ratio"
        evaluated = [name for name in run_names if name in self._validation_metrics]
        selection = pd.DataFrame({
            f"validation/{relevance}": {name: self._validation_metrics[name][relevance] for name in evaluated},
            f"validation/{reward}": {name: self._validation_metrics[name][reward] for name in evaluated},
        })
        lowest_relevance = (1.0 - self.config.relevance_tolerance) * selection.loc["pretrained", f"validation/{relevance}"]
        selection["keeps_relevance"] = selection[f"validation/{relevance}"] >= lowest_relevance
        best_admissible = selection[f"validation/{reward}"].where(selection["keeps_relevance"]).idxmax()
        selection["selected"] = selection.index == best_admissible
        return selection

    def _generated_lists_of(self, sweep_runs) -> dict[str, ListEvaluation]:
        """Test evaluation of the lists generated by each named sweep run that was trained, in the given order."""
        return {name: self._sweep_evaluation(name).generated_lists() for name in sweep_runs if self._sweep_checkpoint(name).exists()}

    def _reranked(self, label: str, candidates: ScoredCandidates, reward: RewardSpec) -> dict[str, ListEvaluation]:
        """Test evaluation of the candidates re-ranked by `reward` at every relevance weight, computed once per label."""
        reward_function = RewardFunction(reward, self.log.items, "cpu", item_embeddings=self.pretrained_policy.item_embedding.weight)
        evaluations = {}
        for weight in self.config.rerank_relevance_weights:
            name = f"{label}, weight={weight}"
            if name not in self._reranked_evaluations:
                self._reranked_evaluations[name] = self._evaluate_lists(rerank(candidates, reward_function, self.config.horizon, weight))
            evaluations[name] = self._reranked_evaluations[name]
        return evaluations

    @cached_property
    def _candidates_scored_by_item_mean(self) -> ScoredCandidates:
        """The same candidates with the reward model's prediction replaced by each item's mean played ratio in the train period."""
        candidates = self._scored_test_candidates
        item_mean = torch.as_tensor(self.log.items.mean_played_ratio)[candidates.items]
        feedback = Feedback(
            played_ratio=item_mean,
            session_continues=torch.zeros_like(item_mean),
            returns=torch.zeros_like(item_mean),
            is_chosen=torch.ones_like(item_mean),
            relative_ratio=torch.full_like(item_mean, 0.5),
            liked=torch.zeros_like(item_mean),
            disliked=torch.zeros_like(item_mean),
            adopts=torch.zeros_like(item_mean),
        )
        return ScoredCandidates(candidates.items, candidates.log_probs, feedback, feedback)

    def _trade_off_figures(
        self, report_name: str, curves: dict[str, dict[str, ListEvaluation]], signals: tuple[str, ...], measures: list[str] | None = None
    ) -> None:
        """One figure per signal: `measures` against that signal as the held-out reward model predicts it.

        curves: method -> its runs in the order its line is drawn. Methods without runs are left out.
        measures: metrics of the lists, by default relevance on all and on liked items and catalogue coverage.
        """
        k = max(self.config.ks)
        measures = measures or [f"all/ndcg@{k}", f"liked/ndcg@{k}", "profile/catalog_coverage"]
        tables = {method: _runs_table(runs) for method, runs in curves.items() if runs}
        for signal in signals:
            figure = plots.trade_off(tables, reward_column=f"predicted/heldout/{signal}", measure_columns=measures)
            self.reports.figure(f"{report_name}_by_{signal}", figure)

    @property
    def _kl_sweep_runs(self) -> dict[str, RLConfig]:
        return {f"kl={weight}": self.config.rl_config(num_steps=self.config.sweep_steps, reference_kl_weight=weight) for weight in self.config.kl_weights}

    @property
    def _kl_budget_runs(self) -> dict[str, RLConfig]:
        """Runs of the main reward whose anchor weight starts at the main run's and adapts to hold each KL budget."""
        reward, starting_weight = self.config.rl.reward, self.config.rl.reference_kl_weight
        return {f"kl_budget={budget}": self.config.sweep_run(reward, starting_weight, target_kl=budget) for budget in sorted(self.config.kl_budgets)}

    def _record_stage_time(self, stage: str, seconds: float) -> None:
        self._stage_seconds[stage] = self._stage_seconds.get(stage, 0.0) + seconds
        self.reports.table("stage_minutes", pd.Series({name: total / 60 for name, total in self._stage_seconds.items()}, name="minutes"))

    def _test_evaluation(self, name: str, policy: NextItemPolicy) -> PolicyEvaluation:
        """Test-period evaluation of a named recommender, computed once per experiment."""
        if name not in self._test_evaluations:
            self._test_evaluations[name] = self._evaluate(policy, "test")
        return self._test_evaluations[name]

    def _validation_ranking_quality(self, ranker: NextItemScorer) -> dict[str, float]:
        """NDCG of ranking the catalogue for validation users, under the name the GPT recommender reports it with.

        Losses of different models are not comparable; this number, printed after every epoch, is.
        """
        config, k, examples = self.config.evaluation_config(), max(self.config.ks), self.examples("validation")

        def top_of_ranking(history):
            return recommend_top_k(ranker, history, k, config.generation)

        lists = collect_lists(top_of_ranking, examples, config.batch_size, self.device)
        return {f"top_k/all/ndcg@{k}": ranking_metrics(lists, examples.future_items, [k])[f"ndcg@{k}"]}

    def _validation_relevance(self, policy: NextItemPolicy) -> dict[str, float]:
        k = max(self.config.ks)
        metrics = self._evaluate(policy, "validation", reward_models={}).metrics
        return {name: metrics[name] for name in (f"top_k/all/ndcg@{k}", f"next_k/all/ndcg@{k}")}

    def _monitor(self, policy: NextItemPolicy) -> dict[str, float]:
        """What is tracked on validation users while RL runs: relevance, both predicted rewards, catalogue coverage."""
        k = max(self.config.ks)
        monitored = [
            f"top_k/all/ndcg@{k}",
            f"next_k/all/ndcg@{k}",
            f"next_k/liked/ndcg@{k}",
            "predicted/train/played_ratio",
            "predicted/heldout/played_ratio",
            "predicted/heldout/personal_played_ratio",
            "predicted/heldout/relative_ratio",
            "predicted/heldout/returns",
            "next_k/logged/played_ratio_per_item",
            "next_k/profile/catalog_coverage",
        ]
        metrics = self._evaluate(policy, "validation").metrics
        return {name: metrics[name] for name in monitored}

    @property
    def _summary_rows(self) -> list[str]:
        k = max(self.config.ks)
        return [
            f"top_k/all/ndcg@{k}",
            f"next_k/all/ndcg@{k}",
            f"next_k/liked/ndcg@{k}",
            f"next_k/skipped/recall@{k}",
            f"next_k/above_usual/ndcg@{k}",
            f"next_k/below_usual/recall@{k}",
            f"next_k/new/ndcg@{k}",
            f"next_k/repeat/ndcg@{k}",
            *TRAINING_STATISTICS,
            "next_k/logged/hits_per_list",
            "next_k/logged/played_ratio_of_hits",
            "next_k/logged/relative_ratio_of_hits",
            "next_k/logged/played_ratio_per_item",
            "next_k/logged/relative_surplus_per_item",
            "predicted/train/played_ratio",
            "predicted/heldout/played_ratio",
            "predicted/heldout/played_ratio_for_other_user",
            "predicted/heldout/personal_played_ratio",
            "predicted/heldout/relative_ratio",
            "predicted/heldout/chosen_played_ratio",
            "predicted/heldout/is_chosen",
            "predicted/heldout/played_seconds",
            "predicted/heldout/session_continues",
            "predicted/heldout/returns",
            "predicted/heldout/returns_after_list",
            "predicted/heldout/liked",
            "predicted/heldout/disliked",
            "predicted/heldout/adopts",
            "next_k/logged/liked_per_item",
            "next_k/logged/adopted_per_item",
            "next_k/profile/catalog_coverage",
            "next_k/profile/top_exposure_share",
            "next_k/profile/mean_user_pair_overlap",
            f"next_k/{INTRA_LIST_DISTANCE}",
            "next_k/profile/head_item_share",
            "next_k/profile/mean_duration_seconds",
            "next_k/profile/history_repeat_rate",
        ]

    def _sweep(self, report_name: str, runs: dict[str, RLConfig], reward_model: RewardModel | None = None) -> pd.DataFrame:
        """Fine-tunes one policy per run and tabulates all test metrics, one column per run plus the base recommender.

        The policies learn from `reward_model`, by default the training reward model. Runs that no longer fit the
        time budget are left out of the table.
        """
        evaluations = {"pretrained": self._test_evaluation("pretrained", self.pretrained_policy)}
        evaluations["pretrained"].metrics.setdefault(REALISED_KL, 0.0)
        for run_name, rl_config in runs.items():
            if self._is_finetuned(run_name, rl_config, reward_model):
                evaluations[run_name] = self._sweep_evaluation(run_name)
        if self.config.evaluate_sweeps_on_validation:
            on_validation = pd.DataFrame({run_name: self._metrics_on_validation(run_name) for run_name in evaluations})
            self.reports.table(f"{report_name}_validation", on_validation)
        return self.reports.table(report_name, metrics_table(evaluations))

    def _is_finetuned(self, run_name: str, rl_config: RLConfig, reward_model: RewardModel | None = None) -> bool:
        """Trains the policy of a sweep run unless its checkpoint exists. False if there is no time left to train it."""
        path = self._sweep_checkpoint(run_name)
        if not path.exists():
            if not self._fits_time_budget(rl_config):
                print(f"skipped the run '{run_name}': it would not finish inside the time budget")
                return False
            policy, records = self._run_rl(rl_config, reward_model=reward_model)
            save_model(policy, path)
            self.reports.table(f"sweep_training_{_file_name(run_name)}", pd.DataFrame(records).set_index("step"))
        return True

    def _sweep_checkpoint(self, run_name: str) -> Path:
        return self.checkpoints / "sweep" / f"{_file_name(run_name)}.pt"

    def _run_rl(self, rl_config: RLConfig, evaluate=None, reward_model: RewardModel | None = None) -> tuple[NextItemPolicy, list[dict[str, float]]]:
        """One RL fine-tuning of the base recommender, timed so that later runs can be checked against the time budget."""
        started = time.monotonic()
        reward_model = self.reward_models["train"] if reward_model is None else reward_model
        policy, records = finetune_with_rl(self.pretrained_policy, reward_model, self.log, rl_config, self.device, evaluate=evaluate)
        self._seconds_per_rl_step = (time.monotonic() - started) / rl_config.num_steps
        return policy, records

    def _fits_time_budget(self, rl_config: RLConfig) -> bool:
        """Whether an RL run of this length would end before the deadline, judging by the speed of the last run."""
        if self._deadline is None or self._seconds_per_rl_step is None:
            return True
        expected_seconds = rl_config.num_steps * self._seconds_per_rl_step * RUN_TIME_SAFETY_FACTOR
        return time.monotonic() + expected_seconds + RESERVE_FOR_LAST_STAGES_SECONDS < self._deadline


def _file_name(name: str) -> str:
    """`name` with everything that is awkward in a file name replaced by underscores."""
    return re.sub(r"[^0-9a-zA-Z.]+", "_", name)


def _runs_table(runs: dict[str, ListEvaluation]) -> pd.DataFrame:
    """One row per run, one column per metric."""
    return pd.DataFrame({name: evaluation.metrics for name, evaluation in runs.items()}).T


def _bind_directory_to_config(home: Path, config: ExperimentConfig) -> None:
    """Records the configuration in `home`, and refuses a directory that holds the results of a different one."""
    home.mkdir(parents=True, exist_ok=True)
    record = home / CONFIG_RECORD
    serialized = json.dumps(asdict(config), indent=2, sort_keys=True)
    if record.exists() and json.loads(record.read_text()) != json.loads(serialized):
        raise ValueError(f"{home} holds the results of a different configuration: use another home directory, or delete this one")
    record.write_text(serialized)


def _events_of_first_users(raw_events: pl.DataFrame, num_users: int) -> pl.DataFrame:
    first_users = raw_events.select(pl.col("user_id").unique().sort().head(num_users))
    return raw_events.join(first_users, on="user_id", how="semi")


def _log_summary(log: InteractionLog) -> pd.Series:
    def num_events_in(period: Period) -> int:
        starts, ends = log.period_rows(period)
        return int((ends - starts).sum())

    return pd.Series({
        "users": log.num_users,
        "items": log.num_items,
        "events": log.num_events,
        "train events": num_events_in("train"),
        "validation events": num_events_in("validation"),
        "test events": num_events_in("test"),
        "median events per user": float(np.median(np.diff(log.user_offsets))),
        "mean played ratio": float(np.mean(log.played_ratio)),
        "share of events played at least half": float(np.mean(np.asarray(log.played_ratio) >= 0.5)),
        "share of events that continue the session": _rate_where_known(log.session_continues, log.session_continues_known),
        "share of events after which the user returns": _rate_where_known(log.returns, log.returns_known),
        "share of events whose return outcome is known": float(np.mean(log.returns_known)),
        "share of organic events": float(np.mean(log.is_organic)),
        "share of events the user liked": _rate_where_known(log.liked, log.reactions_known),
        "share of events the user disliked": _rate_where_known(log.disliked, log.reactions_known),
        "share of first events with an item that the user came back to": _rate_where_known(log.adopts, log.adopts_known),
    })


def _rate_where_known(signal: np.ndarray, is_known: np.ndarray) -> float:
    """How often a signal is true among the events whose outcome the log knows; NaN where it knows none."""
    is_known = np.asarray(is_known)
    return float(np.asarray(signal)[is_known].mean()) if is_known.any() else float("nan")


def _decile(values_per_item: np.ndarray) -> np.ndarray:
    """Decile (0 = lowest) of every item token by the given attribute."""
    return pd.qcut(pd.Series(values_per_item).rank(method="first"), 10, labels=False).to_numpy()


def _exposure_change(log: InteractionLog, before: PolicyEvaluation, after: PolicyEvaluation) -> pd.DataFrame:
    """Per item, the number of test users whose generated list contains it under each recommender, biggest winners first.

    Next to the item's popularity, duration and average played ratio, this shows what the fine-tuned recommender learned to prefer.
    """
    def exposure(evaluation: PolicyEvaluation) -> np.ndarray:
        return np.bincount(evaluation.generated.actions.flatten().numpy(), minlength=log.num_items + 1)

    table = pd.DataFrame({
        "popularity": log.items.train_popularity,
        "duration_seconds": log.items.duration_seconds,
        "mean_played_ratio": log.items.mean_played_ratio,
        "exposure_before": exposure(before),
        "exposure_after": exposure(after),
    }).iloc[1:].rename_axis("item_token")
    table["exposure_change"] = table.exposure_after - table.exposure_before
    return table.sort_values("exposure_change", ascending=False)

In [ ]:
%%writefile /kaggle/working/src/gptrec/labels.py
"""Names of the reported quantities in words, for figure titles and for the tables of the write-up.

The reports name every quantity by a path such as "next_k/liked/ndcg@10" or "predicted/heldout/played_ratio".
This module has no dependencies, so the scripts that typeset the reports can use it without the training stack.
"""

from __future__ import annotations

LIST_KINDS = {"top_k": "ranked lists", "next_k": "generated lists"}
TARGET_SETS = {
    "all": "all next items",
    "liked": "liked items",
    "skipped": "skipped items",
    "above_usual": "items consumed above usual",
    "below_usual": "items consumed below usual",
    "repeat": "items from the history",
    "new": "items not in the history",
}
RANKING_METRICS = {"ndcg": "NDCG", "recall": "Recall", "map": "MAP", "hit_rate": "HitRate"}
REWARD_MODELS = {"train": "training reward model", "heldout": "held-out reward model"}
SIGNALS = {
    "played_ratio": "played ratio",
    "played_ratio_for_other_user": "played ratio if shown to another user",
    "personal_played_ratio": "personal part of the played ratio",
    "relative_ratio": "played ratio relative to the item's usual",
    "personal_relative_ratio": "personal part of the relative played ratio",
    "session_continues": "probability that the session continues",
    "returns": "probability that the user returns",
    "is_chosen": "probability that the user chooses the item",
    "chosen_played_ratio": "played ratio weighted by choice",
    "played_seconds": "seconds played",
}
NAMED = {
    "num_users": "users",
    "profile/catalog_coverage": "catalogue coverage",
    "profile/item_gini": "Gini coefficient of item exposure",
    "profile/top_item_user_share": "share of users who get the most frequent item",
    "profile/mean_user_pair_overlap": "overlap between the lists of two users",
    "profile/intra_list_distance": "mean distance between two items of a list",
    "profile/duplicate_rate": "share of repeated items in a list",
    "profile/history_repeat_rate": "share of recommendations from the history",
    "profile/mean_popularity": "mean popularity of recommended items",
    "profile/head_item_share": "share of recommendations from the most popular 1% of items",
    "profile/mean_duration_seconds": "mean duration of recommended items, seconds",
    "profile/mean_item_played_ratio": "mean usual played ratio of recommended items",
    "logged/hits_per_list": "recommended items the user went on to consume, per list",
    "logged/played_ratio_of_hits": "logged played ratio of the items the user went on to consume",
    "logged/liked_share_of_hits": "share of those items the user liked",
    "logged/relative_ratio_of_hits": "logged relative played ratio of those items",
    "logged/played_ratio_per_item": "logged played ratio per recommended item",
    "logged/relative_surplus_per_item": "logged played ratio above the item's usual, per recommended item",
    "logged/served_hits_per_list": "recommended items a recommender later served, per list",
    "logged/played_ratio_of_served_hits": "logged played ratio of the items a recommender later served",
    "training/kl_to_reference": "KL to the base recommender, nats per item",
    "training/anchor_weight": "anchor weight at the end of fine-tuning",
    "training/clip_fraction": "share of actions beyond the clipping threshold",
    "training/value_explained_variance": "variance of returns explained by the value model",
    "policy_entropy": "entropy of the policy, nats",
    "in_true_continuation": "probability that the item is among the user's next items",
    "in_history": "probability that the item is in the history",
    "popularity": "popularity of the item",
    "duration_seconds": "duration of the item, seconds",
    "played_ratio_disagreement/train_vs_heldout": "disagreement of the two reward models on the played ratio",
    "reward": "reward per item",
    "kl_to_reference": "KL to the base recommender, nats per item",
    "reference_kl_weight": "anchor weight",
    "entropy": "entropy of the policy, nats",
    "distinct_action_share": "share of distinct items among those sampled",
    "logged_hit_rate": "share of sampled items among the user's next items",
    "history_repeat_rate": "share of sampled items from the history",
    "mean_action_popularity": "mean popularity of sampled items",
    "mean_action_duration_seconds": "mean duration of sampled items, seconds",
    "gradient_norm": "gradient norm",
    "clip_fraction": "share of actions beyond the clipping threshold",
    "value_explained_variance": "variance of returns explained by the value model",
    "train_loss": "training loss",
    "validation_loss": "validation loss",
}


def readable(metric: str) -> str:
    """The reported name of a quantity in words: "next_k/liked/ndcg@10" -> "NDCG@10, liked items (generated lists)".

    A name this module does not know is returned with its underscores turned into spaces.
    """
    qualifiers = []
    if metric.startswith("validation/"):
        metric, qualifiers = metric.removeprefix("validation/"), ["validation requests"]
    list_kind, _, quantity = metric.partition("/")
    if list_kind in LIST_KINDS:
        metric, qualifiers = quantity, [LIST_KINDS[list_kind], *qualifiers]
    label = _quantity(metric)
    return f"{label} ({', '.join(qualifiers)})" if qualifiers else label


def _quantity(metric: str) -> str:
    if metric in NAMED:
        return NAMED[metric]
    parts = metric.split("/")
    if len(parts) == 3 and parts[0] == "predicted":
        return _predicted(signal=parts[2], reward_model=parts[1])
    if len(parts) == 2 and parts[0].startswith("predicted_"):
        return _predicted(signal=parts[0].removeprefix("predicted_"), reward_model=parts[1])
    if len(parts) == 2 and parts[0] in TARGET_SETS and "@" in parts[1]:
        name, _, cutoff = parts[1].partition("@")
        if name in RANKING_METRICS:
            return f"{RANKING_METRICS[name]}@{cutoff}, {TARGET_SETS[parts[0]]}"
    return metric.replace("_", " ")


def _predicted(signal: str, reward_model: str) -> str:
    return f"predicted {SIGNALS.get(signal, signal.replace('_', ' '))}, {REWARD_MODELS.get(reward_model, reward_model)}"

In [ ]:
%%writefile /kaggle/working/src/gptrec/models/adapters.py
"""What fine-tuning may change in the recommender, and operating points reached in weight space without training.

Fine-tuning can be held to one score offset per item, the same for every history, or to rank-r updates of the
transformer's projections (LoRA, Hu et al. 2022), which change how a history is read but no item's embedding.
Moving the weights along the line from the base recommender to a fine-tuned one, or past it, gives operating
points without further training (WiSE-FT, Wortsman et al. 2022; WARP, Ramé et al. 2024; ExPO, Zheng et al. 2024).
"""

from __future__ import annotations

import copy
import math
from dataclasses import replace
from enum import Enum

import torch
import torch.nn.functional as F
from torch import Tensor, nn

from gptrec.models.policy import NextItemPolicy


class Adaptation(str, Enum):
    """What fine-tuning may change in the recommender."""

    FULL = "full"
    ITEM_OFFSETS = "item_offsets"
    LOW_RANK = "low_rank"


class LowRankUpdate(nn.Module):
    """A frozen projection plus a trainable update of rank r. The update starts at zero, so the module starts as the projection."""

    def __init__(self, projection: nn.Module, rank: int) -> None:
        super().__init__()
        in_features, out_features = projection_shape(projection)
        self.projection = projection.requires_grad_(False)
        self.down = nn.Parameter(torch.randn(rank, in_features) / math.sqrt(in_features))
        self.up = nn.Parameter(torch.zeros(out_features, rank))

    def forward(self, inputs: Tensor) -> Tensor:
        return self.projection(inputs) + F.linear(F.linear(inputs, self.down), self.up)

    @torch.no_grad()
    def merged(self) -> nn.Module:
        """The projection with the update folded into its weight."""
        update = (self.up @ self.down).to(self.projection.weight.dtype)
        self.projection.weight += update if isinstance(self.projection, nn.Linear) else update.T
        return self.projection


def projection_shape(projection: nn.Module) -> tuple[int, int]:
    """(input features, output features) of a linear layer, or of GPT-2's Conv1D, which stores its weight as [in, out]."""
    if isinstance(projection, nn.Linear):
        return projection.in_features, projection.out_features
    in_features, out_features = projection.weight.shape
    return in_features, out_features


def add_low_rank_updates(module: nn.Module, rank: int) -> nn.Module:
    """Freezes `module` and gives every projection inside it a trainable update of rank `rank`. Returns `module`."""
    module.requires_grad_(False)
    projections = [(parent, name, child) for parent in module.modules() for name, child in parent.named_children() if _is_projection(child)]
    for parent, name, child in projections:
        setattr(parent, name, LowRankUpdate(child, rank))
    return module


def merge_low_rank_updates(module: nn.Module) -> nn.Module:
    """Folds every low-rank update inside `module` into its projection, which restores the module's own structure. Returns `module`."""
    updates = [(parent, name, child) for parent in module.modules() for name, child in parent.named_children() if isinstance(child, LowRankUpdate)]
    for parent, name, child in updates:
        setattr(parent, name, child.merged())
    return module


def with_item_offsets(policy: NextItemPolicy, offsets: Tensor | None = None) -> NextItemPolicy:
    """A copy of `policy` that adds offsets[j] (float [V+1]) to the score of every item j: zero offsets unless given."""
    shifted = NextItemPolicy(replace(policy.config, item_offsets=True))
    missing, unexpected = shifted.load_state_dict(policy.state_dict(), strict=False)
    if unexpected or set(missing) - {"item_offsets"}:
        raise ValueError(f"the recommender's weights do not fit its configuration: missing {missing}, unexpected {unexpected}")
    if offsets is not None:
        with torch.no_grad():
            shifted.item_offsets += offsets.to(shifted.item_offsets)
    return shifted.to(policy.item_embedding.weight.device).eval()


def adaptable_copy(policy: NextItemPolicy, adaptation: Adaptation, rank: int = 1) -> NextItemPolicy:
    """A copy of `policy` in which exactly the parameters that `adaptation` allows to change require gradients."""
    adaptation = Adaptation(adaptation)
    if adaptation is Adaptation.FULL:
        return copy.deepcopy(policy).requires_grad_(True)
    if adaptation is Adaptation.ITEM_OFFSETS:
        adapted = with_item_offsets(policy).requires_grad_(False)
        adapted.item_offsets.requires_grad_(True)
        return adapted
    adapted = copy.deepcopy(policy).requires_grad_(False)
    add_low_rank_updates(adapted.backbone, rank)
    return adapted


def interpolated(base: NextItemPolicy, finetuned: NextItemPolicy, factor: float) -> NextItemPolicy:
    """The recommender with weights theta_0 + factor * (theta - theta_0): 0 gives `base`, 1 `finetuned`, above 1 extrapolates past it."""
    if base.config != finetuned.config:
        raise ValueError("only recommenders of the same configuration can be interpolated")
    base_weights = base.state_dict()
    blended = copy.deepcopy(finetuned)
    blended.load_state_dict({name: base_weights[name] + factor * (weight - base_weights[name]) for name, weight in finetuned.state_dict().items()})
    return blended.eval()


def _is_projection(module: nn.Module) -> bool:
    return isinstance(module, nn.Linear) or type(module).__name__ == "Conv1D"

In [ ]:
%%writefile /kaggle/working/src/gptrec/models/backbone.py
"""The GPT-2 decoder stack shared by the recommender and the reward model."""

from __future__ import annotations

from dataclasses import dataclass

import torch
from torch import Tensor, nn
from transformers import GPT2Config, GPT2Model

EMBEDDING_INIT_STD = 0.02


@dataclass(frozen=True)
class TransformerConfig:
    hidden_size: int = 256
    num_layers: int = 4
    num_heads: int = 4
    dropout: float = 0.1
    max_positions: int = 128


class CausalTransformer(nn.Module):
    """Hugging Face GPT-2 blocks applied to embeddings supplied by the caller.

    Attention is causal and positions count from 0, which is all a right-padded sequence needs.
    """

    def __init__(self, config: TransformerConfig) -> None:
        super().__init__()
        self.max_positions = config.max_positions
        self.gpt2 = GPT2Model(
            GPT2Config(
                vocab_size=1,
                n_positions=config.max_positions,
                n_embd=config.hidden_size,
                n_layer=config.num_layers,
                n_head=config.num_heads,
                resid_pdrop=config.dropout,
                embd_pdrop=config.dropout,
                attn_pdrop=config.dropout,
                use_cache=False,
                bos_token_id=None,
                eos_token_id=None,
            )
        )

    def forward(self, token_embeddings: Tensor) -> Tensor:
        """[B, L, hidden] -> [B, L, hidden]; output position t depends on inputs 0..t only."""
        sequence_length = token_embeddings.shape[1]
        if sequence_length > self.max_positions:
            raise ValueError(f"sequence of {sequence_length} tokens exceeds max_positions={self.max_positions}")
        return self.gpt2(inputs_embeds=token_embeddings, use_cache=False).last_hidden_state


def new_item_embedding(num_items: int, hidden_size: int, padding_item: int) -> nn.Embedding:
    """Embedding table for item tokens 0..num_items with GPT-2's initialisation and an all-zero padding row."""
    embedding = nn.Embedding(num_items + 1, hidden_size, padding_idx=padding_item)
    nn.init.normal_(embedding.weight, std=EMBEDDING_INIT_STD)
    with torch.no_grad():
        embedding.weight[padding_item].zero_()
    return embedding

In [ ]:
%%writefile /kaggle/working/src/gptrec/models/multi_token.py
"""Lists predicted from the real history alone: one head per position (multi-token prediction, Gloeckle et al. 2024).

A generated list feeds every chosen item back as if the user had consumed it, which most were not. Head k here
predicts the item k events ahead from the real history, so no position of the list rests on an unconfirmed one.
The heads sit on the frozen base recommender; head 1 is the recommender itself.
"""

from __future__ import annotations

from functools import partial
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import Tensor, nn

from gptrec.data.batches import EventBatch, HistoryBatch
from gptrec.data.log import InteractionLog
from gptrec.models.policy import NextItemPolicy
from gptrec.models.sequence_recommender import SequenceRecommender
from gptrec.rollout import GenerationConfig, constrain_logits
from gptrec.training.precision import seed_everything
from gptrec.training.supervised import Metrics, OptimizationConfig, fit_on_train_period


class MultiTokenHeads(nn.Module):
    """Heads 2..K: each maps the recommender's state to the state from which its item k events ahead is scored.

    Each starts as the identity, that is as a copy of head 1, the recommender's own next-item head.
    """

    def __init__(self, hidden_size: int, horizon: int) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.horizon = horizon
        self.transforms = nn.ModuleList(_identity(hidden_size) for _ in range(horizon - 1))

    def logits(self, recommender: SequenceRecommender, hidden: Tensor, offset: int) -> Tensor:
        """Scores of the item `offset` events ahead (1: the next one) from recommender states. [..., hidden] -> [..., V+1]"""
        return recommender.item_logits(hidden if offset == 1 else self.transforms[offset - 2](hidden))


def train_multi_token_heads(
    recommender: NextItemPolicy,
    log: InteractionLog,
    horizon: int,
    config: OptimizationConfig,
    window_stride: int,
    device: torch.device,
    max_targets_per_head: int | None = None,
) -> tuple[MultiTokenHeads, list[Metrics]]:
    """Trains heads 2..`horizon` on the train period's windows; the recommender stays as it is.

    max_targets_per_head: positions per batch that each head learns from, drawn at random; bounds the memory of
    scoring a large catalogue K - 1 times per position. None: every position.
    """
    seed_everything(config.seed)
    recommender = recommender.to(device).eval().requires_grad_(False)
    heads = MultiTokenHeads(recommender.config.transformer.hidden_size, horizon).to(device)
    loss = partial(multi_token_loss, recommender=recommender, max_targets_per_head=max_targets_per_head)
    window_length = recommender.config.transformer.max_positions
    history = fit_on_train_period(heads, log, loss, lambda batch: batch.is_target, window_length, window_stride, config, device)
    return heads.eval().requires_grad_(False), history


def multi_token_loss(heads: MultiTokenHeads, batch: EventBatch, recommender: NextItemPolicy, max_targets_per_head: int | None = None) -> Tensor:
    """Mean over heads 2..K of the cross-entropy of the target item that many events ahead of each position."""
    with torch.no_grad():
        hidden = recommender.hidden_states(batch.items)
    losses = []
    for offset in range(2, min(heads.horizon, batch.items.shape[1] - 1) + 1):
        selected = batch.is_target[:, offset:]
        if selected.any():
            states, targets = hidden[:, :-offset][selected], batch.items[:, offset:][selected]
            if max_targets_per_head is not None and len(targets) > max_targets_per_head:
                kept = torch.randperm(len(targets), device=targets.device)[:max_targets_per_head]
                states, targets = states[kept], targets[kept]
            losses.append(F.cross_entropy(heads.logits(recommender, states, offset), targets))
    if not losses:
        return sum(parameter.sum() for parameter in heads.parameters()) * 0.0
    return torch.stack(losses).mean()


@torch.no_grad()
def multi_token_lists(recommender: NextItemPolicy, heads: MultiTokenHeads, history: HistoryBatch, generation: GenerationConfig) -> Tensor:
    """For each history, position k of the list is head k's best item among those the list constraints still allow. long [B, K]"""
    state = recommender.hidden_states(history.items)[torch.arange(len(history), device=history.items.device), history.lengths - 1]
    chosen = history.items[:, :0]
    for offset in range(1, heads.horizon + 1):
        logits = constrain_logits(heads.logits(recommender, state, offset), history.items, chosen, generation)
        chosen = torch.cat([chosen, logits.argmax(dim=-1, keepdim=True)], dim=1)
    return chosen


def save_multi_token_heads(heads: MultiTokenHeads, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"hidden_size": heads.hidden_size, "horizon": heads.horizon, "weights": heads.state_dict()}, path)


def load_multi_token_heads(path: str | Path, device: torch.device | str = "cpu") -> MultiTokenHeads:
    checkpoint = torch.load(Path(path), map_location="cpu", weights_only=True)
    heads = MultiTokenHeads(checkpoint["hidden_size"], checkpoint["horizon"])
    heads.load_state_dict(checkpoint["weights"])
    return heads.to(device).eval().requires_grad_(False)


def _identity(size: int) -> nn.Linear:
    layer = nn.Linear(size, size)
    with torch.no_grad():
        layer.weight.copy_(torch.eye(size))
        layer.bias.zero_()
    return layer

In [ ]:
%%writefile /kaggle/working/src/gptrec/models/policy.py
"""The recommender: GPT-2 over item tokens, trained to predict the next item."""

from __future__ import annotations

from dataclasses import dataclass, field

import torch
from torch import Tensor, nn

from gptrec.data.log import PAD_ITEM
from gptrec.models.backbone import CausalTransformer, TransformerConfig, new_item_embedding
from gptrec.models.sequence_recommender import MASKED_LOGIT, SequenceRecommender


@dataclass(frozen=True)
class PolicyConfig:
    num_items: int
    transformer: TransformerConfig = field(default_factory=TransformerConfig)
    item_offsets: bool = False

    @classmethod
    def from_dict(cls, values: dict) -> PolicyConfig:
        return cls(
            num_items=values["num_items"],
            transformer=TransformerConfig(**values["transformer"]),
            item_offsets=values.get("item_offsets", False),
        )


class NextItemPolicy(SequenceRecommender):
    """pi(next item | items so far). Input and output share one item embedding table, as in GPT-2.

    With `config.item_offsets` every item's score gets an offset of its own, added whatever the history: all that
    a fine-tuning which may only re-weight the catalogue can change.
    """

    def __init__(self, config: PolicyConfig) -> None:
        super().__init__()
        self.config = config
        self.item_embedding = new_item_embedding(config.num_items, config.transformer.hidden_size, PAD_ITEM)
        self.backbone = CausalTransformer(config.transformer)
        self.register_parameter("item_offsets", nn.Parameter(torch.zeros(config.num_items + 1)) if config.item_offsets else None)

    def hidden_states(self, items: Tensor) -> Tensor:
        return self.backbone(self.item_embedding(items))

    def item_logits(self, hidden: Tensor) -> Tensor:
        logits = super().item_logits(hidden)
        if self.item_offsets is None:
            return logits
        logits = logits + self.item_offsets
        logits[..., PAD_ITEM] = MASKED_LOGIT
        return logits

In [ ]:
%%writefile /kaggle/working/src/gptrec/models/reward_model.py
"""The reward model: predicts how a user responds to an item, given the user's history."""

from __future__ import annotations

from dataclasses import dataclass, field

import torch
from torch import Tensor, nn

from gptrec.data.log import PAD_ITEM
from gptrec.models.backbone import EMBEDDING_INIT_STD, CausalTransformer, TransformerConfig, new_item_embedding
from gptrec.tensors import TensorGroup

UNKNOWN_FEEDBACK = -1.0


@dataclass
class Feedback(TensorGroup):
    """A user's response to items, one value per event. All tensors share one shape.

    Holds observed values, predicted expectations in [0, 1], or raw logits - the producer says which.
    """

    played_ratio: Tensor
    session_continues: Tensor
    returns: Tensor
    is_chosen: Tensor
    relative_ratio: Tensor
    liked: Tensor
    disliked: Tensor
    adopts: Tensor


@dataclass(frozen=True)
class RewardModelConfig:
    num_items: int
    transformer: TransformerConfig = field(default_factory=TransformerConfig)
    num_ratio_buckets: int = 10
    unknown_feedback: bool = False

    @classmethod
    def from_dict(cls, values: dict) -> RewardModelConfig:
        return cls(
            num_items=values["num_items"],
            transformer=TransformerConfig(**values["transformer"]),
            num_ratio_buckets=values["num_ratio_buckets"],
            unknown_feedback=values.get("unknown_feedback", False),
        )


class RewardModel(nn.Module):
    """Causal transformer over (item, feedback) events.

    The token at position t carries item t together with the feedback on item t - 1. The output at
    position t has therefore seen the candidate item t and the full history before it, but not the
    response to item t - which is exactly what it predicts.

    Three heads describe the response to an item the user consumed: how much of it, whether the session
    went on, whether the user came back. Logged events are all items somebody chose, so those heads say
    nothing about an item this user would never pick. The fourth head does: it tells the item the user
    chose from another item put in its place. The fifth predicts the played ratio relative to what is usual
    for the item, a response that no item earns just by being short or easy to finish. Three more describe
    explicit and lasting reactions where the log records them: a like, a dislike, and coming back to a new item.
    """

    def __init__(self, config: RewardModelConfig) -> None:
        super().__init__()
        self.config = config
        hidden_size = config.transformer.hidden_size
        self.item_embedding = new_item_embedding(config.num_items, hidden_size, PAD_ITEM)
        self.previous_ratio_embedding = nn.Embedding(config.num_ratio_buckets + 1 + int(config.unknown_feedback), hidden_size)
        nn.init.normal_(self.previous_ratio_embedding.weight, std=EMBEDDING_INIT_STD)
        self.backbone = CausalTransformer(config.transformer)
        self.feedback_heads = nn.Linear(hidden_size, 5)
        self.reaction_heads = nn.Linear(hidden_size, 3)

    def forward(self, items: Tensor, played_ratio: Tensor) -> Feedback:
        """Feedback logits for every position. items: long [B, L], played_ratio: float [B, L] -> [B, L] tensors.

        played_ratio[:, t] is read only by positions after t, so the last column may hold anything.
        """
        token_embeddings = self.item_embedding(items) + self.previous_ratio_embedding(self._previous_ratio_bucket(played_ratio))
        states = self.backbone(token_embeddings)
        played_ratio_logits, session_continues, returns, is_chosen, relative_ratio = self.feedback_heads(states).float().unbind(dim=-1)
        liked, disliked, adopts = self.reaction_heads(states).float().unbind(dim=-1)
        return Feedback(played_ratio_logits, session_continues, returns, is_chosen, relative_ratio, liked, disliked, adopts)

    def predict(self, items: Tensor, played_ratio: Tensor) -> Feedback:
        """Expected feedback in [0, 1] for every position."""
        return self.forward(items, played_ratio).map(torch.sigmoid)

    def _previous_ratio_bucket(self, played_ratio: Tensor) -> Tensor:
        num_buckets = self.config.num_ratio_buckets
        bucket = (played_ratio * num_buckets).long().clamp(0, num_buckets - 1)
        if self.config.unknown_feedback:
            bucket = bucket.masked_fill(played_ratio == UNKNOWN_FEEDBACK, num_buckets + 1)
        no_previous_event = torch.full_like(bucket[:, :1], num_buckets)
        return torch.cat([no_previous_event, bucket[:, :-1]], dim=1)

In [ ]:
%%writefile /kaggle/working/src/gptrec/models/sequence_recommender.py
"""What every next-item model that ranks by "sequence state . item embedding" has in common."""

from __future__ import annotations

from typing import Protocol

import torch
import torch.nn.functional as F
from torch import Tensor, nn

from gptrec.data.log import PAD_ITEM

MASKED_LOGIT = -1e9


class NextItemScorer(Protocol):
    """Anything that can rank the catalogue for a user: all that score-and-rank recommendation needs."""

    def next_item_logits(self, items: Tensor, lengths: Tensor) -> Tensor:
        """Scores of the item that follows each right-padded sequence. items: long [B, L], lengths: long [B] -> [B, V+1]"""
        ...


class SequenceRecommender(nn.Module):
    """A causal sequence model whose item scores are dot products of its state with the item embeddings.

    Subclasses own `item_embedding` (row PAD_ITEM is padding) and implement `hidden_states`. Ranking,
    item-by-item generation and full-softmax training are then the same for all of them.
    """

    item_embedding: nn.Embedding

    def hidden_states(self, items: Tensor) -> Tensor:
        """long [B, L] -> [B, L, hidden]; state t summarises items 0..t and nothing after them."""
        raise NotImplementedError

    def item_logits(self, hidden: Tensor) -> Tensor:
        """[..., hidden] -> float32 [..., V+1] scores of every item token; PAD_ITEM can never be chosen."""
        logits = F.linear(hidden, self.item_embedding.weight).float()
        logits[..., PAD_ITEM] = MASKED_LOGIT
        return logits

    def forward(self, items: Tensor) -> Tensor:
        """long [B, L] -> [B, L, V+1]; row t scores the item that follows items 0..t."""
        return self.item_logits(self.hidden_states(items))

    def next_item_logits(self, items: Tensor, lengths: Tensor) -> Tensor:
        """Scores of the item that follows each right-padded sequence. [B, V+1]"""
        last_position = lengths - 1
        return self.item_logits(self.hidden_states(items)[torch.arange(len(items), device=items.device), last_position])

In [ ]:
%%writefile /kaggle/working/src/gptrec/models/value_model.py
"""Value models: how much reward the recommender will still collect from a state (PPO's critic), and what the rest
of a list is worth after a given item (the look-ahead of guided decoding)."""

from __future__ import annotations

from dataclasses import replace

from torch import Tensor, nn

from gptrec.data.log import PAD_ITEM
from gptrec.models.backbone import CausalTransformer, new_item_embedding
from gptrec.models.policy import NextItemPolicy, PolicyConfig


class ValueModel(nn.Module):
    """A GPT-2 stack like the recommender's, with one number per position instead of a distribution over items.

    The output at position t is the expected return of everything generated after items 0..t.
    """

    def __init__(self, config: PolicyConfig) -> None:
        super().__init__()
        hidden_size = config.transformer.hidden_size
        self.item_embedding = new_item_embedding(config.num_items, hidden_size, PAD_ITEM)
        self.backbone = CausalTransformer(config.transformer)
        self.value_head = nn.Linear(hidden_size, 1)

    @classmethod
    def from_policy(cls, policy: NextItemPolicy) -> ValueModel:
        """A value model that starts from a copy of the recommender's embeddings and transformer, as in Petrov & Macdonald."""
        value_model = cls(policy.config)
        value_model.item_embedding.load_state_dict(policy.item_embedding.state_dict())
        value_model.backbone.load_state_dict(policy.backbone.state_dict())
        return value_model

    def forward(self, items: Tensor) -> Tensor:
        """long [B, L] -> float32 [B, L]"""
        return self.value_head(self.backbone(self.item_embedding(items))).squeeze(-1).float()


class ContinuationValueModel(nn.Module):
    """What the rest of a list is worth after each of its items: the discounted reward of the items still to come.

    A GPT-2 stack over item tokens like the recommender's, plus an embedding of each item's position in the list
    (0 for the items of the history): what follows an item is worth more the more items are still to come.
    """

    def __init__(self, config: PolicyConfig, horizon: int) -> None:
        super().__init__()
        self.config = config
        self.horizon = horizon
        hidden_size = config.transformer.hidden_size
        self.item_embedding = new_item_embedding(config.num_items, hidden_size, PAD_ITEM)
        self.list_position_embedding = nn.Embedding(horizon + 1, hidden_size)
        nn.init.zeros_(self.list_position_embedding.weight)
        self.backbone = CausalTransformer(config.transformer)
        self.value_head = nn.Linear(hidden_size, 1)

    @classmethod
    def from_policy(cls, policy: NextItemPolicy, horizon: int) -> ContinuationValueModel:
        """A model that starts from a copy of the recommender's embeddings and transformer."""
        value_model = cls(replace(policy.config, item_offsets=False), horizon)
        value_model.item_embedding.load_state_dict(policy.item_embedding.state_dict())
        value_model.backbone.load_state_dict(policy.backbone.state_dict())
        return value_model

    def forward(self, items: Tensor, list_positions: Tensor) -> Tensor:
        """items, list_positions: long [B, L] -> float32 [B, L]; output t is the worth of what follows the item at t."""
        embeddings = self.item_embedding(items) + self.list_position_embedding(list_positions)
        return self.value_head(self.backbone(embeddings)).squeeze(-1).float()

In [ ]:
%%writefile /kaggle/working/src/gptrec/paths.py
"""Where datasets and experiment outputs live.

By default they sit next to the package, in the repository. Set the GPTREC_HOME environment variable
before importing `gptrec` to put them elsewhere, e.g. when the code is mounted read-only.
"""

import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("GPTREC_HOME", Path(__file__).resolve().parents[1]))

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

In [ ]:
%%writefile /kaggle/working/src/gptrec/plots.py
"""Figures for the experiment reports.

Conventions: one measure per axis (measures with different units get their own panel, never a
second y-axis), a fixed colour per series across panels, a legend whenever there is more than
one series, and every reported quantity named in words (see `gptrec.labels`).
"""

from __future__ import annotations

import functools
import textwrap
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cycler import cycler
from matplotlib.figure import Figure

from gptrec.data.log import InteractionLog
from gptrec.labels import readable

SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
REFERENCE_COLOR = "#8a8984"
SURFACE_COLOR = "#fcfcfb"
SECONDARY_TEXT_COLOR = "#52514e"

STYLE = {
    "axes.prop_cycle": cycler(color=SERIES_COLORS),
    "figure.facecolor": SURFACE_COLOR,
    "axes.facecolor": SURFACE_COLOR,
    "axes.edgecolor": "#c9c8c2",
    "axes.grid": True,
    "grid.color": "#e7e6e1",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 10,
    "axes.labelcolor": SECONDARY_TEXT_COLOR,
    "xtick.color": SECONDARY_TEXT_COLOR,
    "ytick.color": SECONDARY_TEXT_COLOR,
    "lines.linewidth": 2.0,
    "lines.markersize": 6.0,
    "legend.frameon": False,
}

PANEL_WIDTH, PANEL_HEIGHT = 4.4, 3.0
MAX_POINTS_WITH_MARKERS = 30
LABEL_WIDTH = 46
HISTOGRAM_SAMPLE_SIZE = 2_000_000


def _styled(plot: Callable[..., Figure]) -> Callable[..., Figure]:
    @functools.wraps(plot)
    def plot_with_style(*args, **kwargs) -> Figure:
        with plt.rc_context(STYLE):
            figure = plot(*args, **kwargs)
            figure.tight_layout()
            return figure

    return plot_with_style


def _panels(num_panels: int, panels_per_row: int = 4):
    """A figure with a grid of axes; returns the figure and the first `num_panels` axes, the rest are hidden."""
    num_columns = min(num_panels, panels_per_row)
    num_rows = -(-num_panels // num_columns)
    figure, grid = plt.subplots(num_rows, num_columns, figsize=(PANEL_WIDTH * num_columns, PANEL_HEIGHT * num_rows), squeeze=False)
    axes = list(grid.flat)
    for unused in axes[num_panels:]:
        unused.set_visible(False)
    return figure, axes[:num_panels]


def _label(metric: str) -> str:
    """The quantity in words, wrapped to the width of a panel."""
    return textwrap.fill(readable(metric), LABEL_WIDTH)


def _minutes(seconds: float) -> str:
    """A duration in minutes, with a decimal only where whole minutes would not tell short items apart."""
    minutes = seconds / 60
    return f"{minutes:.1f}" if minutes < 10 else f"{minutes:.0f}"


def _marker(num_points: int) -> str | None:
    return "o" if num_points <= MAX_POINTS_WITH_MARKERS else None


@_styled
def small_multiples(curves: pd.DataFrame, x_label: str) -> Figure:
    """One panel per column of `curves`, each drawn against the index."""
    figure, axes = _panels(len(curves.columns))
    for axis, column in zip(axes, curves.columns):
        axis.plot(curves.index, curves[column], marker=_marker(len(curves)))
        axis.set(title=_label(column), xlabel=x_label)
    return figure


@_styled
def series_per_panel(panels: dict[str, pd.DataFrame], x_label: str) -> Figure:
    """One panel per measure; inside a panel one line per column. All tables must list the same columns in the same order."""
    figure, axes = _panels(len(panels))
    for axis, (measure, table) in zip(axes, panels.items()):
        for series in table.columns:
            axis.plot(table.index, table[series], marker=_marker(len(table)), label=series)
        axis.set(title=_label(measure), xlabel=x_label)
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=len(labels))
    return figure


@_styled
def calibration_curves(calibration_tables: dict[str, pd.DataFrame]) -> Figure:
    """Mean observed against mean predicted played ratio, one panel per reward model. Tables come from `calibration_table`."""
    figure, axes = _panels(len(calibration_tables))
    for axis, (name, table) in zip(axes, calibration_tables.items()):
        axis.plot([0, 1], [0, 1], color=REFERENCE_COLOR, linestyle="--", linewidth=1.0, label="perfect calibration")
        axis.plot(table.mean_predicted, table.mean_observed, marker="o", color=SERIES_COLORS[0], label="reward model")
        axis.set(title=f"reward model: {name}", xlabel="mean predicted played ratio", ylabel="mean observed played ratio")
        axis.legend()
    return figure


@_styled
def feedback_overview(log: InteractionLog) -> Figure:
    """The played-ratio distribution, and which items are easy to finish: mean played ratio by item duration and popularity."""
    sample_size = min(log.num_events, HISTOGRAM_SAMPLE_SIZE)
    sample = np.sort(np.random.default_rng(0).choice(log.num_events, size=sample_size, replace=False))
    items = pd.DataFrame({
        "duration_seconds": log.items.duration_seconds[1:],
        "popularity": log.items.train_popularity[1:],
        "mean_played_ratio": log.items.mean_played_ratio[1:],
    })
    by_duration = items.groupby(pd.qcut(items.duration_seconds, 10, duplicates="drop"), observed=True).mean_played_ratio.mean()
    by_popularity = items.groupby(pd.qcut(items.popularity.rank(method="first"), 10, labels=False)).mean_played_ratio.mean()

    figure, (histogram, duration, popularity) = _panels(3)
    histogram.hist(np.asarray(log.played_ratio)[sample], bins=50, color=SERIES_COLORS[0])
    histogram.set(title="played ratio of events", xlabel="played ratio", ylabel="events")
    bins = range(len(by_duration))
    duration.bar(bins, by_duration.to_numpy(), width=0.7, color=SERIES_COLORS[0])
    duration.set_xticks(bins, [_minutes(interval.right) for interval in by_duration.index])
    duration.set(title="mean played ratio by item duration", xlabel="item duration up to, minutes (tenths of the catalogue)")
    popularity.bar(by_popularity.index, by_popularity.to_numpy(), width=0.7, color=SERIES_COLORS[0])
    popularity.set(title="mean played ratio by item popularity", xlabel="popularity decile (9 = most popular)")
    return figure


@_styled
def reward_relevance_frontier(metrics: pd.DataFrame, relevance_rows: list[str], reward_rows: dict[str, str], anchor: str = "pretrained") -> Figure:
    """Relevance against predicted reward, one point per run.

    metrics: one column per run (as from `metrics_table`); `anchor` is the run every other run started from
    and is drawn last, where an ever stronger pull towards it ends. reward_rows: judge name -> metric row
    holding the reward that judge predicts. One panel per relevance row, one line per judge.
    """
    runs = [run for run in metrics.columns if run != anchor] + [anchor]
    last_judge = list(reward_rows)[-1]
    figure, axes = _panels(len(relevance_rows))
    for axis, relevance_row in zip(axes, relevance_rows):
        relevance = metrics.loc[relevance_row, runs]
        for judge, reward_row in reward_rows.items():
            axis.plot(metrics.loc[reward_row, runs], relevance, marker="o", label=f"reward judged by: {judge}")
        for run in runs:
            point = (metrics.loc[reward_rows[last_judge], run], relevance[run])
            axis.annotate(run, point, textcoords="offset points", xytext=(4, 4), fontsize=8, color=SECONDARY_TEXT_COLOR)
        axis.set(title=_label(relevance_row), xlabel="predicted played ratio of the generated list")
        axis.legend()
    return figure


@_styled
def trade_off(curves: dict[str, pd.DataFrame], reward_column: str, measure_columns: list[str]) -> Figure:
    """What several methods pay for predicted reward; one panel per measure, e.g. relevance or catalogue coverage.

    curves: method -> table with one row per run and the metrics as columns. A method with several runs is a
    line through them in table order; a method with a single run is one marker.
    """
    figure, axes = _panels(len(measure_columns))
    for axis, measure_column in zip(axes, measure_columns):
        for method, runs in curves.items():
            line_style = "-" if len(runs) > 1 else "none"
            axis.plot(runs[reward_column], runs[measure_column], marker="o", linestyle=line_style, label=method)
        axis.set(title=_label(measure_column), xlabel=_label(reward_column))
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=min(len(labels), 3))
    return figure

In [ ]:
%%writefile /kaggle/working/src/gptrec/reports.py
"""Persisting the tables and figures an experiment produces."""

from __future__ import annotations

from pathlib import Path
from typing import TypeVar

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.figure import Figure

Table = TypeVar("Table", pd.DataFrame, pd.Series)


class ReportWriter:
    """Writes every table as CSV and every figure as PNG into one directory."""

    def __init__(self, directory: str | Path, show_figures: bool = True) -> None:
        self.directory = Path(directory)
        self.directory.mkdir(parents=True, exist_ok=True)
        self._show_figures = show_figures

    def table(self, name: str, table: Table) -> Table:
        """Saves the table as <name>.csv and returns it, so a notebook cell can display what was saved."""
        table.to_csv(self.directory / f"{name}.csv")
        return table

    def figure(self, name: str, figure: Figure) -> None:
        """Saves the figure as <name>.png. It stays open for the notebook to display unless figures are hidden."""
        figure.savefig(self.directory / f"{name}.png", dpi=150, bbox_inches="tight")
        if not self._show_figures:
            plt.close(figure)

In [ ]:
%%writefile /kaggle/working/src/gptrec/rollout.py
"""Generating recommendations item by item, and simulating the user's response to them.

The recommender only reads item tokens, so its choices do not depend on the simulated feedback.
Generation and simulation are therefore separate steps, and any reward model can score any list.
"""

from __future__ import annotations

from dataclasses import dataclass, replace
from enum import Enum

import torch
import torch.nn.functional as F
from torch import Tensor

from gptrec.data.batches import HistoryBatch
from gptrec.data.log import PAD_ITEM, ItemTable
from gptrec.evaluation.behavior import pairwise_item_distances
from gptrec.evaluation.relevance import contains
from gptrec.models.policy import NextItemPolicy
from gptrec.models.sequence_recommender import MASKED_LOGIT, NextItemScorer
from gptrec.models.value_model import ValueModel
from gptrec.models.reward_model import UNKNOWN_FEEDBACK, Feedback, RewardModel
from gptrec.tensors import TensorGroup


@dataclass(frozen=True)
class GenerationConfig:
    horizon: int = 10
    greedy: bool = False
    temperature: float = 1.0
    block_generated_items: bool = True
    block_history_items: bool = False
    sample_from_top: int | None = None


class RatioBaseline(str, Enum):
    """What the predicted played ratio of an item is measured against before it counts as reward."""

    NONE = "none"
    ITEM_MEAN = "item_mean"
    OTHER_USER = "other_user"


class SimulatedFeedback(str, Enum):
    """What the reward model is told about the user's response to an already generated item."""

    EXPECTED = "expected"
    SAMPLED = "sampled"
    ASSUME_COMPLETED = "assume_completed"
    UNKNOWN = "unknown"
    LOGGED = "logged"


SIMULATION_ASSUMPTIONS = (SimulatedFeedback.EXPECTED, SimulatedFeedback.SAMPLED, SimulatedFeedback.ASSUME_COMPLETED)


@dataclass
class GeneratedActions(TensorGroup):
    actions: Tensor
    log_probs: Tensor
    entropy: Tensor


def constrain_logits(logits: Tensor, history_items: Tensor, generated_items: Tensor, config: GenerationConfig) -> Tensor:
    """Applies temperature and removes the items that may not be recommended next.

    logits: [B, V+1], history_items: long [B, H], generated_items: long [B, k] -> [B, V+1]
    """
    blocked = [torch.full_like(history_items[:, :1], PAD_ITEM)]
    if config.block_generated_items:
        blocked.append(generated_items)
    if config.block_history_items:
        blocked.append(history_items)
    allowed = (logits / config.temperature).scatter(1, torch.cat(blocked, dim=1), MASKED_LOGIT)
    if config.sample_from_top is not None:
        lowest_kept_logit = allowed.topk(config.sample_from_top, dim=-1).values[:, -1:]
        allowed = allowed.masked_fill(allowed < lowest_kept_logit, MASKED_LOGIT)
    return allowed


def histories_of_other_users(history: HistoryBatch, shift: int = 1) -> HistoryBatch:
    """Row b gets the history of row b - shift, so that every list in a batch meets a user it was not made for.

    The pairing is arbitrary, not random. `shift` must not be a multiple of the batch size, and where each
    history is repeated several times in a row it must be that number of repeats.
    """
    return history.map(lambda tensor: tensor.roll(shift, dims=0))


def append_actions(history: HistoryBatch, actions: Tensor) -> Tensor:
    """Item tokens of each history followed directly by its generated items. long [B, H + K]"""
    horizon = actions.shape[1]
    items = F.pad(history.items, (0, horizon), value=PAD_ITEM)
    return items.scatter(1, _action_positions(history, horizon), actions)


def _action_positions(history: HistoryBatch, horizon: int) -> Tensor:
    """Column of the k-th generated item in the extended sequence. long [B, K]"""
    return history.lengths[:, None] + torch.arange(horizon, device=history.lengths.device)[None, :]


@torch.no_grad()
def recommend_top_k(scorer: NextItemScorer, history: HistoryBatch, k: int, config: GenerationConfig) -> Tensor:
    """Score-and-rank: the k best next items, none of them fed back to the model. long [B, k], best first."""
    nothing_generated = history.items[:, :0]
    logits = scorer.next_item_logits(history.items, history.lengths)
    return constrain_logits(logits, history.items, nothing_generated, config).topk(k, dim=-1).indices


@torch.no_grad()
def generate_actions(policy: NextItemPolicy, history: HistoryBatch, config: GenerationConfig) -> GeneratedActions:
    """Extends each history with `config.horizon` items, feeding every generated item back as input."""
    batch_index = torch.arange(len(history), device=history.items.device)
    history_width = history.items.shape[1]
    items = F.pad(history.items, (0, config.horizon), value=PAD_ITEM)
    actions = torch.zeros((len(history), 0), dtype=torch.long, device=items.device)
    log_probs, entropies = [], []

    for step in range(config.horizon):
        last_position = history.lengths - 1 + step
        state = policy.hidden_states(items[:, : history_width + step])[batch_index, last_position]
        step_log_probs = F.log_softmax(constrain_logits(policy.item_logits(state), history.items, actions, config), dim=-1)
        action = step_log_probs.argmax(dim=-1) if config.greedy else torch.multinomial(step_log_probs.exp(), num_samples=1).squeeze(1)

        items[batch_index, last_position + 1] = action
        actions = torch.cat([actions, action[:, None]], dim=1)
        log_probs.append(step_log_probs[batch_index, action])
        entropies.append(_entropy(step_log_probs))

    return GeneratedActions(actions, torch.stack(log_probs, dim=1), torch.stack(entropies, dim=1))


def action_log_distributions(policy: NextItemPolicy, history: HistoryBatch, actions: Tensor, config: GenerationConfig) -> Tensor:
    """log pi(. | history, actions[:, :k]) for every step k under the constraints used while generating.

    Differentiable counterpart of `generate_actions`: one pass over the already generated sequence. [B, K, V+1]
    """
    horizon = actions.shape[1]
    hidden = policy.hidden_states(append_actions(history, actions))
    state_positions = _action_positions(history, horizon) - 1
    states = hidden.gather(1, state_positions[:, :, None].expand(-1, -1, hidden.shape[-1]))
    logits = policy.item_logits(states)
    constrained = [constrain_logits(logits[:, step], history.items, actions[:, :step], config) for step in range(horizon)]
    return F.log_softmax(torch.stack(constrained, dim=1), dim=-1)


def state_values(value_model: ValueModel, history: HistoryBatch, actions: Tensor) -> Tensor:
    """The value model's estimate of the return still to come, in each state in which an item of the list was chosen. [B, K]"""
    values = value_model(append_actions(history, actions))
    return values.gather(1, _action_positions(history, actions.shape[1]) - 1)


@torch.no_grad()
def simulate_feedback(reward_model: RewardModel, history: HistoryBatch, actions: Tensor, mode: SimulatedFeedback, logged_played_ratio: Tensor | None = None) -> Feedback:
    """The reward model's expected response to every generated item, played out step by step. [B, K] tensors.

    The model sees the real feedback on the history; for generated items it sees the feedback chosen by `mode`.
    logged_played_ratio: float [B, K], the real responses to `actions`; needed only for the LOGGED mode.
    """
    mode = SimulatedFeedback(mode)
    if mode is SimulatedFeedback.UNKNOWN and not reward_model.config.unknown_feedback:
        raise ValueError("this reward model was trained without hidden responses, so it cannot read an unknown one")
    if mode is SimulatedFeedback.LOGGED and logged_played_ratio is None:
        raise ValueError("the logged mode needs the responses the user really gave")
    batch_index = torch.arange(len(history), device=actions.device)
    history_width, horizon = history.items.shape[1], actions.shape[1]
    items = append_actions(history, actions)
    played_ratio = F.pad(history.played_ratio, (0, horizon), value=0.0)
    steps = []

    for step in range(horizon):
        position = history.lengths + step
        visible = history_width + step + 1
        prediction = reward_model.predict(items[:, :visible], played_ratio[:, :visible])
        expected = prediction.map(lambda values: values[batch_index, position])
        logged = None if logged_played_ratio is None else logged_played_ratio[:, step].to(played_ratio.dtype)
        played_ratio[batch_index, position] = _fed_back_played_ratio(expected.played_ratio.to(played_ratio.dtype), mode, logged)
        steps.append(expected)

    return Feedback.stack(steps, dim=1)


def _fed_back_played_ratio(expected_played_ratio: Tensor, mode: SimulatedFeedback, logged_played_ratio: Tensor | None) -> Tensor:
    if mode is SimulatedFeedback.EXPECTED:
        return expected_played_ratio
    if mode is SimulatedFeedback.SAMPLED:
        return torch.bernoulli(expected_played_ratio)
    if mode is SimulatedFeedback.ASSUME_COMPLETED:
        return torch.ones_like(expected_played_ratio)
    if mode is SimulatedFeedback.UNKNOWN:
        return torch.full_like(expected_played_ratio, UNKNOWN_FEEDBACK)
    if mode is SimulatedFeedback.LOGGED:
        return logged_played_ratio
    raise ValueError(f"unknown simulated feedback mode: {mode}")


def _entropy(log_probs: Tensor) -> Tensor:
    return -(log_probs.exp() * log_probs).sum(dim=-1)


@dataclass(frozen=True)
class RewardSpec:
    """How the response to one recommended item becomes a scalar reward.

    reward = P(chosen) * (played_ratio_weight * ratio term
                          + relative_ratio_weight * relative played ratio
                          + session_continues_weight * P(session continues)
                          + returns_weight * P(user returns)
                          + liked_weight * P(like) - disliked_weight * P(dislike)
                          + adoption_weight * P(the user comes back to the item))
             + logged_relevance_weight * [the user really consumed the item next] / log2(position in the list + 1)
             + diversity_weight * distance of the item to the items before it in the list
             + final_returns_weight * P(user returns after the last item), for the last item only

    P(chosen) is left out unless `weigh_by_choice` is set. The first line is predicted by a reward model
    for any item. The second is read from the log and exists only for moments whose future is known:
    it is the immediate reward of NDCG in Petrov & Macdonald (arXiv:2403.04875). The third depends on the
    rest of the list, so it is the one term that scoring items one at a time cannot maximise. The fourth
    pays the whole list for what happens after it: no item earns it on its own, so it is the one reward
    whose credit has to travel back along the list.

    The relative played ratio is the played ratio as a quantile among everyone who consumed the item. Its
    mean is 0.5 for every item, so no item earns it by being short or easy to finish.

    The ratio term is the predicted played ratio minus `ratio_baseline`. Without a baseline a policy earns
    the most by showing everybody the items everybody finishes. With one it earns only what depends on who
    receives the item: measured against another user, a list that ignores its user earns zero on average.
    """

    played_ratio_weight: float = 1.0
    ratio_baseline: RatioBaseline = RatioBaseline.NONE
    relative_ratio_weight: float = 0.0
    session_continues_weight: float = 0.0
    returns_weight: float = 0.0
    weigh_ratio_by_duration: bool = False
    weigh_by_choice: bool = False
    logged_relevance_weight: float = 0.0
    diversity_weight: float = 0.0
    liked_weight: float = 0.0
    disliked_weight: float = 0.0
    adoption_weight: float = 0.0
    final_returns_weight: float = 0.0

    @property
    def needs_logged_future(self) -> bool:
        return self.logged_relevance_weight != 0.0

    @property
    def needs_other_user(self) -> bool:
        return RatioBaseline(self.ratio_baseline) is RatioBaseline.OTHER_USER

    @property
    def depends_on_the_list(self) -> bool:
        return self.diversity_weight != 0.0

    @property
    def rewards_the_end_of_the_list(self) -> bool:
        return self.final_returns_weight != 0.0


class RewardFunction:
    """Computes the reward of `RewardSpec` for lists of items."""

    def __init__(self, spec: RewardSpec, items: ItemTable, device: torch.device | str, item_embeddings: Tensor | None = None) -> None:
        """item_embeddings: [V+1, d], the space in which the distance between two items is measured; needed only for the diversity term."""
        self.spec = spec
        duration = torch.as_tensor(items.duration_seconds, dtype=torch.float32, device=device)
        self._relative_duration = duration / duration[1:].median().clamp_min(1e-6)
        self._item_mean_played_ratio = torch.as_tensor(items.mean_played_ratio, dtype=torch.float32, device=device)
        if spec.depends_on_the_list and item_embeddings is None:
            raise ValueError("this reward has a diversity term, which needs item embeddings to measure distances in")
        self.unit_item_embeddings = None if item_embeddings is None else F.normalize(item_embeddings.detach().float().to(device), dim=-1)

    def __call__(self, feedback: Feedback, lists: Tensor, future_items: Tensor | None = None, feedback_for_other_user: Feedback | None = None) -> Tensor:
        """Reward of every item of every list. [B, K]

        feedback: the response to each item, predicted or logged, with the shape of `lists`.
        lists: long [B, K], column k is position k + 1 of a recommendation list.
        future_items: long [B, M], what each user really consumed next; needed only for logged relevance.
        feedback_for_other_user: the predicted response of another user to the same lists; needed only for that baseline.
        """
        reward = self.item_reward(feedback, lists, future_items, feedback_for_other_user)
        if self.spec.depends_on_the_list:
            reward = reward + self.spec.diversity_weight * distance_to_earlier_items(lists, self.unit_item_embeddings)
        if self.spec.rewards_the_end_of_the_list:
            is_last_item = torch.zeros_like(reward)
            is_last_item[:, -1] = 1.0
            reward = reward + self.spec.final_returns_weight * is_last_item * feedback.returns
        return reward

    def item_reward(self, feedback: Feedback, lists: Tensor, future_items: Tensor | None = None, feedback_for_other_user: Feedback | None = None) -> Tensor:
        """The part of the reward an item earns whatever else is in its list. Arguments and shape as in `__call__`."""
        spec = self.spec
        ratio_term = feedback.played_ratio - self._ratio_baseline(lists, feedback_for_other_user)
        if spec.weigh_ratio_by_duration:
            ratio_term = ratio_term * self._relative_duration[lists]
        reward = (
            spec.played_ratio_weight * ratio_term
            + spec.relative_ratio_weight * feedback.relative_ratio
            + spec.session_continues_weight * feedback.session_continues
            + spec.returns_weight * feedback.returns
            + spec.liked_weight * feedback.liked
            - spec.disliked_weight * feedback.disliked
            + spec.adoption_weight * feedback.adopts
        )
        if spec.weigh_by_choice:
            reward = reward * feedback.is_chosen
        if spec.needs_logged_future:
            if future_items is None:
                raise ValueError("this reward has a logged-relevance term, which needs the items the user really consumed next")
            reward = reward + spec.logged_relevance_weight * discounted_logged_relevance(lists, future_items)
        return reward

    def _ratio_baseline(self, lists: Tensor, feedback_for_other_user: Feedback | None) -> Tensor | float:
        baseline = RatioBaseline(self.spec.ratio_baseline)
        if baseline is RatioBaseline.ITEM_MEAN:
            return self._item_mean_played_ratio[lists]
        if baseline is RatioBaseline.OTHER_USER:
            if feedback_for_other_user is None:
                raise ValueError("this reward is measured against another user, whose predicted response is missing")
            return feedback_for_other_user.played_ratio
        return 0.0


def discounted_logged_relevance(lists: Tensor, future_items: Tensor) -> Tensor:
    """1 / log2(position + 1) where the listed item is among the user's real next items, else 0: the terms of DCG. [B, K]"""
    position = torch.arange(1, lists.shape[1] + 1, device=lists.device)
    return contains(lists, future_items).float() / torch.log2(position + 1.0)


def distance_to_earlier_items(lists: Tensor, unit_item_embeddings: Tensor) -> Tensor:
    """For each listed item, its cosine distance to the items before it in the list, summed and divided by K - 1. [B, K]

    The first item earns nothing. Over a list the terms add up to K / 2 times its mean pairwise distance, so
    what an item earns depends on its neighbours: this is a reward for the list as a whole.
    """
    list_length = lists.shape[1]
    comes_earlier = torch.ones(list_length, list_length, dtype=torch.bool, device=lists.device).tril(diagonal=-1)
    return (pairwise_item_distances(lists, unit_item_embeddings) * comes_earlier).sum(dim=2) / max(1, list_length - 1)


MAX_SEQUENCES_PER_PASS = 4096
USERS_PER_GREEDY_PASS = 1024


@dataclass
class ScoredCandidates(TensorGroup):
    """The recommender's most likely next items for each user, with the response a reward model predicts to each."""

    items: Tensor
    log_probs: Tensor
    feedback: Feedback
    feedback_for_other_user: Feedback


@torch.no_grad()
def score_candidates(scorer: NextItemScorer, reward_model: RewardModel, history: HistoryBatch, num_candidates: int, config: GenerationConfig) -> ScoredCandidates:
    """Lets the reward model judge each of the recommender's `num_candidates` most likely next items on its own.

    Every candidate is judged twice: for the user it is a candidate for, and for another user of the batch.
    """
    nothing_generated = history.items[:, :0]
    logits = constrain_logits(scorer.next_item_logits(history.items, history.lengths), history.items, nothing_generated, config)
    log_probs, candidates = F.log_softmax(logits, dim=-1).topk(num_candidates, dim=-1)
    users_per_pass = max(1, MAX_SEQUENCES_PER_PASS // num_candidates)

    def judged_for(recipients: HistoryBatch) -> Feedback:
        feedback = []
        for first_user in range(0, len(recipients), users_per_pass):
            users = slice(first_user, first_user + users_per_pass)
            one_item_lists = candidates[users].reshape(-1, 1)
            history_per_candidate = recipients[users].repeat_interleave(num_candidates)
            predicted = simulate_feedback(reward_model, history_per_candidate, one_item_lists, SimulatedFeedback.EXPECTED)
            feedback.append(predicted.map(lambda values: values.reshape(-1, num_candidates)))
        return Feedback.concatenate(feedback)

    return ScoredCandidates(candidates, log_probs, judged_for(history), judged_for(histories_of_other_users(history)))


def rerank(candidates: ScoredCandidates, reward_function: RewardFunction, list_length: int, relevance_weight: float) -> Tensor:
    """The greedy alternative to RL: the `list_length` candidates with the highest `predicted reward + relevance_weight * log-probability`.

    Weight 0 ranks by reward alone; a large weight returns the recommender's own top list. A reward that depends
    on the list is maximised one position at a time, see `pick_greedily`. long [U, list_length], best first.
    """
    reward = reward_function.item_reward(candidates.feedback, candidates.items, feedback_for_other_user=candidates.feedback_for_other_user)
    score = reward + relevance_weight * candidates.log_probs
    if not reward_function.spec.depends_on_the_list:
        return candidates.items.gather(1, score.topk(list_length, dim=-1).indices)
    return pick_greedily(candidates.items, score, reward_function.unit_item_embeddings, reward_function.spec.diversity_weight, list_length)


def pick_greedily(items: Tensor, scores: Tensor, unit_item_embeddings: Tensor, diversity_weight: float, list_length: int) -> Tensor:
    """Maximal marginal relevance: fills a list one position at a time with the best candidate given the items already picked.

    A candidate is worth its score plus `diversity_weight` times what `distance_to_earlier_items` would pay it
    at that position. This is the greedy maximiser of the list reward RL is trained on, and needs no training.
    items: long [U, N] candidates; scores: float [U, N] -> long [U, list_length]
    """
    picked_lists = []
    for first_user in range(0, len(items), USERS_PER_GREEDY_PASS):
        users = slice(first_user, first_user + USERS_PER_GREEDY_PASS)
        candidates, remaining_scores = items[users], scores[users].clone()
        embedded = unit_item_embeddings[candidates]
        user_index = torch.arange(len(candidates), device=candidates.device)
        similarity_to_picked = torch.zeros_like(remaining_scores)
        picked = []
        for position in range(list_length):
            distance_to_picked = (position - similarity_to_picked) / max(1, list_length - 1)
            best = (remaining_scores + diversity_weight * distance_to_picked).argmax(dim=1)
            picked.append(candidates[user_index, best])
            similarity_to_picked = similarity_to_picked + (embedded @ embedded[user_index, best][:, :, None]).squeeze(2).float()
            remaining_scores[user_index, best] = MASKED_LOGIT
        picked_lists.append(torch.stack(picked, dim=1))
    return torch.cat(picked_lists)


@torch.no_grad()
def best_of_sampled_lists(
    policy: NextItemPolicy,
    reward_model: RewardModel,
    reward_function: RewardFunction,
    history: HistoryBatch,
    num_lists: int,
    discount: float,
    config: GenerationConfig,
    simulated_feedback: SimulatedFeedback,
) -> Tensor:
    """Samples `num_lists` lists per user and keeps the one with the highest predicted discounted return. long [B, K]

    The test-time scaling of Matveev et al. (arXiv:2607.02818), which serves the first item of the best
    roll-out; here the whole roll-out is the recommendation list. Nothing is trained.
    """
    sampling = replace(config, greedy=False)
    discounts = discount ** torch.arange(config.horizon, device=history.items.device)
    users_per_pass = max(1, MAX_SEQUENCES_PER_PASS // num_lists)
    best_lists = []
    for first_user in range(0, len(history), users_per_pass):
        repeated_history = history[first_user : first_user + users_per_pass].repeat_interleave(num_lists)
        lists = generate_actions(policy, repeated_history, sampling).actions
        rewards = reward_function(simulate_feedback(reward_model, repeated_history, lists, simulated_feedback), lists)
        predicted_return = (rewards * discounts).sum(dim=1).reshape(-1, num_lists)
        lists = lists.reshape(-1, num_lists, config.horizon)
        best_lists.append(lists[torch.arange(len(lists), device=lists.device), predicted_return.argmax(dim=1)])
    return torch.cat(best_lists)

In [ ]:
%%writefile /kaggle/working/src/gptrec/seeds.py
"""The same study under several random seeds: how much of a result is the draw of one run.

A seed fixes every source of randomness of one experiment: the initial weights of the recommender, of the
reward models and of the baselines, the order of their training batches, and the lists sampled while
fine-tuning. The interaction log and the requests the recommenders are judged on do not depend on it.
"""

from __future__ import annotations

import shutil
import time
from dataclasses import replace
from pathlib import Path
from typing import Sequence

import pandas as pd
import polars as pl
import torch

from gptrec.experiment import Experiment, ExperimentConfig, smoke_test_config

SEED_STAGES = (
    "pretrain_policy",
    "train_reward_models",
    "finetune",
    "sweep_kl_weight",
    "sweep_kl_budget",
    "compare_advantage_estimators",
    "explain_reward_gain",
    "compare_alternatives_to_rl",
    "sweep_relevance_in_reward",
    "sweep_personal_rewards",
    "sweep_relative_reward",
    "ablate_sequence_credit",
)
FRONTIER_SEED_STAGES = (
    "pretrain_policy",
    "train_reward_models",
    "finetune",
    "sweep_kl_weight",
    "restrict_finetuning_capacity",
    "decompose_policy_change",
    "align_at_inference",
    "learn_from_logged_preferences",
    "predict_lists_without_feedback",
    "interpolate_weights",
)
FRONTIER_SEED_REPORTS = {
    "rl_vs_pretrained": [0],
    "kl_sweep": [0],
    "capacity_sweep": [0],
    "policy_change_parts": [0],
    "item_effect_share": [0],
    "inference_time_alignment": [0],
    "logged_preference_methods": [0],
    "logged_preferences_significance_reward_weighted_relative_ratio_temperature_0.3": [0],
    "logged_preferences_significance_within_user_DPO_beta_0.5": [0],
    "multi_token_lists": [0],
    "multi_token_position_hits": [0],
    "weight_interpolation": [0],
    "stage_minutes": [0],
}
CONTROL_SEED_STAGES = (
    "pretrain_policy",
    "train_reward_models",
    "control_for_continued_training",
)
CONTROL_SEED_REPORTS = {
    "continued_training_control": [0],
    "continued_training_significance": [0, 1, 2],
    "policy_test_relevance": [0],
    "reward_model_played_ratio": [0, 1],
    "policy_training": [0],
    "continued_next_item_training": [0],
    "reward_weighted_training_relative_ratio_0.3": [0],
    "stage_minutes": [0],
}
SEED_REPORTS = {
    "rl_vs_pretrained": [0],
    "policy_test_relevance": [0],
    "reward_model_played_ratio": [0, 1],
    "reward_model_relative_ratio": [0, 1],
    "reward_model_retention": [0, 1, 2],
    "rl_vs_pretrained_significance": [0],
    "kl_sweep": [0],
    "kl_sweep_validation": [0],
    "kl_budget_sweep": [0],
    "advantage_estimators": [0],
    "sequence_credit_ablation": [0],
    "reward_gain_decomposition": [0],
    "alternatives_to_rl": [0],
    "relevance_reward_sweep": [0],
    "personal_reward_sweep": [0],
    "relative_reward_sweep": [0],
    "relative_reward_reranking": [0],
    "stage_minutes": [0],
}
TIME_SAFETY_FACTOR = 1.2
CHANGE_INTERVAL = ("change_ci_low", "change_ci_high")


def with_seed(config: ExperimentConfig, seed: int) -> ExperimentConfig:
    """`config` with every source of randomness drawn from `seed`."""

    def reseeded(optimization):
        return replace(optimization, seed=seed)

    return replace(
        config,
        policy_optimization=reseeded(config.policy_optimization),
        reward_model_optimization=reseeded(config.reward_model_optimization),
        baselines=replace(config.baselines, optimization=reseeded(config.baselines.optimization)),
        liked_finetuning=reseeded(config.liked_finetuning),
        logged_returns=replace(config.logged_returns, optimization=reseeded(config.logged_returns.optimization)),
        rl=replace(config.rl, seed=seed),
        preference_finetuning=reseeded(config.preference_finetuning),
        multi_token_optimization=reseeded(config.multi_token_optimization),
        continuation_training=replace(config.continuation_training, seed=seed),
    )


def run_seeds(
    raw_events: pl.DataFrame,
    config: ExperimentConfig,
    seeds: Sequence[int],
    home: str | Path,
    device: torch.device,
    stages: Sequence[str] = SEED_STAGES,
    time_budget_hours: float | None = None,
) -> dict[int, Path]:
    """Runs `stages` of the experiment once per seed, each in its own directory under `home`. Returns the report directory of every finished seed.

    The interaction log is built for the first seed and copied for the others. `time_budget_hours` counts
    from now: a seed that would not finish inside it, judging by the one before, is skipped and reported.
    """
    home = Path(home)
    deadline = None if time_budget_hours is None else time.monotonic() + time_budget_hours * 3600
    report_directories: dict[int, Path] = {}
    log_of_finished_seed: Path | None = None
    seconds_of_last_seed = 0.0
    for seed in seeds:
        started = time.monotonic()
        if deadline is not None and started + TIME_SAFETY_FACTOR * seconds_of_last_seed > deadline:
            print(f"skipped seed {seed}: it would not finish inside the time budget")
            continue
        hours_left = None if deadline is None else (deadline - started) / 3600
        experiment = Experiment(with_seed(config, seed), home / f"seed_{seed}", device, show_figures=False, time_budget_hours=hours_left)
        if log_of_finished_seed is not None and not experiment.log_directory.exists():
            shutil.copytree(log_of_finished_seed, experiment.log_directory)
        experiment.prepare_data(raw_events)
        for stage in stages:
            getattr(experiment, stage)()
        log_of_finished_seed = experiment.log_directory
        report_directories[seed] = experiment.reports.directory
        seconds_of_last_seed = time.monotonic() - started
        print(f"seed {seed} finished in {seconds_of_last_seed / 60:.0f} minutes")
        del experiment
        if device.type == "cuda":
            torch.cuda.empty_cache()
    return report_directories


def summarise_seeds(report_directories: dict[int, Path], output_directory: str | Path, reports: dict[str, list[int]] = SEED_REPORTS) -> dict[str, pd.DataFrame]:
    """Mean and standard deviation over seeds of every number of the named reports.

    For each report, writes "<report>_mean.csv", "<report>_std.csv", "<report>_num_seeds.csv" and
    "<report>_mean_std.csv" into `output_directory`; the last holds "mean ± std" as text and is what is returned.
    A report missing for a seed, or a run missing from a seed's report, is left out of that number's average.
    A report of paired comparisons (`gptrec.evaluation.significance.paired_bootstrap` per row) gets one more column
    there, "seeds_excluding_zero": in how many of its seeds the 95% interval of the change excludes zero.
    """
    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)
    summaries = {}
    for report, label_columns in reports.items():
        per_seed = {
            seed: pd.read_csv(directory / f"{report}.csv", index_col=label_columns)
            for seed, directory in report_directories.items()
            if (directory / f"{report}.csv").exists()
        }
        if not per_seed:
            continue
        stacked = pd.concat(per_seed, names=["seed"])
        over_seeds = stacked.groupby(level=list(range(1, stacked.index.nlevels)), sort=False)
        mean, deviation, num_seeds = over_seeds.mean(), over_seeds.std(), over_seeds.count()
        mean.to_csv(output_directory / f"{report}_mean.csv")
        deviation.to_csv(output_directory / f"{report}_std.csv")
        num_seeds.to_csv(output_directory / f"{report}_num_seeds.csv")
        summaries[report] = _as_text(mean) + " ± " + _as_text(deviation)
        if set(CHANGE_INTERVAL) <= set(stacked.columns):
            low, high = (stacked[column] for column in CHANGE_INTERVAL)
            seeds_excluding_zero = ((low > 0) | (high < 0)).groupby(level=list(range(1, stacked.index.nlevels)), sort=False).sum()
            summaries[report]["seeds_excluding_zero"] = seeds_excluding_zero.astype(int).astype(str) + " of " + num_seeds[CHANGE_INTERVAL[0]].astype(str)
        summaries[report].to_csv(output_directory / f"{report}_mean_std.csv")
    return summaries


def run_seeds_smoke_test(
    raw_events: pl.DataFrame,
    config: ExperimentConfig,
    home: str | Path,
    device: torch.device,
    max_users: int = 200,
    stages: Sequence[str] = SEED_STAGES,
    reports: dict[str, list[int]] = SEED_REPORTS,
) -> None:
    """Two seeds of `config`, shrunk as in `gptrec.experiment.run_smoke_test`, through `stages` and the summary of `reports` over seeds.

    The first of `reports` is the session's main result, which the smoke test must produce.
    """
    report_directories = run_seeds(raw_events, smoke_test_config(config, max_users), seeds=(0, 1), home=home, device=device, stages=stages)
    summaries = summarise_seeds(report_directories, Path(home) / "over_seeds", reports)
    main_report = next(iter(reports))
    if main_report not in summaries:
        raise RuntimeError(f"the summary over seeds is missing {main_report}, the main result of the session")


def _as_text(table: pd.DataFrame) -> pd.DataFrame:
    return table.apply(lambda column: column.map("{:.4f}".format))

In [ ]:
%%writefile /kaggle/working/src/gptrec/tensors.py
"""A base for dataclasses that bundle tensors describing the same batch."""

from __future__ import annotations

from dataclasses import fields
from typing import Callable, Sequence, TypeVar

import torch
from torch import Tensor

Group = TypeVar("Group", bound="TensorGroup")


class TensorGroup:
    """Dataclass mixin: every field is a tensor, or another TensorGroup, and all share the leading dimension."""

    def map(self: Group, function: Callable[[Tensor], Tensor]) -> Group:
        """A new group with `function` applied to every tensor."""
        return type(self)(**{name: _map(value, function) for name, value in self._fields().items()})

    def to(self: Group, device: torch.device | str) -> Group:
        return self.map(lambda tensor: tensor.to(device))

    def cpu(self: Group) -> Group:
        return self.to("cpu")

    def __getitem__(self: Group, index) -> Group:
        return self.map(lambda tensor: tensor[index])

    def __len__(self) -> int:
        return len(next(iter(self._fields().values())))

    @classmethod
    def concatenate(cls: type[Group], parts: Sequence[Group], dim: int = 0) -> Group:
        return cls._combine(parts, lambda tensors: torch.cat(tensors, dim))

    @classmethod
    def stack(cls: type[Group], parts: Sequence[Group], dim: int = 0) -> Group:
        return cls._combine(parts, lambda tensors: torch.stack(tensors, dim))

    @classmethod
    def _combine(cls: type[Group], parts: Sequence[Group], combine: Callable[[list[Tensor]], Tensor]) -> Group:
        combined = {}
        for field in fields(cls):
            values = [getattr(part, field.name) for part in parts]
            is_nested = isinstance(values[0], TensorGroup)
            combined[field.name] = type(values[0])._combine(values, combine) if is_nested else combine(values)
        return cls(**combined)

    def _fields(self) -> dict:
        return {field.name: getattr(self, field.name) for field in fields(self)}


def _map(value, function: Callable[[Tensor], Tensor]):
    return value.map(function) if isinstance(value, TensorGroup) else function(value)

In [ ]:
%%writefile /kaggle/working/src/gptrec/training/checkpoint.py
"""Saving and restoring models together with the configuration needed to rebuild them."""

from __future__ import annotations

from dataclasses import asdict
from pathlib import Path

import torch

from gptrec.models.policy import NextItemPolicy, PolicyConfig
from gptrec.models.reward_model import RewardModel, RewardModelConfig
from gptrec.models.value_model import ContinuationValueModel


def save_model(model: NextItemPolicy | RewardModel, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"config": asdict(model.config), "weights": model.state_dict()}, path)


def load_policy(path: str | Path, device: torch.device | str = "cpu") -> NextItemPolicy:
    """The saved recommender, in evaluation mode."""
    checkpoint = _read(path)
    policy = NextItemPolicy(PolicyConfig.from_dict(checkpoint["config"]))
    policy.load_state_dict(checkpoint["weights"])
    return policy.to(device).eval()


def load_reward_model(path: str | Path, device: torch.device | str = "cpu") -> RewardModel:
    """The saved reward model, frozen and in evaluation mode."""
    checkpoint = _read(path)
    reward_model = RewardModel(RewardModelConfig.from_dict(checkpoint["config"]))
    reward_model.load_state_dict(checkpoint["weights"])
    return reward_model.to(device).eval().requires_grad_(False)


def _read(path: str | Path) -> dict:
    return torch.load(Path(path), map_location="cpu", weights_only=True)


def save_continuation_value_model(model: ContinuationValueModel, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"config": asdict(model.config), "horizon": model.horizon, "weights": model.state_dict()}, path)


def load_continuation_value_model(path: str | Path, device: torch.device | str = "cpu") -> ContinuationValueModel:
    """The saved value model of what follows each item of a list, frozen and in evaluation mode."""
    checkpoint = _read(path)
    model = ContinuationValueModel(PolicyConfig.from_dict(checkpoint["config"]), checkpoint["horizon"])
    model.load_state_dict(checkpoint["weights"])
    return model.to(device).eval().requires_grad_(False)

In [ ]:
%%writefile /kaggle/working/src/gptrec/training/off_policy.py
"""Off-policy REINFORCE on logged sessions, after Matveev et al. (arXiv:2607.02818).

Nothing is generated and no reward model is asked. The recommender re-reads what users really did, and the
log-probability of every logged item is raised in proportion to the reward that followed it in the log:

    loss = - sum over logged actions of  discount^t * weight_t * return_t * log pi(a_t | x_t)  /  sum of discount^t * weight_t

    return_t   discounted sum of the logged rewards from action t to the end of its session
    weight_t   min(cap, product of pi / pi_behaviour over the next `importance_weight_factors` actions of the session)
    t          how far into its session the action is

The behaviour policy is the frozen pre-trained recommender, as in the paper, where the target policy also
starts from the behaviour model. Two things differ from the paper. Every logged event is an action here;
the paper uses only the events a recommender served, which KION does not mark. And a session that
started before the training window is read from the window's first event.
"""

from __future__ import annotations

import copy
from dataclasses import dataclass, field
from functools import partial

import torch
import torch.nn.functional as F
from torch import Tensor

from gptrec.data.batches import EventBatch
from gptrec.data.log import InteractionLog
from gptrec.models.policy import NextItemPolicy
from gptrec.models.reward_model import Feedback
from gptrec.rollout import RewardFunction, RewardSpec
from gptrec.training.precision import seed_everything
from gptrec.training.supervised import Metrics, OptimizationConfig, fit_on_train_period, next_item_loss, predicts_target


@dataclass(frozen=True)
class LoggedReturnConfig:
    optimization: OptimizationConfig = field(default_factory=lambda: OptimizationConfig(epochs=1, learning_rate=1e-4))
    reward: RewardSpec = field(default_factory=RewardSpec)
    discount: float = 0.9
    importance_weight_factors: int = 10
    importance_weight_cap: float = 0.5


def finetune_on_logged_returns(
    pretrained_policy: NextItemPolicy,
    log: InteractionLog,
    config: LoggedReturnConfig,
    window_stride: int,
    device: torch.device,
) -> tuple[NextItemPolicy, list[Metrics]]:
    """Fine-tunes a copy of the recommender with off-policy REINFORCE; the original is the behaviour policy.

    The validation loss, which picks the best epoch, is the plain next-item loss.
    """
    reward = config.reward
    if reward.weigh_by_choice or reward.needs_logged_future or reward.needs_other_user or reward.depends_on_the_list:
        raise ValueError("a reward read from the log has one logged response per item: it cannot weigh by choice, by logged relevance, against another user or by the rest of a list")
    seed_everything(config.optimization.seed)
    behaviour_policy = pretrained_policy.to(device).eval().requires_grad_(False)
    policy = copy.deepcopy(behaviour_policy).requires_grad_(True)
    reward_function = RewardFunction(config.reward, log.items, device)
    loss = partial(logged_return_loss, behaviour_policy=behaviour_policy, reward_function=reward_function, config=config)
    window_length = policy.config.transformer.max_positions
    history = fit_on_train_period(policy, log, loss, predicts_target, window_length, window_stride, config.optimization, device, validation_loss=next_item_loss)
    return policy.eval(), history


def logged_return_loss(policy: NextItemPolicy, batch: EventBatch, behaviour_policy: NextItemPolicy, reward_function: RewardFunction, config: LoggedReturnConfig) -> Tensor:
    """The loss of the module docstring for one batch of windows. Action a is the window's event a + 1, chosen after events 0..a."""
    is_action = predicts_target(batch)
    stays_in_session = (batch.session_continues[:, 1:] > 0.5) & _ahead(is_action, 1)
    log_probs = _logged_item_log_probs(policy, batch)
    with torch.no_grad():
        log_policy_ratio = (log_probs - _logged_item_log_probs(behaviour_policy, batch)).masked_fill(~is_action, 0.0)
        importance_weight = _capped_importance_weights(log_policy_ratio, stays_in_session, config.importance_weight_factors, config.importance_weight_cap)
        returns = _session_returns(_logged_rewards(batch, reward_function)[:, 1:], stays_in_session, config.discount)
        action_weight = config.discount ** _steps_into_session(stays_in_session) * importance_weight * is_action
    return -(action_weight * returns * log_probs).sum() / action_weight.sum().clamp_min(1e-8)


def _logged_item_log_probs(policy: NextItemPolicy, batch: EventBatch) -> Tensor:
    """log pi(event a + 1 | events 0..a) for every action of the window. [B, L - 1]"""
    log_distributions = F.log_softmax(policy.item_logits(policy.hidden_states(batch.items)[:, :-1]), dim=-1)
    return log_distributions.gather(2, batch.items[:, 1:, None]).squeeze(2)


def _logged_rewards(batch: EventBatch, reward_function: RewardFunction) -> Tensor:
    """The reward of every logged event; a retention outcome the log does not know counts as absent. [B, L]"""
    logged = Feedback(
        played_ratio=batch.played_ratio,
        session_continues=batch.session_continues * batch.session_continues_known,
        returns=batch.returns * batch.returns_known,
        is_chosen=torch.ones_like(batch.played_ratio),
        relative_ratio=batch.relative_ratio,
        liked=batch.liked * batch.reactions_known,
        disliked=batch.disliked * batch.reactions_known,
        adopts=batch.adopts * batch.adopts_known,
    )
    return reward_function(logged, batch.items)


def _session_returns(rewards: Tensor, stays_in_session: Tensor, discount: float) -> Tensor:
    """returns[:, a] = rewards[:, a] + discount * returns[:, a + 1] while the session lasts. [B, A]"""
    returns = torch.zeros_like(rewards)
    return_from_next_action = torch.zeros_like(rewards[:, 0])
    for action in reversed(range(rewards.shape[1])):
        return_from_next_action = rewards[:, action] + discount * stays_in_session[:, action] * return_from_next_action
        returns[:, action] = return_from_next_action
    return returns


def _capped_importance_weights(log_policy_ratio: Tensor, stays_in_session: Tensor, num_factors: int, cap: float) -> Tensor:
    """min(cap, product of the policy ratios of an action and of the up to `num_factors` - 1 actions after it in its session). [B, A]"""
    log_weight = log_policy_ratio.clone()
    session_lasts = torch.ones_like(stays_in_session)
    for actions_ahead in range(1, num_factors):
        session_lasts = session_lasts & _ahead(stays_in_session, actions_ahead - 1)
        log_weight = log_weight + _ahead(log_policy_ratio, actions_ahead) * session_lasts
    return log_weight.exp().clamp(max=cap)


def _steps_into_session(stays_in_session: Tensor) -> Tensor:
    """0 for the first action of a session (or of the window), then 1, 2, ... [B, A]"""
    steps = torch.zeros(stays_in_session.shape, device=stays_in_session.device)
    for action in range(1, stays_in_session.shape[1]):
        steps[:, action] = (steps[:, action - 1] + 1) * stays_in_session[:, action - 1]
    return steps


def _ahead(values: Tensor, steps: int) -> Tensor:
    """values[:, a + steps] in column a; zero (False) where that is beyond the last column. [B, A]"""
    if steps == 0:
        return values
    return torch.cat([values[:, steps:], values.new_zeros(values.shape[0], min(steps, values.shape[1]))], dim=1)

In [ ]:
%%writefile /kaggle/working/src/gptrec/training/precision.py
"""Device, numeric precision and reproducibility helpers shared by every training and evaluation loop."""

from __future__ import annotations

import random
from typing import Iterable

import numpy as np
import torch
from torch import Tensor, nn

FIRST_GPU_GENERATION_WITH_BFLOAT16 = 8


def default_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def mixed_precision_dtype(device: torch.device) -> torch.dtype | None:
    """The reduced precision the device computes natively: bfloat16 on recent GPUs, float16 on older ones, none on a CPU."""
    if device.type != "cuda":
        return None
    gpu_generation, _ = torch.cuda.get_device_capability(device)
    return torch.bfloat16 if gpu_generation >= FIRST_GPU_GENERATION_WITH_BFLOAT16 else torch.float16


def autocast(device: torch.device, enabled: bool = True) -> torch.autocast:
    """Mixed-precision context for the device; a no-op on a CPU or when disabled."""
    dtype = mixed_precision_dtype(device)
    return torch.autocast(device_type=device.type, dtype=dtype, enabled=enabled and dtype is not None)


def gradient_scaler(device: torch.device, enabled: bool = True) -> torch.amp.GradScaler:
    """Loss scaling, which float16 needs to keep small gradients from underflowing; a pass-through at any other precision."""
    return torch.amp.GradScaler(device.type, enabled=enabled and mixed_precision_dtype(device) is torch.float16)


def clipped_optimizer_step(
    loss: Tensor,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    parameters: Iterable[nn.Parameter],
    max_grad_norm: float,
) -> Tensor:
    """Backpropagates `loss`, clips the gradient norm and updates the parameters. Returns the norm before clipping."""
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    gradient_norm = nn.utils.clip_grad_norm_(parameters, max_grad_norm)
    scaler.step(optimizer)
    scaler.update()
    return gradient_norm


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [ ]:
%%writefile /kaggle/working/src/gptrec/training/preference.py
"""Learning from logged responses alone: no reward model and no generated lists.

Both methods start from the base recommender, read the train period window by window like next-item training,
and keep the epoch with the lowest next-item loss on the validation period:

reward-weighted fine-tuning  next-item training in which each logged item weighs exp(r / temperature), r its logged
                             response. Under the anchored objective this is the closed-form optimum fitted to the
                             log (exponential reward-weighted SFT, Chidambaram et al. 2026; AWR, Peng et al. 2019).
                             Without a response every item weighs one: plain continued training, the control for
                             what the extra epoch at a lower learning rate does by itself.
within-user DPO              direct preference optimisation (Rafailov et al., 2023) on pairs of items that one user
                             consumed after the same history: one taken to more than is usual for it, one to less.
                             Neither item is preferred for having been started, since both were, nor for being short,
                             since each is compared with its own usual response: only this user's taste remains.
"""

from __future__ import annotations

import copy
from dataclasses import dataclass, field
from functools import partial

import torch
import torch.nn.functional as F
from torch import Tensor, nn

from gptrec.data.batches import EventBatch
from gptrec.data.log import PAD_ITEM, InteractionLog
from gptrec.models.policy import NextItemPolicy
from gptrec.models.sequence_recommender import SequenceRecommender
from gptrec.tensors import TensorGroup
from gptrec.training.precision import seed_everything
from gptrec.training.supervised import Metrics, OptimizationConfig, fit_on_train_period, next_item_loss, predicts_target

LOGGED_RESPONSES = ("relative_ratio", "played_ratio")
USUAL_QUANTILE = 0.5


@dataclass(frozen=True)
class RewardWeightedConfig:
    optimization: OptimizationConfig = field(default_factory=lambda: OptimizationConfig(epochs=1, learning_rate=1e-4))
    response: str | None = "relative_ratio"
    temperature: float = 0.3


@dataclass(frozen=True)
class LoggedPreferenceConfig:
    optimization: OptimizationConfig = field(default_factory=lambda: OptimizationConfig(epochs=1, learning_rate=1e-4))
    beta: float = 0.1
    margin: float = 0.1
    lookahead: int = 20


@dataclass
class PreferencePairs(TensorGroup):
    """Pairs of items one user consumed after the same moment of a window, preferred first."""

    window: Tensor
    moment: Tensor
    preferred: Tensor
    rejected: Tensor


def finetune_with_reward_weights(
    pretrained_policy: NextItemPolicy, log: InteractionLog, config: RewardWeightedConfig, window_stride: int, device: torch.device
) -> tuple[NextItemPolicy, list[Metrics]]:
    """Continues next-item training on a copy of the policy, each target weighted by exp(its logged response / temperature).

    With no response every target weighs one, and the same epoch is plain continued next-item training.
    """
    if config.response is not None and config.response not in LOGGED_RESPONSES:
        raise ValueError(f"unknown logged response {config.response!r}: expected one of {LOGGED_RESPONSES}, or None for equal weights")
    seed_everything(config.optimization.seed)
    policy = copy.deepcopy(pretrained_policy).to(device).requires_grad_(True)
    loss = partial(reward_weighted_loss, response=config.response, temperature=config.temperature)
    window_length = policy.config.transformer.max_positions
    history = fit_on_train_period(policy, log, loss, predicts_target, window_length, window_stride, config.optimization, device, validation_loss=next_item_loss)
    return policy.eval(), history


def reward_weighted_loss(model: SequenceRecommender, batch: EventBatch, response: str | None, temperature: float) -> Tensor:
    """Cross-entropy of every target event weighted by exp(response / temperature), the weights normalised to a mean of one per batch.

    Without a response every weight is one, which is the plain next-item loss.
    """
    positions = predicts_target(batch)
    states = model.hidden_states(batch.items)[:, :-1][positions]
    if len(states) == 0:
        return states.sum()
    cross_entropy = F.cross_entropy(model.item_logits(states), batch.items[:, 1:][positions], reduction="none")
    if response is None:
        return cross_entropy.mean()
    responses = getattr(batch, response)[:, 1:][positions].float()
    weights = torch.softmax(responses / temperature, dim=0) * len(responses)
    return (weights * cross_entropy).mean()


def finetune_on_logged_preferences(
    pretrained_policy: NextItemPolicy, log: InteractionLog, config: LoggedPreferenceConfig, window_stride: int, device: torch.device
) -> tuple[NextItemPolicy, list[Metrics]]:
    """DPO from the base recommender on within-user pairs (see `within_user_pairs`); the base recommender is the reference."""
    seed_everything(config.optimization.seed)
    reference = pretrained_policy.to(device).eval().requires_grad_(False)
    policy = _without_dropout(copy.deepcopy(reference)).requires_grad_(True)
    loss = partial(logged_preference_loss, reference=reference, config=config)
    window_length = policy.config.transformer.max_positions
    history = fit_on_train_period(policy, log, loss, predicts_target, window_length, window_stride, config.optimization, device, validation_loss=next_item_loss)
    return policy.eval(), history


def logged_preference_loss(policy: NextItemPolicy, batch: EventBatch, reference: NextItemPolicy, config: LoggedPreferenceConfig) -> Tensor:
    """The DPO loss over one pair per window; windows without a pair are left out."""
    pairs = within_user_pairs(batch, config.margin, config.lookahead)
    if len(pairs) == 0:
        return policy.item_embedding.weight.sum() * 0.0
    margin = _preference_margin(policy, batch, pairs)
    with torch.no_grad():
        reference_margin = _preference_margin(reference, batch, pairs)
    return -F.logsigmoid(config.beta * (margin - reference_margin)).mean()


def within_user_pairs(batch: EventBatch, margin: float, lookahead: int) -> PreferencePairs:
    """At most one pair per window: a random moment, then one event above and one below the usual among the next `lookahead` events.

    Only target events form pairs, so the context of a held-out window never does.
    """
    positions = torch.arange(batch.items.shape[1], device=batch.items.device)
    comes_after = (positions[None, :] > positions[:, None]) & (positions[None, :] <= positions[:, None] + lookahead)
    above = batch.is_target & (batch.relative_ratio >= USUAL_QUANTILE + margin)
    below = batch.is_target & (batch.relative_ratio <= USUAL_QUANTILE - margin)
    above_after = above[:, None, :] & comes_after[None]
    below_after = below[:, None, :] & comes_after[None]
    has_pair = above_after.any(dim=2) & below_after.any(dim=2) & (batch.items != PAD_ITEM)
    window = has_pair.any(dim=1).nonzero().squeeze(1)
    if len(window) == 0:
        nothing = torch.zeros(0, dtype=torch.long, device=batch.items.device)
        return PreferencePairs(nothing, nothing, nothing, nothing)
    moment = torch.multinomial(has_pair[window].float(), num_samples=1).squeeze(1)
    preferred = torch.multinomial(above_after[window, moment].float(), num_samples=1).squeeze(1)
    rejected = torch.multinomial(below_after[window, moment].float(), num_samples=1).squeeze(1)
    return PreferencePairs(window, moment, batch.items[window, preferred], batch.items[window, rejected])


def _preference_margin(model: NextItemPolicy, batch: EventBatch, pairs: PreferencePairs) -> Tensor:
    """log pi(preferred | history) - log pi(rejected | history) for every pair. [P]"""
    rows = torch.arange(len(pairs), device=pairs.window.device)
    states = model.hidden_states(batch.items[pairs.window])[rows, pairs.moment]
    log_probs = F.log_softmax(model.item_logits(states), dim=-1)
    return log_probs[rows, pairs.preferred] - log_probs[rows, pairs.rejected]


def _without_dropout(model: nn.Module) -> nn.Module:
    """`model` with every dropout switched off, so that its log-probabilities and the reference's differ only by what it learned."""
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.p = 0.0
    return model

In [ ]:
%%writefile /kaggle/working/src/gptrec/training/rl.py
"""RL fine-tuning of the recommender against a reward model, with PPO.

One iteration:
  1. sample recommendation lists from the current policy, for random moments of train-period histories;
  2. let the reward model predict the user's response to every item of every list: the rewards;
  3. estimate advantages - how much better each item did than its state promised;
  4. take several gradient steps on PPO's clipped objective, reusing the sampled lists.

A value model (the critic) learns what return a state promises and supplies the baseline of step 3
through generalised advantage estimation. A KL penalty towards the pre-trained policy keeps the
fine-tuned one close to observed behaviour. Unless the reward itself asks for relevance (see
`RewardSpec`), that penalty is the only thing that does, and its weight is the main knob of the
relevance / reward trade-off.
"""

from __future__ import annotations

import math
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable

import torch
import torch.nn.functional as F
from torch import Tensor
from tqdm.auto import tqdm

from gptrec.data.batches import EvalExamples, HistoryBatch, TrainingHistorySampler
from gptrec.data.log import InteractionLog
from gptrec.evaluation.relevance import contains
from gptrec.models.adapters import Adaptation, adaptable_copy, merge_low_rank_updates
from gptrec.models.policy import NextItemPolicy
from gptrec.models.reward_model import RewardModel
from gptrec.models.value_model import ValueModel
from gptrec.rollout import (
    GeneratedActions,
    GenerationConfig,
    RewardFunction,
    RewardSpec,
    SimulatedFeedback,
    action_log_distributions,
    generate_actions,
    histories_of_other_users,
    simulate_feedback,
    state_values,
)
from gptrec.training.precision import autocast, clipped_optimizer_step, gradient_scaler, seed_everything

Metrics = dict[str, float]

PROGRESS_REFRESH_SECONDS = 5.0
ANCHOR_WEIGHT_RANGE = (1e-4, 1e3)
MAX_RELATIVE_KL_EXCESS = 5.0
GROUP_STD_FLOOR = 1e-6


class AdvantageEstimator(str, Enum):
    """Where the baseline comes from that turns returns into advantages."""

    GAE = "gae"
    LEAVE_ONE_OUT = "leave_one_out"
    GROUP_RELATIVE = "group_relative"


@dataclass(frozen=True)
class RLConfig:
    num_steps: int = 2000
    histories_per_step: int = 512
    rollouts_per_history: int = 1
    history_len: int = 118
    min_history: int = 8
    max_future: int = 20
    generation: GenerationConfig = field(default_factory=GenerationConfig)
    simulated_feedback: SimulatedFeedback = SimulatedFeedback.EXPECTED
    reward: RewardSpec = field(default_factory=RewardSpec)

    discount: float = 0.99
    advantage_estimator: AdvantageEstimator = AdvantageEstimator.GAE
    gae_lambda: float = 0.9
    normalize_advantages: bool = True

    ppo_epochs: int = 4
    ppo_clip: float = 0.2
    reference_kl_weight: float = 0.3
    target_kl: float | None = None
    anchor_adaptation_rate: float = 0.03
    entropy_weight: float = 0.0
    adaptation: Adaptation = Adaptation.FULL
    low_rank: int = 1

    learning_rate: float = 1e-5
    value_learning_rate: float = 1e-4
    value_warmup_steps: int = 50
    max_grad_norm: float = 1.0
    evaluate_every: int = 200
    mixed_precision: bool = False
    seed: int = 0


def finetune_with_rl(
    pretrained_policy: NextItemPolicy,
    reward_model: RewardModel,
    log: InteractionLog,
    config: RLConfig,
    device: torch.device,
    evaluate: Callable[[NextItemPolicy], Metrics] | None = None,
) -> tuple[NextItemPolicy, list[Metrics]]:
    """Fine-tunes a copy of `pretrained_policy`; the original stays frozen as the KL reference.

    Returns the fine-tuned policy and one metrics record per iteration. Every `evaluate_every` iterations
    (and before the first one) the record also holds `evaluate(policy)` under "validation/" keys.
    """
    seed_everything(config.seed)
    reference = pretrained_policy.to(device).eval().requires_grad_(False)
    policy = adaptable_copy(reference, config.adaptation, config.low_rank).to(device).eval()
    trainable = [parameter for parameter in policy.parameters() if parameter.requires_grad]
    reward_model = reward_model.to(device).eval()
    reward_function = RewardFunction(config.reward, log.items, device, item_embeddings=reference.item_embedding.weight)
    moments = TrainingHistorySampler(log, config.history_len, config.max_future, config.min_history, seed=config.seed)
    optimizer = torch.optim.AdamW(trainable, lr=config.learning_rate, weight_decay=0.0)
    scaler = gradient_scaler(device, config.mixed_precision)
    uses_critic = config.advantage_estimator is AdvantageEstimator.GAE
    critic = _Critic(reference, config, device) if uses_critic else None
    anchor = _AnchorWeight(config)
    sampling_statistics = _SamplingStatistics(log, device)

    def validation_metrics() -> Metrics:
        return {f"validation/{name}": value for name, value in evaluate(policy).items()} if evaluate else {}

    records: list[Metrics] = [{"step": 0} | validation_metrics()]
    for step in tqdm(range(1, config.num_steps + 1), desc="RL fine-tuning", mininterval=PROGRESS_REFRESH_SECONDS):
        sampled = moments.sample(config.histories_per_step).to(device).map(lambda tensor: tensor.repeat_interleave(config.rollouts_per_history, dim=0))
        history = sampled.history
        with torch.no_grad(), autocast(device, config.mixed_precision):
            generated = generate_actions(policy, history, config.generation)
            feedback = simulate_feedback(reward_model, history, generated.actions, config.simulated_feedback)
            feedback_for_other_user = None
            if config.reward.needs_other_user:
                other_users = histories_of_other_users(history, shift=config.rollouts_per_history)
                feedback_for_other_user = simulate_feedback(reward_model, other_users, generated.actions, config.simulated_feedback)
            rewards = reward_function(feedback, generated.actions, sampled.future_items, feedback_for_other_user)
            returns = discounted_returns(rewards, config.discount)
            if critic is not None:
                advantages = generalised_advantages(rewards, critic.values(history, generated.actions), config.discount, config.gae_lambda)
            elif config.advantage_estimator is AdvantageEstimator.GROUP_RELATIVE:
                advantages = group_relative_advantages(returns, config.rollouts_per_history)
            else:
                advantages = leave_one_out_advantages(returns, config.rollouts_per_history)
            advantage_scale = advantages.std().item()
            if config.normalize_advantages:
                advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
            reference_log_distributions = action_log_distributions(reference, history, generated.actions, config.generation)

        record: Metrics = {"step": step, "reward": rewards.mean().item(), "advantage_scale": advantage_scale}
        record |= {f"predicted_{signal}": values.mean().item() for signal, values in vars(feedback).items()}
        if feedback_for_other_user is not None:
            record["predicted_personal_played_ratio"] = (feedback.played_ratio - feedback_for_other_user.played_ratio).mean().item()
        record |= sampling_statistics(sampled, generated)
        policy_is_learning = critic is None or step > config.value_warmup_steps
        for _ in range(config.ppo_epochs):
            if policy_is_learning:
                with autocast(device, config.mixed_precision):
                    log_distributions = action_log_distributions(policy, history, generated.actions, config.generation)
                    loss, loss_terms = policy_improvement_loss(log_distributions, reference_log_distributions, generated, advantages, config, anchor.weight)
                gradient_norm = clipped_optimizer_step(loss, optimizer, scaler, trainable, config.max_grad_norm)
                record |= loss_terms | {"gradient_norm": gradient_norm.item(), "reference_kl_weight": anchor.weight}
            if critic is not None:
                record |= critic.learn(history, generated.actions, returns)
        if policy_is_learning:
            anchor.adapt(record["kl_to_reference"])

        if step % config.evaluate_every == 0 or step == config.num_steps:
            record |= validation_metrics()
        records.append(record)

    return merge_low_rank_updates(policy).eval(), records


def discounted_returns(rewards: Tensor, discount: float) -> Tensor:
    """returns[:, k] = rewards[:, k] + discount * rewards[:, k + 1] + ...  for rewards of shape [B, K]."""
    returns = torch.zeros_like(rewards)
    return_from_next_step = torch.zeros_like(rewards[:, 0])
    for step in reversed(range(rewards.shape[1])):
        return_from_next_step = rewards[:, step] + discount * return_from_next_step
        returns[:, step] = return_from_next_step
    return returns


def generalised_advantages(rewards: Tensor, values: Tensor, discount: float, gae_lambda: float) -> Tensor:
    """Generalised advantage estimation (Schulman et al., 2016) for lists that end after their last item.

    rewards, values: [B, K]; values[:, k] is the value of the state in which item k was chosen.
    advantage[:, k] = sum over l of (discount * gae_lambda)^l * residual[:, k + l], where
    residual[:, k] = rewards[:, k] + discount * values[:, k + 1] - values[:, k] and the value after the last item is 0.
    """
    value_of_next_state = torch.cat([values[:, 1:], torch.zeros_like(values[:, :1])], dim=1)
    residuals = rewards + discount * value_of_next_state - values
    advantages = torch.zeros_like(residuals)
    advantage_of_next_step = torch.zeros_like(residuals[:, 0])
    for step in reversed(range(residuals.shape[1])):
        advantage_of_next_step = residuals[:, step] + discount * gae_lambda * advantage_of_next_step
        advantages[:, step] = advantage_of_next_step
    return advantages


def leave_one_out_advantages(returns: Tensor, rollouts_per_history: int) -> Tensor:
    """Each return minus the mean return, at the same step, of the other rollouts that share its history.

    The baseline of a rollout never depends on that rollout's own actions, so the policy gradient stays
    unbiased. With one rollout per history, all the other rollouts of the batch serve as the baseline.

    returns: [B * rollouts_per_history, K] with the rollouts of one history in consecutive rows.
    """
    horizon = returns.shape[1]
    grouped = returns.reshape(-1, rollouts_per_history, horizon) if rollouts_per_history > 1 else returns.reshape(1, -1, horizon)
    group_size = grouped.shape[1]
    if group_size < 2:
        raise ValueError("a leave-one-out baseline needs at least two rollouts")
    mean_of_others = (grouped.sum(dim=1, keepdim=True) - grouped) / (group_size - 1)
    return (grouped - mean_of_others).reshape(returns.shape)


def group_relative_advantages(returns: Tensor, rollouts_per_history: int) -> Tensor:
    """GRPO's advantages (Shao et al., 2024): each return minus the mean of its group, over the group's standard deviation, step by step.

    A group is the rollouts of one history. Every history therefore weighs the same however much its returns vary,
    and one whose lists all earn the same return contributes nothing. returns: as for `leave_one_out_advantages`.
    """
    if rollouts_per_history < 2:
        raise ValueError("group-relative advantages need at least two rollouts per history")
    grouped = returns.reshape(-1, rollouts_per_history, returns.shape[1])
    centred = grouped - grouped.mean(dim=1, keepdim=True)
    return (centred / (grouped.std(dim=1, keepdim=True) + GROUP_STD_FLOOR)).reshape(returns.shape)


def policy_improvement_loss(
    log_distributions: Tensor,
    reference_log_distributions: Tensor,
    generated: GeneratedActions,
    advantages: Tensor,
    config: RLConfig,
    reference_kl_weight: float,
) -> tuple[Tensor, Metrics]:
    """PPO's clipped surrogate, plus the KL penalty towards the reference policy and the entropy bonus.

    log_distributions, reference_log_distributions: [B, K, V+1]; advantages: [B, K]. The weight of the KL
    penalty is passed separately from `config` because a KL budget changes it between iterations.
    """
    action_log_probs = log_distributions.gather(2, generated.actions[:, :, None]).squeeze(2)
    probability_ratio = (action_log_probs - generated.log_probs).exp()
    clipped_ratio = probability_ratio.clamp(1.0 - config.ppo_clip, 1.0 + config.ppo_clip)
    surrogate = torch.minimum(probability_ratio * advantages, clipped_ratio * advantages).mean()

    probabilities = log_distributions.exp()
    kl_to_reference = (probabilities * (log_distributions - reference_log_distributions)).sum(dim=-1).mean()
    entropy = -(probabilities * log_distributions).sum(dim=-1).mean()

    loss = -surrogate + reference_kl_weight * kl_to_reference - config.entropy_weight * entropy
    was_clipped = (probability_ratio - 1.0).abs() > config.ppo_clip
    return loss, {
        "loss": loss.item(),
        "kl_to_reference": kl_to_reference.item(),
        "entropy": entropy.item(),
        "clip_fraction": was_clipped.float().mean().item(),
    }


class _AnchorWeight:
    """The weight of the KL penalty towards the pre-trained policy: fixed, or adapted to keep the KL at a budget."""

    def __init__(self, config: RLConfig) -> None:
        self.weight = config.reference_kl_weight
        self._target_kl = config.target_kl
        self._adaptation_rate = config.anchor_adaptation_rate

    def adapt(self, kl_to_reference: float) -> None:
        """Tightens the anchor when the policy is further from the reference than the budget allows, loosens it when nearer.

        The change is multiplicative and proportional to how far off the KL is, relative to the budget. Without a
        budget the weight never changes.
        """
        if self._target_kl is None:
            return
        lowest, highest = ANCHOR_WEIGHT_RANGE
        relative_excess = min(kl_to_reference / self._target_kl - 1.0, MAX_RELATIVE_KL_EXCESS)
        self.weight = min(highest, max(lowest, self.weight * math.exp(self._adaptation_rate * relative_excess)))


class _Critic:
    """PPO's value model together with its optimiser. It starts from the pre-trained recommender's transformer."""

    def __init__(self, pretrained_policy: NextItemPolicy, config: RLConfig, device: torch.device) -> None:
        self._config = config
        self._device = device
        self._value_model = ValueModel.from_policy(pretrained_policy).to(device).eval()
        self._optimizer = torch.optim.AdamW(self._value_model.parameters(), lr=config.value_learning_rate, weight_decay=0.0)
        self._scaler = gradient_scaler(device, config.mixed_precision)

    @torch.no_grad()
    def values(self, history: HistoryBatch, actions: Tensor) -> Tensor:
        """Expected return from the state in which each item of the list was chosen. [B, K]"""
        return state_values(self._value_model, history, actions)

    def learn(self, history: HistoryBatch, actions: Tensor, returns: Tensor) -> Metrics:
        """One gradient step towards the returns that were actually collected."""
        with autocast(self._device, self._config.mixed_precision):
            values = state_values(self._value_model, history, actions)
            loss = F.mse_loss(values, returns)
        clipped_optimizer_step(loss, self._optimizer, self._scaler, self._value_model.parameters(), self._config.max_grad_norm)
        unexplained_variance = (returns - values.detach()).var() / returns.var().clamp_min(1e-8)
        return {"value_loss": loss.item(), "value_explained_variance": 1.0 - unexplained_variance.item()}


class _SamplingStatistics:
    """Cheap per-iteration indicators of what the policy currently samples, to watch for collapse while training."""

    def __init__(self, log: InteractionLog, device: torch.device) -> None:
        self._popularity = torch.as_tensor(log.items.train_popularity, dtype=torch.float32, device=device)
        self._duration = torch.as_tensor(log.items.duration_seconds, dtype=torch.float32, device=device)

    def __call__(self, sampled: EvalExamples, generated: GeneratedActions) -> Metrics:
        actions = generated.actions
        return {
            "distinct_action_share": actions.unique().numel() / actions.numel(),
            "logged_hit_rate": contains(actions, sampled.future_items).float().mean().item(),
            "history_repeat_rate": contains(actions, sampled.history.items).float().mean().item(),
            "mean_action_popularity": self._popularity[actions].mean().item(),
            "mean_action_duration_seconds": self._duration[actions].mean().item(),
        }

In [ ]:
%%writefile /kaggle/working/src/gptrec/training/supervised.py
"""Supervised training: next-item prediction for the recommender, feedback prediction for the reward model."""

from __future__ import annotations

import copy
import math
from dataclasses import dataclass, field
from functools import partial
from typing import Callable

import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor, nn
from tqdm.auto import tqdm

from gptrec.data.batches import EventBatch, EventWindows, swap_one_target_item
from gptrec.data.log import PAD_ITEM, InteractionLog
from gptrec.evaluation.reward_model_metrics import choice_quality, predict_heldout_feedback, prediction_quality, retention_quality
from gptrec.models.backbone import TransformerConfig
from gptrec.models.policy import NextItemPolicy, PolicyConfig
from gptrec.models.reward_model import UNKNOWN_FEEDBACK, RewardModel, RewardModelConfig
from gptrec.models.sequence_recommender import SequenceRecommender
from gptrec.training.precision import autocast, clipped_optimizer_step, gradient_scaler, seed_everything

LossFunction = Callable[[nn.Module, EventBatch], Tensor]
Validation = Callable[[nn.Module], dict[str, float]]
Metrics = dict[str, float]

PROGRESS_REFRESH_SECONDS = 5.0


@dataclass(frozen=True)
class OptimizationConfig:
    epochs: int = 10
    batch_size: int = 128
    learning_rate: float = 1e-3
    weight_decay: float = 0.01
    warmup_steps: int = 500
    max_grad_norm: float = 1.0
    early_stopping_patience: int = 2
    mixed_precision: bool = True
    seed: int = 0


@dataclass(frozen=True)
class PolicyTrainingConfig:
    """Training windows are `transformer.max_positions` events long, so every position the model can read gets trained."""

    transformer: TransformerConfig = field(default_factory=TransformerConfig)
    optimization: OptimizationConfig = field(default_factory=OptimizationConfig)
    window_stride: int = 64


@dataclass(frozen=True)
class FeedbackLossWeights:
    """Weight of each feedback signal in the reward model's loss."""

    played_ratio: float = 1.0
    session_continues: float = 0.25
    returns: float = 0.25
    is_chosen: float = 0.25
    relative_ratio: float = 0.5
    liked: float = 0.0
    disliked: float = 0.0
    adopts: float = 0.0


@dataclass(frozen=True)
class RewardModelTrainingConfig:
    transformer: TransformerConfig = field(default_factory=TransformerConfig)
    optimization: OptimizationConfig = field(default_factory=OptimizationConfig)
    window_stride: int = 64
    num_ratio_buckets: int = 10
    loss_weights: FeedbackLossWeights = field(default_factory=FeedbackLossWeights)
    hidden_response_share: float = 0.0
    max_hidden_responses: int = 10


def train_policy(log: InteractionLog, config: PolicyTrainingConfig, device: torch.device, extra_validation: Validation | None = None) -> tuple[NextItemPolicy, list[Metrics]]:
    """Trains the recommender on the train period; the validation period decides when to stop.

    `extra_validation` adds metrics (e.g. NDCG@10) to the per-epoch history without affecting training.
    """
    seed_everything(config.optimization.seed)
    policy = NextItemPolicy(PolicyConfig(log.num_items, config.transformer)).to(device)
    window_length = config.transformer.max_positions
    history = fit_on_train_period(
        policy, log, next_item_loss, predicts_target, window_length, config.window_stride, config.optimization, device, extra_validation=extra_validation
    )
    return policy.eval(), history


def finetune_on_liked_items(
    pretrained_policy: NextItemPolicy,
    log: InteractionLog,
    config: OptimizationConfig,
    window_stride: int,
    liked_threshold: float,
    device: torch.device,
) -> tuple[NextItemPolicy, list[Metrics]]:
    """Continues next-item training on a copy of the policy, but only towards items the user played at least `liked_threshold` of.

    The supervised alternative to RL: it learns from the logged feedback directly and needs no reward model.
    """
    seed_everything(config.seed)
    policy = copy.deepcopy(pretrained_policy).to(device).requires_grad_(True)
    loss = partial(liked_next_item_loss, liked_threshold=liked_threshold)
    scored_positions = partial(predicts_liked_target, liked_threshold=liked_threshold)
    window_length = policy.config.transformer.max_positions
    history = fit_on_train_period(policy, log, loss, scored_positions, window_length, window_stride, config, device)
    return policy.eval(), history


def train_reward_model(log: InteractionLog, config: RewardModelTrainingConfig, device: torch.device, users: np.ndarray | None = None) -> tuple[RewardModel, list[Metrics]]:
    """Trains a reward model on the train period of `users` (default: everyone)."""
    seed_everything(config.optimization.seed)
    learns_unknown_responses = config.hidden_response_share > 0
    reward_model = RewardModel(RewardModelConfig(log.num_items, config.transformer, config.num_ratio_buckets, learns_unknown_responses)).to(device)
    loss = partial(feedback_loss, weights=config.loss_weights, hidden_response_share=config.hidden_response_share, max_hidden_responses=config.max_hidden_responses)
    validation_loss = partial(feedback_loss, weights=config.loss_weights)
    window_length = config.transformer.max_positions
    validation_windows = EventWindows.for_heldout(log, "validation", window_length, users)

    def prediction_quality_on_validation(trained: nn.Module) -> Metrics:
        """The validation loss in readable terms: how well each signal of validation-period events is predicted."""
        predicted = predict_heldout_feedback(trained, validation_windows, config.optimization.batch_size, device, config.optimization.mixed_precision)
        played_ratio = prediction_quality(predicted.played_ratio, predicted.predicted_played_ratio)
        return {
            "validation_auc": played_ratio["auc"],
            "validation_mae": played_ratio["mae"],
            "validation_relative_auc": prediction_quality(predicted.relative_ratio, predicted.predicted_relative_ratio)["auc"],
            "validation_returns_auc": retention_quality(predicted).loc["returns", "auc"],
            "validation_choice_auc": choice_quality(predicted)["auc"],
        }

    history = fit_on_train_period(
        reward_model,
        log,
        loss,
        lambda batch: batch.is_target,
        window_length,
        config.window_stride,
        config.optimization,
        device,
        validation_loss=validation_loss,
        users=users,
        extra_validation=prediction_quality_on_validation,
    )
    return reward_model.eval().requires_grad_(False), history


def fit_on_train_period(
    model: nn.Module,
    log: InteractionLog,
    loss_function: LossFunction,
    scored_positions: Callable[[EventBatch], Tensor],
    window_length: int,
    window_stride: int,
    config: OptimizationConfig,
    device: torch.device,
    validation_loss: LossFunction | None = None,
    users: np.ndarray | None = None,
    extra_validation: Validation | None = None,
) -> list[Metrics]:
    """Trains `model` on windows of the train period and keeps the weights with the lowest validation-period loss.

    `scored_positions` marks, for a batch, the positions the validation loss averages over, so that batches
    with different numbers of targets are weighted correctly. The validation loss defaults to the training loss.
    `extra_validation` adds readable quality metrics to every epoch's record; they do not affect training.
    """
    train_windows = EventWindows.for_training(log, window_length, window_stride, users)
    validation_windows = EventWindows.for_heldout(log, "validation", window_length, users)

    def validate(trained: nn.Module) -> Metrics:
        loss = mean_heldout_loss(trained, validation_windows, validation_loss or loss_function, scored_positions, config, device)
        return {"validation_loss": loss} | (extra_validation(trained) if extra_validation else {})

    return fit(model, train_windows, loss_function, validate, config, device)


def next_item_loss(model: SequenceRecommender, batch: EventBatch) -> Tensor:
    """Cross-entropy of every target event given the events before it in its window."""
    return _next_item_cross_entropy(model, batch, predicts_target(batch))


def liked_next_item_loss(model: SequenceRecommender, batch: EventBatch, liked_threshold: float) -> Tensor:
    """Cross-entropy of the target events the user played at least `liked_threshold` of; skipped events are not imitated."""
    return _next_item_cross_entropy(model, batch, predicts_liked_target(batch, liked_threshold))


def predicts_target(batch: EventBatch) -> Tensor:
    """bool [B, L - 1]: positions whose next event is a target. The first event of a window has no context, so it is never one."""
    return batch.is_target[:, 1:]


def predicts_liked_target(batch: EventBatch, liked_threshold: float) -> Tensor:
    """bool [B, L - 1]: positions whose next event is a target that the user played at least `liked_threshold` of."""
    return batch.is_target[:, 1:] & (batch.played_ratio[:, 1:] >= liked_threshold)


def _next_item_cross_entropy(model: SequenceRecommender, batch: EventBatch, positions: Tensor) -> Tensor:
    """Full-softmax cross-entropy of the next item at the given positions (bool [B, L - 1])."""
    states = model.hidden_states(batch.items)[:, :-1][positions]
    if len(states) == 0:
        return states.sum()
    return F.cross_entropy(model.item_logits(states), batch.items[:, 1:][positions])


def feedback_loss(
    reward_model: RewardModel, batch: EventBatch, weights: FeedbackLossWeights, hidden_response_share: float = 0.0, max_hidden_responses: int = 1
) -> Tensor:
    """Weighted binary cross-entropy of every feedback signal on target events.

    The played ratio and its quantile inside the item are soft labels in [0, 1]. The retention signals, likes,
    dislikes and adoption are learned only from events whose outcome the log knows. Whether an item is chosen
    is learned by contrast, see `choice_loss`. With `hidden_response_share` the model reads some responses as
    unknown (see `hide_last_responses`); the labels it is trained on are always the observed ones.
    """
    played_ratio_read = hide_last_responses(batch, hidden_response_share, max_hidden_responses) if hidden_response_share > 0 else batch.played_ratio
    logits = reward_model(batch.items, played_ratio_read)
    reactions_known = batch.is_target & batch.reactions_known
    return (
        weights.played_ratio * _mean_binary_cross_entropy(logits.played_ratio, batch.played_ratio, batch.is_target)
        + weights.session_continues * _mean_binary_cross_entropy(logits.session_continues, batch.session_continues, batch.is_target & batch.session_continues_known)
        + weights.returns * _mean_binary_cross_entropy(logits.returns, batch.returns, batch.is_target & batch.returns_known)
        + weights.is_chosen * choice_loss(reward_model, batch, logits.is_chosen, played_ratio_read)
        + weights.relative_ratio * _mean_binary_cross_entropy(logits.relative_ratio, batch.relative_ratio, batch.is_target)
        + weights.liked * _mean_binary_cross_entropy(logits.liked, batch.liked, reactions_known)
        + weights.disliked * _mean_binary_cross_entropy(logits.disliked, batch.disliked, reactions_known)
        + weights.adopts * _mean_binary_cross_entropy(logits.adopts, batch.adopts, batch.is_target & batch.adopts_known)
    )


def hide_last_responses(batch: EventBatch, share: float, max_hidden: int) -> Tensor:
    """The played ratios a reward model reads, with the responses to the last 1..max_hidden events of a random `share` of windows unknown. [B, L]

    This is the situation of a generated list: the history's responses are known, the generated items' are not.
    The first event of a window always keeps its response.
    """
    num_events = (batch.items != PAD_ITEM).sum(dim=1)
    num_hidden = torch.randint(1, max_hidden + 1, num_events.shape, device=num_events.device)
    first_hidden = (num_events - num_hidden).clamp_min(1)
    is_hiding = torch.rand(num_events.shape, device=num_events.device) < share
    position = torch.arange(batch.items.shape[1], device=num_events.device)
    hidden = is_hiding[:, None] & (position[None, :] >= first_hidden[:, None]) & (position[None, :] < num_events[:, None])
    return batch.played_ratio.masked_fill(hidden, UNKNOWN_FEEDBACK)


def choice_loss(reward_model: RewardModel, batch: EventBatch, logits_of_logged_items: Tensor, played_ratio_read: Tensor | None = None) -> Tensor:
    """Binary cross-entropy of telling the item a user chose from an item they did not; the two kinds weigh the same.

    Logged target events are the chosen items. For the others, one target event per window is scored again
    with another item of the batch in its place (see `swap_one_target_item`). Such replacements are as popular
    as real choices, so the head has to use the history, not popularity. logits_of_logged_items: [B, L].
    played_ratio_read: the responses the model reads, if not the observed ones.
    """
    swapped = swap_one_target_item(batch)
    played_ratio_read = batch.played_ratio if played_ratio_read is None else played_ratio_read
    logits_of_swapped_items = reward_model(swapped.items, played_ratio_read).is_chosen.gather(1, swapped.position[:, None]).squeeze(1)
    chosen_loss = _mean_binary_cross_entropy(logits_of_logged_items, torch.ones_like(logits_of_logged_items), batch.is_target)
    not_chosen_loss = _mean_binary_cross_entropy(logits_of_swapped_items, torch.zeros_like(logits_of_swapped_items), swapped.is_different)
    return 0.5 * (chosen_loss + not_chosen_loss)


def _mean_binary_cross_entropy(logits: Tensor, labels: Tensor, is_selected: Tensor) -> Tensor:
    """Mean over the selected entries; a zero that still backpropagates when nothing is selected."""
    if not is_selected.any():
        return logits.sum() * 0.0
    return F.binary_cross_entropy_with_logits(logits[is_selected], labels[is_selected])


def fit(model: nn.Module, train_windows: EventWindows, loss_function: LossFunction, validate: Validation, config: OptimizationConfig, device: torch.device) -> list[Metrics]:
    """Minimises `loss_function` with AdamW and returns per-epoch metrics.

    `validate` must report "validation_loss". The model ends with the weights of its best epoch.
    """
    steps_per_epoch = train_windows.num_batches(config.batch_size, drop_last=True)
    if steps_per_epoch == 0:
        raise ValueError(f"{len(train_windows)} training windows do not fill one batch of {config.batch_size}")
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _warmup_then_cosine(config.warmup_steps, config.epochs * steps_per_epoch))
    scaler = gradient_scaler(device, config.mixed_precision)

    history: list[Metrics] = []
    best_loss, best_weights, epochs_without_improvement = math.inf, copy.deepcopy(model.state_dict()), 0
    for epoch in range(1, config.epochs + 1):
        model.train()
        loss_sum = 0.0
        batches = train_windows.batches(config.batch_size, shuffle_seed=config.seed + epoch, drop_last=True)
        progress = tqdm(batches, total=steps_per_epoch, desc=f"epoch {epoch}", mininterval=PROGRESS_REFRESH_SECONDS)
        for step, batch in enumerate(progress, start=1):
            with autocast(device, config.mixed_precision):
                loss = loss_function(model, batch.to(device))
            clipped_optimizer_step(loss, optimizer, scaler, model.parameters(), config.max_grad_norm)
            scheduler.step()
            loss_sum += loss.item()
            progress.set_postfix(train_loss=loss_sum / step)

        metrics = {"epoch": epoch, "train_loss": loss_sum / steps_per_epoch} | validate(model)
        history.append(metrics)
        print({name: round(value, 5) for name, value in metrics.items()})
        if not math.isfinite(metrics["validation_loss"]):
            raise FloatingPointError(f"validation loss is {metrics['validation_loss']} after epoch {epoch}: training diverged")

        if metrics["validation_loss"] < best_loss:
            best_loss, best_weights, epochs_without_improvement = metrics["validation_loss"], copy.deepcopy(model.state_dict()), 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= config.early_stopping_patience:
                break

    model.load_state_dict(best_weights)
    return history


def _warmup_then_cosine(warmup_steps: int, total_steps: int) -> Callable[[int], float]:
    def learning_rate_factor(step: int) -> float:
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    return learning_rate_factor


@torch.no_grad()
def mean_heldout_loss(model: nn.Module, windows: EventWindows, loss_function: LossFunction, scored_positions: Callable[[EventBatch], Tensor], config: OptimizationConfig, device: torch.device) -> float:
    """Average over all windows of a loss that is a mean over each batch's scored positions."""
    model.eval()
    loss_sum, num_targets = 0.0, 0
    for batch in windows.batches(config.batch_size):
        batch = batch.to(device)
        batch_targets = int(scored_positions(batch).sum())
        if batch_targets == 0:
            continue
        with autocast(device, config.mixed_precision):
            loss_sum += loss_function(model, batch).item() * batch_targets
        num_targets += batch_targets
    return loss_sum / max(1, num_targets)

## Setup

In [ ]:
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

MAX_MOUNT_DEPTH = 4
manifests = [
    manifest_path
    for depth in range(1, MAX_MOUNT_DEPTH + 1)
    for manifest_path in sorted(Path("/kaggle/input").glob("*/" * depth + "gptrec_bundle/manifest.json"))
]
assert manifests, "gptrec_bundle not found under /kaggle/input"
BUNDLE_DIR = manifests[0].parent
manifest = json.loads((BUNDLE_DIR / "manifest.json").read_text())
print("bundle:", BUNDLE_DIR)
print("side data:", {dataset: sorted(paths) for dataset, paths in manifest.get("side_data", {}).items()})

WORKING = Path("/kaggle/working")
os.environ.update(GPTREC_HOME=str(WORKING), HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
sys.path.insert(0, str(SOURCE_DIR))

missing = [package for package in ['torch', 'transformers', 'polars', 'pandas', 'numpy', 'huggingface_hub', 'tqdm', 'matplotlib'] if importlib.util.find_spec(package) is None]
if missing:
    wheels = BUNDLE_DIR / "wheels"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-index", f"--find-links={wheels}", *missing])

In [ ]:
import os

import polars
import torch
import transformers

print(f"torch {torch.__version__} | transformers {transformers.__version__} | polars {polars.__version__} | {os.cpu_count()} CPU cores")
assert torch.cuda.is_available(), "no CUDA device"
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {properties.name}, {properties.total_memory / 2**30:.0f} GB, compute capability {torch.cuda.get_device_capability(index)}")

device = torch.device("cuda:0")
assert torch.isfinite((torch.randn(64, 64, device=device) @ torch.randn(64, 64, device=device)).sum()).item()
torch.set_float32_matmul_precision("high")

In [ ]:
import shutil
from dataclasses import replace

import pandas as pd

from gptrec.baselines.suite import BaselineConfig
from gptrec.data.datasets import download_raw_data, download_side_data, load_item_metadata, load_raw_events, load_reactions
from gptrec.data.preprocess import PreprocessingConfig
from gptrec.experiment import ADOPTION, DEFAULT_REWARDS, LIKES, RETURN_AFTER_THE_LIST, Experiment, ExperimentConfig, reuse_artifacts, run_smoke_test
from gptrec.models.backbone import TransformerConfig
from gptrec.rollout import GenerationConfig, RewardSpec
from gptrec.seeds import (
    CONTROL_SEED_REPORTS,
    CONTROL_SEED_STAGES,
    FRONTIER_SEED_REPORTS,
    FRONTIER_SEED_STAGES,
    SEED_REPORTS,
    SEED_STAGES,
    run_seeds,
    run_seeds_smoke_test,
    summarise_seeds,
)
from gptrec.training.rl import RLConfig
from gptrec.training.supervised import FeedbackLossWeights, OptimizationConfig

SECONDS_PER_DAY = 86_400


def show(table):
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.float_format", "{:.4f}".format):
        display(table)


def show_report(name, **read_options):
    path = experiment.reports.directory / f"{name}.csv"
    if path.exists():
        print(name)
        show(pd.read_csv(path, **read_options))

## Configuration

In [ ]:
DATASET = "yambda"
SESSION = "final"

YAMBDA = ExperimentConfig(
    preprocessing=PreprocessingConfig(max_items=50_000, min_item_events=20, min_user_events=50, validation_days=7, test_days=7),
    transformer=TransformerConfig(hidden_size=256, num_layers=4, num_heads=4, dropout=0.1, max_positions=128),
    window_stride=32,
    policy_optimization=OptimizationConfig(epochs=14, batch_size=256, learning_rate=1e-3),
    reward_model_optimization=OptimizationConfig(epochs=8, batch_size=512, learning_rate=1e-3),
    baselines=BaselineConfig(optimization=OptimizationConfig(epochs=14, batch_size=256, learning_rate=1e-3)),
    horizon=10,
    max_future=20,
    ks=(1, 5, 10),
    evaluation_batch_size=1024,
    rl=RLConfig(num_steps=2000, histories_per_step=512, reference_kl_weight=0.3, learning_rate=1e-5, evaluate_every=250, mixed_precision=True),
    sweep_steps=1000,
    kl_weights=(0.0, 0.03, 0.05, 0.1, 0.3),
    rewards=DEFAULT_REWARDS[3:4],
    reward_kl_weights=(0.1,),
    choice_reward_kl_weights=(),
    logged_relevance_weights=(10.0,),
    logged_relevance_kl_weights=(0.0,),
    personal_reward_kl_weights=(0.1,),
    credit_ablation_kl_weights=(),
    importance_weight_caps=(0.5,),
    sampled_list_counts=(8,),
)
KION = ExperimentConfig(
    preprocessing=PreprocessingConfig(max_items=20_000, min_item_events=10, min_user_events=5, validation_days=7, test_days=7,
                                      session_gap_seconds=0, return_window_seconds=7 * SECONDS_PER_DAY),
    transformer=TransformerConfig(hidden_size=256, num_layers=4, num_heads=4, dropout=0.2, max_positions=48),
    window_stride=24,
    policy_optimization=OptimizationConfig(epochs=30, batch_size=512, learning_rate=1e-3),
    reward_model_optimization=OptimizationConfig(epochs=15, batch_size=512, learning_rate=1e-3),
    baselines=BaselineConfig(optimization=OptimizationConfig(epochs=30, batch_size=512, learning_rate=1e-3)),
    horizon=10,
    max_future=20,
    ks=(1, 5, 10),
    evaluation_batch_size=2048,
    rl=RLConfig(num_steps=2000, histories_per_step=512, min_history=3, generation=GenerationConfig(block_history_items=True),
                reference_kl_weight=0.3, learning_rate=1e-5, evaluate_every=250, mixed_precision=True),
    sweep_steps=1000,
    kl_weights=(0.0, 0.1, 0.2, 0.3, 0.5, 1.0),
    rewards=DEFAULT_REWARDS[2:5],
    reward_kl_weights=(0.1, 0.3),
    choice_reward_kl_weights=(),
    logged_relevance_weights=(10.0, 3.0),
    logged_relevance_kl_weights=(0.0, 0.03, 0.1, 0.3),
    personal_reward_kl_weights=(0.0, 0.1, 0.3),
    credit_ablation_kl_weights=(0.1, 0.3),
    importance_weight_caps=(0.5, 2.0),
    sampled_list_counts=(8,),
)

CONFIGS = {
    ("kion", "diagnostics"): replace(
        KION,
        kl_weights=(0.0, 0.1, 0.3),
        rewards=(),
        logged_relevance_weights=(),
        personal_reward_kl_weights=(),
        credit_ablation_kl_weights=(0.3,),
        importance_weight_caps=(0.5,),
        kl_budgets=(0.1, 0.5),
        estimator_kl_weights=(0.1, 0.3),
        high_learning_rate_factor=10.0,
        sampled_feedback_kl_weights=(0.1, 0.3),
        relative_reward_kl_weights=(0.0, 0.1, 0.3),
        diversity_weights=(1.0,),
        diversity_kl_weights=(0.1, 0.3),
    ),
    ("kion", "final"): replace(
        KION,
        kl_budgets=(0.1, 0.5),
        estimator_kl_weights=(0.1, 0.3),
        high_learning_rate_factor=10.0,
        sampled_feedback_kl_weights=(0.1, 0.3),
        relative_reward_kl_weights=(0.0, 0.1, 0.3),
        diversity_weights=(1.0,),
        diversity_kl_weights=(0.1, 0.3),
    ),
    ("kion", "seeds"): replace(
        KION,
        kl_weights=(0.1, 0.3, 0.5, 1.0),
        rewards=(),
        logged_relevance_weights=(10.0,),
        logged_relevance_kl_weights=(0.3,),
        personal_reward_kl_weights=(),
        credit_ablation_kl_weights=(),
        importance_weight_caps=(0.5,),
        sampled_list_counts=(),
        kl_budgets=(0.1,),
        estimator_kl_weights=(0.3,),
        relative_reward_kl_weights=(0.3,),
    ),
    ("yambda", "diagnostics"): replace(
        YAMBDA,
        kl_weights=(0.0, 0.03, 0.1),
        rewards=(),
        logged_relevance_weights=(),
        personal_reward_kl_weights=(),
        kl_budgets=(0.5,),
        estimator_kl_weights=(0.03, 0.1),
        relative_reward_kl_weights=(0.03, 0.1),
    ),
    ("yambda", "final"): replace(
        YAMBDA,
        logged_relevance_kl_weights=(0.0, 0.1),
        kl_budgets=(0.2, 0.5),
        estimator_kl_weights=(0.03, 0.1),
        sampled_feedback_kl_weights=(0.1,),
        relative_reward_kl_weights=(0.03, 0.1),
        diversity_weights=(1.0,),
        diversity_kl_weights=(0.1,),
    ),
    ("kion", "review"): replace(
        KION,
        kl_weights=(0.0, 0.1, 0.3, 1.0),
        rewards=(("return after the list", RETURN_AFTER_THE_LIST),),
        reward_kl_weights=(0.1, 0.3),
        masked_feedback_kl_weights=(0.1, 0.3),
    ),
    ("yambda", "review"): replace(
        YAMBDA,
        preprocessing=replace(YAMBDA.preprocessing, adoption_window_seconds=14 * SECONDS_PER_DAY),
        reward_model_loss=FeedbackLossWeights(liked=0.25, disliked=0.25, adopts=0.25),
        kl_weights=(0.05, 0.1, 0.3),
        rewards=(("likes", LIKES), ("adoption", ADOPTION), ("return after the list", RETURN_AFTER_THE_LIST)),
        reward_kl_weights=(0.1, 0.3),
        masked_feedback_kl_weights=(0.1, 0.3),
    ),
    ("yambda", "seeds"): replace(
        YAMBDA,
        kl_weights=(0.03, 0.1),
        rewards=(),
        logged_relevance_weights=(),
        logged_relevance_kl_weights=(),
        personal_reward_kl_weights=(),
        importance_weight_caps=(0.5,),
        sampled_list_counts=(),
        kl_budgets=(0.5,),
    ),
    ("kion", "frontier"): replace(
        KION,
        kl_weights=(0.1, 0.3, 1.0),
        rewards=(("return after the list", RETURN_AFTER_THE_LIST),),
        reward_kl_weights=(0.3,),
        logged_relevance_weights=(),
        personal_reward_kl_weights=(),
        credit_ablation_kl_weights=(),
        importance_weight_caps=(0.5,),
        kl_budgets=(0.1, 0.5),
        estimator_kl_weights=(0.3,),
        return_label_check=True,
        capacity_kl_budgets=(0.1, 0.5),
        low_ranks=(1, 8),
        group_relative_kl_budgets=(0.1, 0.5),
        analyse_policy_change=True,
        guidance_strengths=(1.0, 0.3, 0.1),
        guided_rewards=(("played ratio", RewardSpec()), ("return after the list", RETURN_AFTER_THE_LIST)),
        reward_weighted_temperatures=(0.3, 1.0),
        reward_weighted_responses=("relative_ratio", "played_ratio"),
        preference_betas=(0.1, 0.5),
        multi_token_lists=True,
        interpolation_factors=(0.25, 0.5, 0.75, 1.5, 2.0),
    ),
    ("yambda", "frontier"): replace(
        YAMBDA,
        preprocessing=replace(YAMBDA.preprocessing, adoption_window_seconds=14 * SECONDS_PER_DAY),
        reward_model_loss=FeedbackLossWeights(liked=0.25, disliked=0.25, adopts=0.25),
        kl_weights=(0.05, 0.1),
        rewards=(),
        logged_relevance_weights=(),
        personal_reward_kl_weights=(),
        importance_weight_caps=(0.5,),
        sampled_list_counts=(),
        kl_budgets=(0.5,),
        return_label_check=True,
        capacity_kl_budgets=(0.5,),
        low_ranks=(1,),
        group_relative_kl_budgets=(0.5,),
        analyse_policy_change=True,
        guidance_strengths=(1.0, 0.3, 0.1),
        guided_rewards=(("played ratio", RewardSpec()), ("adoption", ADOPTION)),
        reward_weighted_temperatures=(0.3, 1.0),
        reward_weighted_responses=("relative_ratio",),
        preference_betas=(0.1,),
        multi_token_lists=True,
        multi_token_optimization=OptimizationConfig(epochs=2, batch_size=256, learning_rate=1e-3),
        multi_token_targets_per_head=4096,
        interpolation_factors=(0.5, 1.5, 2.0),
    ),
    ("kion", "frontier_seeds"): replace(
        KION,
        kl_weights=(0.1, 0.3, 1.0),
        rewards=(),
        logged_relevance_weights=(),
        personal_reward_kl_weights=(),
        credit_ablation_kl_weights=(),
        sampled_list_counts=(),
        kl_budgets=(),
        capacity_kl_budgets=(0.1, 0.5),
        low_ranks=(1,),
        analyse_policy_change=True,
        guidance_strengths=(1.0, 0.3, 0.1),
        guidance_discounts=(0.0,),
        guided_rewards=(("played ratio", RewardSpec()),),
        reward_weighted_temperatures=(0.3, 1.0),
        reward_weighted_responses=("relative_ratio", "played_ratio"),
        preference_betas=(0.1, 0.5),
        multi_token_lists=True,
        interpolation_factors=(0.25, 0.5, 0.75, 1.5, 2.0),
    ),
    ("kion", "control"): replace(
        KION,
        kl_weights=(),
        rewards=(),
        logged_relevance_weights=(),
        logged_relevance_kl_weights=(),
        personal_reward_kl_weights=(),
        credit_ablation_kl_weights=(),
        importance_weight_caps=(),
        sampled_list_counts=(),
        reward_weighted_temperatures=(0.3, 1.0),
        reward_weighted_responses=("relative_ratio",),
        preference_betas=(0.5,),
        continued_training_control=True,
    ),
    ("kion", "quick"): replace(
        KION,
        kl_weights=(0.1, 0.3, 1.0),
        rewards=(),
        logged_relevance_weights=(),
        logged_relevance_kl_weights=(),
        personal_reward_kl_weights=(),
        credit_ablation_kl_weights=(),
        importance_weight_caps=(),
        sampled_list_counts=(),
    ),
}
SMOKE_TEST_USERS = {"yambda": 200, "kion": 30_000}

In [ ]:
DATASET = "yambda"
SESSION = "frontier"
config = CONFIGS[(DATASET, SESSION)]
HOME = WORKING / f"{DATASET}_{SESSION}"
TIME_BUDGET_HOURS = 11.0

In [ ]:
raw_events = load_raw_events(DATASET, BUNDLE_DIR / manifest["raw_data"][DATASET])
side_data = {name: BUNDLE_DIR / path for name, path in manifest.get("side_data", {}).get(DATASET, {}).items()}
reactions = load_reactions(DATASET, side_data) if side_data else None
item_metadata = load_item_metadata(DATASET, side_data) if side_data else None
print(f"{len(raw_events):,} raw events; beyond them: {sorted(side_data) or 'nothing'}")

## Smoke test

All stages on a subsample of users, with small models and a few optimisation steps.

In [ ]:
run_smoke_test(raw_events, config, WORKING / "smoke_test" / DATASET, device, max_users=SMOKE_TEST_USERS[DATASET], reactions=reactions, item_metadata=item_metadata)
shutil.rmtree(WORKING / "smoke_test")

## 1. Data, base models and the reference runs

In [ ]:
experiment = Experiment(config, HOME, device, time_budget_hours=TIME_BUDGET_HOURS)
summary = experiment.prepare_data(raw_events, reactions)
del raw_events
show(summary)
show(experiment.pretrain_policy())
show(experiment.train_reward_models())

In [ ]:
show(experiment.finetune())
show(experiment.sweep_kl_weight())
show(experiment.sweep_kl_budget())
show(experiment.compare_advantage_estimators())
show(experiment.explain_reward_gain())

## 2. What a return label records

Return rate after an event by the duration of the event's own item and of the next item the user consumed. If the
next item decides it, as one row per user and title dated by its last view would make it, a reward for returns
pays for bookkeeping.

In [ ]:
show(experiment.diagnose_return_label())

## 3. Is what RL learns one preference per item?

**Restricted capacity.** At each KL budget: every weight, one score offset per item, and low-rank updates of the
transformer. *Refuted* if item offsets reach far less predicted reward than full fine-tuning at the same KL.
`training/advantage_scale` is the standard deviation of the advantages before standardisation, by which the anchor
weight is effectively multiplied.

In [ ]:
show(experiment.restrict_finetuning_capacity())
show(experiment.compare_group_relative_advantages())

**Decomposition.** The main run's change of log-probabilities, averaged per item, added to the base recommender
("base + RL's item offsets") and removed from the fine-tuned one ("RL - its item offsets"). Then the share of
within-history variation that one effect per item explains: in the reward model's predictions (by the closed form,
the optimal change at any anchor weight) and in what each fine-tuned recommender changed. *A policy above the
reward's share amplified the per-item part beyond what the optimum asks for.*

In [ ]:
show(experiment.decompose_policy_change())
show_report("policy_change_parts", index_col=0)

## 4. Reaching the optimum at inference

Each position takes the candidate with the highest $\log \pi_0 + Q / \beta$ among the base recommender's most likely
next items, after the items already chosen. Discount 0 judges an item by its own reward; discount 0.9 adds a learned
value of what follows it. *If guidance at discount 0 matches RL at equal relevance, RL is an expensive way to compute
a closed form; if look-ahead helps adoption more than the share, what follows a new track decides whether it sticks.*

In [ ]:
show(experiment.align_at_inference())

## 5. Learning from the log alone

Reward-weighted fine-tuning (each logged item weighs $\exp(r/\tau)$) and within-user DPO (pairs of items one user
consumed after the same history, one above and one below the usual response for it). *The test is the logged
surplus: the one outcome that neither a reward model nor item length can inflate.* Per-user significance tables are in
the reports.

In [ ]:
show(experiment.learn_from_logged_preferences())

## 6. Lists that never read an unconfirmed item

One head per position, each predicting the item that many events ahead from the real history only.

In [ ]:
show(experiment.predict_lists_without_feedback())
show_report("multi_token_position_hits", index_col=0)

## 7. Operating points in weight space

$\theta_0 + a(\theta_{RL} - \theta_0)$: interpolation below 1, extrapolation above. Every recommender's distance is
the KL along its own lists.

In [ ]:
show(experiment.interpolate_weights())

## Reports

In [ ]:
show(pd.read_csv(experiment.reports.directory / "stage_minutes.csv", index_col=0))
shutil.make_archive(str(WORKING / f"{DATASET}_{SESSION}_reports"), "zip", experiment.reports.directory)
sorted(path.name for path in experiment.reports.directory.iterdir())